In [1]:
"""
完全修正版：リアリスティック・バックテスト（CASH完全対応 + 列名自動検出）
- 取引コスト計算の修正
- リスクオフ判定の修正（CASH明示追加）
- 信用金利計算の修正
- 列名自動検出機能の追加
"""

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 基本設定
# ============================================================

OUTPUT_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks")
SNAP_PATH = OUTPUT_DIR / "factors" / "month_end_snapshot.parquet"
MERGED_PARTS_DIR = OUTPUT_DIR / "merged_parts"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_daily"
ANALYSIS_DIR.mkdir(exist_ok=True)

# ============================================================
# パラメータ（修正版）
# ============================================================

### 基本パラメータ
INITIAL_CAPITAL = 10_000_000
TOP_N = 20
VO_TOP_PCT = 0.70
DEFAULT_TRADING_UNIT = 100

### 株価・財務データフィルタ
MIN_PRICE = 800
MAX_PRICE = None
ROE_MIN = -0.5
ROE_MAX = 1.0
BM_RATIO_MIN = 0.1
BM_RATIO_MAX = 10.0

### リスクオフ判定
OFF_TH = -0.115
ON_TH = -0.085
TREND3_TH = -0.02
VOL3_Q = 0.90

### 取引コスト
COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.003
MARGIN_INTEREST_RATE = 0.028
TOTAL_COST_PER_TRADE = COMMISSION_RATE + SLIPPAGE_RATE

### 流動性制約
MAX_VOLUME_PARTICIPATION = 0.10
MIN_DAILY_TURNOVER = 5_000_000

### 価格ギャップ対策
PRICE_GAP_BUFFER = 0.95
ENTRY_SLIPPAGE = 0.01

### レバレッジ
MARGIN_SAFETY_BUFFER = 0.15
MAX_SAFE_LEVERAGE = 1.0 + (0.2 / (1 + MARGIN_SAFETY_BUFFER))

DD_THRESHOLD_MID = -0.05
DD_THRESHOLD_LOW = -0.08
LEVERAGE_HIGH = min(1.3, MAX_SAFE_LEVERAGE)
LEVERAGE_MID = 1.1
LEVERAGE_LOW = 1.0

print("=" * 70)
print("完全修正版バックテスト：CASH完全対応 + 列名自動検出")
print("=" * 70)
print(f"初期資金: ¥{INITIAL_CAPITAL:,}")
print(f"TOP_N: {TOP_N}")
print(f"株価フィルタ: ¥{MIN_PRICE}以上")
print(f"売買代金上位: {VO_TOP_PCT*100:.0f}%")
print(f"最低日次売買代金: ¥{MIN_DAILY_TURNOVER:,}")
print(f"取引コスト: 往復{TOTAL_COST_PER_TRADE*100:.2f}% + 信用金利{MARGIN_INTEREST_RATE*100:.1f}%")
print(f"レバレッジ上限: {LEVERAGE_HIGH:.2f}x（追証安全率考慮）")
print(f"価格ギャップバッファ: {(1-PRICE_GAP_BUFFER)*100:.0f}%")
print("=" * 70)

# ============================================================
# 列名自動検出関数
# ============================================================

def detect_stock_code_column(df):
    """StockCode列名を自動検出"""
    candidates = ['Code', 'StockCode', 'Symbol', 'Ticker', 'stock_code', 'code']
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"銘柄コード列が見つかりません。利用可能な列: {list(df.columns)}")

def normalize_columns(df, stock_code_col=None):
    """列名を統一"""
    rename_map = {}
    
    # StockCode列の正規化
    if stock_code_col and stock_code_col != 'Code':
        rename_map[stock_code_col] = 'Code'
    
    # 価格列の正規化
    if 'AdjustedClose' in df.columns and 'AdjustmentClose' not in df.columns:
        rename_map['AdjustedClose'] = 'AdjustmentClose'
    
    # Volume列の正規化
    if 'Volume' in df.columns and 'Vo' not in df.columns:
        rename_map['Volume'] = 'Vo'
    
    return df.rename(columns=rename_map)

# ============================================================
# Step 1: 月次スナップショット読み込み
# ============================================================

print("\n[1/8] 月次スナップショット読み込み中（列名自動検出）...")
snap = pd.read_parquet(SNAP_PATH)
snap['MonthEnd'] = pd.to_datetime(snap['MonthEnd'])

# 列名を確認
print(f"📋 検出された列（最初の10列）: {list(snap.columns[:10])}")

# StockCode列名を自動検出
stock_code_col = detect_stock_code_column(snap)
print(f"✅ 銘柄コード列を検出: '{stock_code_col}' → 'Code'")

# 列名正規化
snap = normalize_columns(snap, stock_code_col)

# 必須列チェック
required_cols = ['MonthEnd', 'Code', 'AdjustmentClose', 'Vo', 'MarketCap', 
                 'BM_Ratio', 'ROE', 'INV_Growth']
missing_cols = [c for c in required_cols if c not in snap.columns]

if missing_cols:
    print(f"\n⚠️ 警告: 以下の列が不足しています: {missing_cols}")
    
    if 'AdjustmentClose' not in snap.columns:
        if 'Close' in snap.columns:
            snap['AdjustmentClose'] = snap['Close']
            print("  代替: Close → AdjustmentClose")
        elif 'AdjustedClose' in snap.columns:
            snap['AdjustmentClose'] = snap['AdjustedClose']
            print("  代替: AdjustedClose → AdjustmentClose")
    
    if 'Vo' not in snap.columns and 'Volume' in snap.columns:
        snap['Vo'] = snap['Volume']
        print("  代替: Volume → Vo")
    
    missing_cols = [c for c in required_cols if c not in snap.columns]
    if missing_cols:
        raise ValueError(f"必須列が不足しており代替もできません: {missing_cols}")

print(f"✅ 列名正規化完了")
print(f"✅ 期間: {snap['MonthEnd'].min().date()} 〜 {snap['MonthEnd'].max().date()}")
print(f"✅ データ: {len(snap):,} 行, {snap['Code'].nunique()} 銘柄")

# ============================================================
# Step 2: 月次ポートフォリオ形成
# ============================================================

print("\n[2/8] 月次ポートフォリオ形成中（CASH対応版）...")

snap = snap.sort_values(['Code', 'MonthEnd'])
snap['MonthKey'] = snap['MonthEnd'].dt.to_period('M').astype(str)
snap['ret_m_fwd'] = snap.groupby('Code')['AdjustmentClose'].pct_change().shift(-1)
snap['TurnoverValue'] = snap['Vo'] * snap['AdjustmentClose']

monthly = snap.groupby('MonthEnd').apply(
    lambda g: pd.Series({
        'mkt_vw': np.average(g['ret_m_fwd'].fillna(0), 
                            weights=g['MarketCap'].fillna(1)),
        'count': len(g)
    })
).reset_index()

monthly['cum_ret'] = (1 + monthly['mkt_vw']).cumprod()
monthly['peak'] = monthly['cum_ret'].cummax()
monthly['mkt_dd'] = (monthly['cum_ret'] / monthly['peak']) - 1
monthly['trend3'] = monthly['mkt_vw'].rolling(3, min_periods=1).mean()
monthly['vol3'] = monthly['mkt_vw'].rolling(3, min_periods=2).std()

vol3_th = monthly['vol3'].quantile(VOL3_Q)
print(f"✅ Vol3閾値（{VOL3_Q*100:.0f}%ile）: {vol3_th:.6f}")

snap = snap.merge(
    monthly[['MonthEnd', 'mkt_dd', 'trend3', 'vol3']], 
    on='MonthEnd', 
    how='left'
)

snap['risk_off'] = (
    (snap['mkt_dd'] <= OFF_TH) | 
    (snap['trend3'] <= TREND3_TH) | 
    (snap['vol3'] >= vol3_th)
)

portfolio_list = []
filter_stats = []
risk_off_months = 0

for month_end, df_month in snap.groupby('MonthEnd'):
    month_key = df_month['MonthKey'].iloc[0]
    risk_off = df_month['risk_off'].iloc[0]
    
    stats = {
        'MonthEnd': month_end,
        'MonthKey': month_key,
        'initial_count': len(df_month),
        'risk_off': risk_off
    }
    
    if risk_off:
        risk_off_months += 1
        portfolio_list.append({
            'MonthEnd': month_end,
            'MonthKey': month_key,
            'Code': 'CASH',
            'Weight': 1.0,
            'risk_off': True
        })
        stats.update({
            'after_price': 0,
            'after_roe_bm': 0,
            'after_turnover': 0,
            'after_liquidity': 0,
            'final_count': 0,
            'selected_count': 0
        })
        filter_stats.append(stats)
        continue
    
    df_valid = df_month[df_month['AdjustmentClose'] >= MIN_PRICE].copy()
    if MAX_PRICE:
        df_valid = df_valid[df_valid['AdjustmentClose'] <= MAX_PRICE]
    stats['after_price'] = len(df_valid)
    
    df_valid = df_valid[
        (df_valid['BM_Ratio'].notna()) & 
        (df_valid['ROE'].notna()) &
        (df_valid['ROE'] >= ROE_MIN) & 
        (df_valid['ROE'] <= ROE_MAX) &
        (df_valid['BM_Ratio'] >= BM_RATIO_MIN) & 
        (df_valid['BM_Ratio'] <= BM_RATIO_MAX)
    ]
    stats['after_roe_bm'] = len(df_valid)
    
    if 'TurnoverValue' in df_valid.columns and len(df_valid) > 0:
        turnover_th = df_valid['TurnoverValue'].quantile(1 - VO_TOP_PCT)
        df_valid = df_valid[df_valid['TurnoverValue'] >= turnover_th]
    stats['after_turnover'] = len(df_valid)
    
    df_valid = df_valid[df_valid['TurnoverValue'] >= MIN_DAILY_TURNOVER]
    
    df_valid['MaxBuyableAmount'] = (
        df_valid['Vo'] * MAX_VOLUME_PARTICIPATION * df_valid['AdjustmentClose']
    )
    
    target_capital = INITIAL_CAPITAL * LEVERAGE_HIGH * PRICE_GAP_BUFFER
    capital_per_stock = target_capital / TOP_N
    
    df_valid = df_valid[df_valid['MaxBuyableAmount'] >= capital_per_stock * 0.8]
    stats['after_liquidity'] = len(df_valid)
    
    if len(df_valid) < TOP_N:
        portfolio_list.append({
            'MonthEnd': month_end,
            'MonthKey': month_key,
            'Code': 'CASH',
            'Weight': 1.0,
            'risk_off': False
        })
        stats.update({
            'final_count': len(df_valid),
            'selected_count': 0
        })
        filter_stats.append(stats)
        continue
    
    for col in ['BM_Ratio', 'ROE', 'INV_Growth']:
        mean_val = df_valid[col].mean()
        std_val = df_valid[col].std()
        if std_val > 0:
            df_valid[f'{col}_z'] = (df_valid[col] - mean_val) / std_val
        else:
            df_valid[f'{col}_z'] = 0
    
    df_valid['composite_score'] = (
        df_valid['BM_Ratio_z'] + 
        df_valid['ROE_z'] - 
        df_valid['INV_Growth_z'].fillna(0)
    )
    
    top_n = df_valid.nlargest(TOP_N, 'composite_score')
    
    stats['final_count'] = len(df_valid)
    stats['selected_count'] = len(top_n)
    filter_stats.append(stats)
    
    weight = 1.0 / len(top_n)
    
    for _, row in top_n.iterrows():
        portfolio_list.append({
            'MonthEnd': month_end,
            'MonthKey': month_key,
            'Code': row['Code'],
            'Weight': weight,
            'AdjustmentClose': row['AdjustmentClose'],
            'TurnoverValue': row['TurnoverValue'],
            'MaxBuyableAmount': row['MaxBuyableAmount'],
            'risk_off': False
        })

df_portfolio = pd.DataFrame(portfolio_list)
df_filter_stats = pd.DataFrame(filter_stats)

total_months = len(df_filter_stats)
cash_months = len(df_portfolio[df_portfolio['Code'] == 'CASH']['MonthKey'].unique())

print(f"✅ ポートフォリオ形成完了: {len(df_portfolio)} レコード")
print(f"✅ リスクオフ月数: {risk_off_months}/{total_months} ({risk_off_months/total_months*100:.1f}%)")
print(f"✅ CASH月数: {cash_months}/{total_months} ({cash_months/total_months*100:.1f}%)")

print(f"\n[新] フィルタ統計（全期間平均）:")
print(f"  初期銘柄数: {df_filter_stats['initial_count'].mean():.0f}")
print(f"  株価フィルタ後: {df_filter_stats['after_price'].mean():.0f} "
      f"(除外: {df_filter_stats['initial_count'].mean() - df_filter_stats['after_price'].mean():.0f})")
print(f"  財務データ後: {df_filter_stats['after_roe_bm'].mean():.0f} "
      f"(除外: {df_filter_stats['after_price'].mean() - df_filter_stats['after_roe_bm'].mean():.0f})")
print(f"  売買代金後: {df_filter_stats['after_turnover'].mean():.0f} "
      f"(除外: {df_filter_stats['after_roe_bm'].mean() - df_filter_stats['after_turnover'].mean():.0f})")
print(f"  流動性制約後: {df_filter_stats['after_liquidity'].mean():.0f} "
      f"(除外: {df_filter_stats['after_turnover'].mean() - df_filter_stats['after_liquidity'].mean():.0f})")
print(f"  最終選定: {df_filter_stats['selected_count'].mean():.0f}")

# ============================================================
# Step 3: 日次データ読み込み
# ============================================================

print("\n[3/8] 日次データ読み込み中（列名自動検出）...")

daily_files = sorted(MERGED_PARTS_DIR.glob("merged-part-*.parquet"))
if not daily_files:
    raise FileNotFoundError(f"日次データが見つかりません: {MERGED_PARTS_DIR}")

# サンプルファイルから列名を確認
sample_df = pd.read_parquet(daily_files[0])
daily_stock_code_col = detect_stock_code_column(sample_df)
print(f"✅ 日次データの銘柄コード列: '{daily_stock_code_col}' → 'Code'")

daily_list = []
for f in daily_files:
    df_part = pd.read_parquet(f)
    df_part = normalize_columns(df_part, daily_stock_code_col)
    daily_list.append(df_part)

daily = pd.concat(daily_list, ignore_index=True)
daily['Date'] = pd.to_datetime(daily['Date'])
daily = daily.sort_values(['Code', 'Date'])

# 列名の正規化確認
if 'AdjustedClose' in daily.columns and 'AdjustmentClose' not in daily.columns:
    daily = daily.rename(columns={'AdjustedClose': 'AdjustmentClose'})

daily['ret_d'] = daily.groupby('Code')['AdjustmentClose'].pct_change()
daily['MonthKey'] = daily['Date'].dt.to_period('M').astype(str)

print(f"✅ 日次データ: {len(daily):,} 行, {daily['Code'].nunique()} 銘柄")

# ============================================================
# Step 4: 日次ポートフォリオリターン計算（改善版）
# - レバレッジ1.0固定（信用金利ゼロ）
# - 月初コストを固定で入れるのではなく、ターンオーバー比例に変更
# - 条件付きリバランス + 部分リバランスを導入
# ============================================================

print("\n[4/8] 日次ポートフォリオリターン計算中（改善版：部分リバランス+ターンオーバーコスト）...")

# --- 改善版の新パラメータ ---
LEVERAGE_FIXED = 1.0                 # 信用取引しない
REBALANCE_MIN_TURNOVER_STOCKS = 5    # 入替銘柄数がこれ未満なら“売買抑制”検討
REBALANCE_MIN_TURNOVER_PCT = 0.10    # ターンオーバーがこれ未満なら“売買抑制”検討
WEIGHT_ADJUST_DEADBAND = 0.05        # 同一銘柄のウェイト差が5%未満なら調整しない

# ポートフォリオは「MonthEndで決めて翌月に適用」
df_portfolio['MonthKey_next'] = (
    pd.to_datetime(df_portfolio['MonthEnd']) + pd.DateOffset(months=1)
).dt.to_period('M').astype(str)

# CASH月マスタ
cash_tbl = df_portfolio[df_portfolio['Code'] == 'CASH'][['MonthKey_next', 'risk_off']].copy()
cash_tbl = cash_tbl.rename(columns={'MonthKey_next': 'MonthKey'})
cash_tbl['is_cash_month'] = True

# 株式ポート（CASH除外）
stock_portfolio = df_portfolio[df_portfolio['Code'] != 'CASH'].copy()

# 日次：株式ポートのみを付与（inner）
daily_port = daily.merge(
    stock_portfolio[['MonthKey_next', 'Code', 'Weight', 'risk_off']],
    left_on=['MonthKey', 'Code'],
    right_on=['MonthKey_next', 'Code'],
    how='inner'
)

# 日次：CASH判定付与（left）
daily_port = daily_port.merge(
    cash_tbl[['MonthKey', 'is_cash_month', 'risk_off']],
    on='MonthKey',
    how='left',
    suffixes=('', '_cash')
)

daily_port['is_cash_month'] = daily_port['is_cash_month'].fillna(False)
# risk_off は cash側があれば優先
if 'risk_off_cash' in daily_port.columns:
    daily_port['risk_off'] = daily_port['risk_off_cash'].fillna(daily_port['risk_off'])
    daily_port = daily_port.drop(columns=['risk_off_cash'])

daily_port = daily_port.sort_values('Date')

print(f"✅ 日次ポートフォリオ（株式側）: {len(daily_port):,} 行")

# --- 月次ターゲット・ポートフォリオ辞書化（ターンオーバー計算用）---
# month_key -> {code: weight}
monthly_target = {}
for mk, g in stock_portfolio.groupby('MonthKey_next'):
    monthly_target[mk] = dict(zip(g['Code'].astype(str), g['Weight'].astype(float)))

# cash月判定用set
cash_month_set = set(cash_tbl['MonthKey'].unique())

# --- 全営業日（CASH日も含む）---
all_dates = sorted(daily['Date'].unique())

# --- 状態変数 ---
global_wealth = INITIAL_CAPITAL
global_peak = INITIAL_CAPITAL
current_dd = 0.0

daily_returns = []
leverage_events = []
rebalance_count = 0
skip_rebalance_count = 0

total_trading_cost = 0.0
total_interest_cost = 0.0  # 改善版では常に0のはず（検算用）

prev_month_key = None

# 「前月末時点の保有ウェイト」（理想ウェイトとして保持：実売買の近似）
current_hold_weights = {}  # code -> weight（合計1.0想定）

def calc_turnover_and_new_hold(prev_w, target_w, deadband=WEIGHT_ADJUST_DEADBAND):
    """
    部分リバランス（デッドバンド付き）を近似。
    - まず、共通銘柄はウェイト差がdeadband未満なら据え置き
    - 売買が必要な銘柄だけ調整する
    - turnover = sum(|w_after - w_before|)
      （理論上、全入替だとおおむね2に近づく）
    """
    prev = prev_w.copy()
    target = target_w.copy()

    # まず、targetに存在しない銘柄は売却（0へ）
    all_codes = set(prev.keys()) | set(target.keys())
    after = {}

    for c in all_codes:
        w0 = prev.get(c, 0.0)
        w1 = target.get(c, 0.0)

        if (c in prev) and (c in target):
            if abs(w1 - w0) < deadband:
                after[c] = w0  # 据え置き
            else:
                after[c] = w1
        else:
            # 片方にしかない → 目標通り（新規 or 売却）
            after[c] = w1

    # 0ウェイト銘柄を落とす
    after = {c: w for c, w in after.items() if w > 0}

    # 正規化（丸め誤差対策）
    s = sum(after.values())
    if s > 0:
        after = {c: w/s for c, w in after.items()}

    # turnover
    codes2 = set(prev.keys()) | set(after.keys())
    turnover = sum(abs(after.get(c, 0.0) - prev.get(c, 0.0)) for c in codes2)

    # 銘柄入替数（ユニバース変化量）
    turnover_stocks = len(set(prev.keys()) ^ set(after.keys()))

    return turnover, turnover_stocks, after

for date in all_dates:
    current_month_key = pd.Timestamp(date).to_period('M').strftime('%Y-%m')

    # --- CASH判定 ---
    is_cash_month = current_month_key in cash_month_set
    is_risk_off = False
    if is_cash_month:
        # cash_tblにrisk_off情報がある場合だけ参照
        tmp = cash_tbl[cash_tbl['MonthKey'] == current_month_key]
        if len(tmp) > 0:
            is_risk_off = bool(tmp['risk_off'].iloc[0])
    is_cash = is_cash_month or is_risk_off

    # --- 当日の株式リターン計算（CASHなら0）---
    if is_cash:
        port_ret_gross = 0.0
        port_ret_net = 0.0
        trading_cost = 0.0
        interest_cost_pct = 0.0
        leverage_used = 1.0

        # CASH期間は保有ウェイトを空に近づける（次のONで全入替が起こる想定）
        # ただし“CASH月”は翌月のターゲットが株式なら月初でターンオーバー計算されるのでここは維持でもOK
        # ここでは保守的に「維持」する
    else:
        # 当日の銘柄別リターン（その月のターゲット構成の銘柄しかdaily_portには来ない）
        df_day = daily_port[daily_port['Date'] == date].copy()
        if len(df_day) == 0:
            # データ欠損日はスキップ
            continue

        # レバレッジ固定
        leverage_used = LEVERAGE_FIXED

        df_day['weighted_ret'] = df_day['ret_d'].fillna(0) * df_day['Weight']
        port_ret_gross = df_day['weighted_ret'].sum() * leverage_used

        # 月初判定（MonthKeyが変わった日＝その月の初登場日）
        is_month_start = (prev_month_key is None) or (str(current_month_key) != str(prev_month_key))

        trading_cost = 0.0
        interest_cost_pct = 0.0  # レバなし

        # --- 月初にのみ、ターンオーバー比例のコストを入れる ---
        if is_month_start:
            target_w = monthly_target.get(current_month_key, {})
            # ターゲットが空なら（あり得ないが）CASH扱い
            if len(target_w) == 0:
                port_ret_net = 0.0
            else:
                turnover, turnover_stocks, new_hold = calc_turnover_and_new_hold(current_hold_weights, target_w)

                # “売買抑制”（条件付きリバランス）
                # ・入替銘柄数が少ない AND ・ターンオーバー小さい → コストかけてまで調整しない
                if (turnover_stocks < REBALANCE_MIN_TURNOVER_STOCKS) and (turnover < REBALANCE_MIN_TURNOVER_PCT):
                    # リバランス見送り：保有ウェイト維持（=売買なし）
                    skip_rebalance_count += 1
                    turnover_used = 0.0
                    trading_cost = 0.0
                    # ただし、df_dayのWeightはターゲットなので、そのままだと「見送り」の意味が薄い
                    # ここでは簡略化として「見送り＝コストゼロ、リターンはターゲットで近似」
                    # （厳密にやるなら、当日の加重リターンをcurrent_hold_weightsで再計算する必要あり）
                    current_hold_weights = current_hold_weights  # 変更なし
                else:
                    # リバランス実行
                    rebalance_count += 1
                    turnover_used = turnover
                    trading_cost = turnover_used * TOTAL_COST_PER_TRADE
                    total_trading_cost += trading_cost
                    current_hold_weights = new_hold

                port_ret_net = port_ret_gross - trading_cost - interest_cost_pct

        else:
            port_ret_net = port_ret_gross  # 月中はコストなし（改善版の思想）

    # 資産更新
    global_wealth *= (1 + port_ret_net)

    if global_wealth > global_peak:
        global_peak = global_wealth
        leverage_events.append({
            'Date': date,
            'Event': 'NewPeak',
            'Wealth': global_wealth,
            'Leverage': leverage_used
        })

    current_dd = (global_wealth / global_peak) - 1
    prev_month_key = current_month_key

    daily_returns.append({
        'Date': date,
        'port_ret_gross': port_ret_gross,
        'port_ret_net': port_ret_net,
        'trading_cost': trading_cost,
        'interest_cost': interest_cost_pct,
        'leverage': leverage_used,
        'wealth': global_wealth,
        'peak': global_peak,
        'dd': current_dd,
        'is_cash': is_cash,
        'is_risk_off': is_risk_off,
        'is_cash_month': is_cash_month
    })

df_daily_returns = pd.DataFrame(daily_returns)

print(f"\n✅ 日次ポートフォリオリターン: {len(df_daily_returns)} 日")
print(f"✅ リバランス実行回数: {rebalance_count} 回")
print(f"✅ リバランス見送り回数: {skip_rebalance_count} 回")
print(f"✅ レバレッジイベント: {len(leverage_events)} 回（改善版では参考）")

print(f"\n[デバッグ] コスト累積（改善版）:")
print(f"  取引コスト合計: {total_trading_cost*100:.2f}%")
print(f"  信用金利合計: {total_interest_cost*100:.2f}%（常に0が期待）")
print(f"  コスト合計: {(total_trading_cost + total_interest_cost)*100:.2f}%")

# ============================================================
# Step 5: 日次MDD計算
# ============================================================

print("\n[5/8] 日次MDD計算中...")

df_daily_returns['cum_ret'] = df_daily_returns['port_ret_net'].add(1).cumprod().sub(1)
df_daily_returns['cum_wealth'] = INITIAL_CAPITAL * (1 + df_daily_returns['cum_ret'])

total_days = len(df_daily_returns)
final_wealth = df_daily_returns['cum_wealth'].iloc[-1]
cum_return = (final_wealth / INITIAL_CAPITAL) - 1

first_date = df_daily_returns['Date'].min()
last_date = df_daily_returns['Date'].max()
years = (last_date - first_date).days / 365.25

ann_return = (1 + cum_return) ** (1 / years) - 1
ann_vol = df_daily_returns['port_ret_net'].std() * np.sqrt(252)
sharpe = ann_return / ann_vol if ann_vol > 0 else 0

daily_mdd = df_daily_returns['dd'].min()
worst_dd_date = df_daily_returns.loc[df_daily_returns['dd'].idxmin(), 'Date']

cash_pct = cash_days / total_days * 100
avg_leverage = df_daily_returns['leverage'].mean()

total_gross_return = df_daily_returns['port_ret_gross'].add(1).cumprod().iloc[-1] - 1
total_net_return = cum_return
cost_drag = total_gross_return - total_net_return

print("\n" + "=" * 70)
print("最終結果サマリー（完全修正版：CASH完全対応）")
print("=" * 70)
print(f"期間: {first_date.date()} 〜 {last_date.date()}")
print(f"総日数: {total_days}日 ({years:.2f}年)")
print(f"初期資金: ¥{INITIAL_CAPITAL:,}")
print(f"最終資産: ¥{final_wealth:,.0f}")
print(f"累積リターン: {cum_return*100:.2f}%")
print(f"年率リターン: {ann_return*100:.2f}%")
print(f"年率ボラティリティ: {ann_vol*100:.2f}%")
print(f"シャープレシオ: {sharpe:.4f}")
print(f"日次最大DD: {daily_mdd*100:.2f}% (日付: {worst_dd_date.date()})")
print(f"平均CASH比率: {cash_pct:.2f}%")
print(f"平均レバレッジ: {avg_leverage:.2f}x")
print(f"\n[新] 取引コストの影響:")
print(f"  グロスリターン: {total_gross_return*100:.2f}%")
print(f"  ネットリターン: {total_net_return*100:.2f}%")
print(f"  コストドラッグ: {cost_drag*100:.2f}%")
print(f"  年率コストドラッグ: {(cost_drag/years)*100:.2f}%/年")
print("=" * 70)

print("\nワースト5ドローダウン日（CASH除外）:")
worst_days = df_daily_returns[~df_daily_returns['is_cash']].nsmallest(5, 'dd')
for _, row in worst_days.iterrows():
    print(f"  {row['Date'].date()}: dd {row['dd']:.6f}, "
          f"port_ret {row['port_ret_net']:.6f}, leverage {row['leverage']:.1f}")

# ============================================================
# Step 6: ファイル保存
# ============================================================

print("\n[6/8] ファイル保存中...")

out_file1 = ANALYSIS_DIR / "daily_portfolio_returns_realistic_cash_final.parquet"
df_daily_returns.to_parquet(out_file1, index=False)
print(f"  ✅ {out_file1}")

if leverage_events:
    df_leverage_events = pd.DataFrame(leverage_events)
    out_file2 = ANALYSIS_DIR / "leverage_events_realistic_cash_final.csv"
    df_leverage_events.to_csv(out_file2, index=False, encoding='utf-8-sig')
    print(f"  ✅ {out_file2}")

summary_data = {
    'period_start': [first_date],
    'period_end': [last_date],
    'total_days': [total_days],
    'initial_capital': [INITIAL_CAPITAL],
    'final_wealth': [final_wealth],
    'cumulative_return': [cum_return],
    'annual_return': [ann_return],
    'annual_volatility': [ann_vol],
    'sharpe_ratio': [sharpe],
    'daily_max_dd': [daily_mdd],
    'worst_dd_date': [worst_dd_date],
    'cash_days': [cash_days],
    'cash_pct': [cash_pct],
    'avg_leverage': [avg_leverage],
    'gross_return': [total_gross_return],
    'net_return': [total_net_return],
    'cost_drag': [cost_drag],
    'annual_cost_drag': [cost_drag / years],
    'rebalance_count': [rebalance_count],
    'total_trading_cost': [total_trading_cost],
    'total_interest_cost': [total_interest_cost]
}
df_summary = pd.DataFrame(summary_data)
out_file3 = ANALYSIS_DIR / "daily_mdd_summary_realistic_cash_final.csv"
df_summary.to_csv(out_file3, index=False, encoding='utf-8-sig')
print(f"  ✅ {out_file3}")

out_file4 = ANALYSIS_DIR / "filter_statistics_realistic_cash_final.csv"
df_filter_stats.to_csv(out_file4, index=False, encoding='utf-8-sig')
print(f"  ✅ {out_file4}")

# ============================================================
# Step 7: 比較表作成
# ============================================================

print("\n[7/8] 戦略比較表作成中...")

comparison_data = {
    '指標': [
        '年率リターン',
        '年率ボラティリティ',
        'シャープレシオ',
        '最大DD',
        '累積リターン',
        'CASH比率',
        '平均レバレッジ',
        'コストドラッグ',
        '年率コストドラッグ'
    ],
    '改善版（前回）': [
        '29.75%',
        '15.29%',
        '1.9450',
        '-15.41%',
        '1146.92%',
        '36.10%',
        '1.23x',
        '-',
        '-'
    ],
    'バグ版（修正前）': [
        '19.85%',
        '19.77%',
        '1.0044',
        '-23.59%',
        '478.17%',
        '0.00%',
        '1.11x',
        '609.45%',
        '62.9%/年'
    ],
    '完全修正版（今回）': [
        f'{ann_return*100:.2f}%',
        f'{ann_vol*100:.2f}%',
        f'{sharpe:.4f}',
        f'{daily_mdd*100:.2f}%',
        f'{cum_return*100:.2f}%',
        f'{cash_pct:.2f}%',
        f'{avg_leverage:.2f}x',
        f'{cost_drag*100:.2f}%',
        f'{(cost_drag/years)*100:.2f}%/年'
    ]
}
df_comparison = pd.DataFrame(comparison_data)

print("\n" + "=" * 70)
print("戦略比較表（新旧比較）")
print("=" * 70)
print(df_comparison.to_string(index=False))
print("=" * 70)

# ============================================================
# Step 8: 完了メッセージ
# ============================================================

print("\n" + "=" * 70)
print("[8/8] 完全修正版バックテスト完了！ ✅")
print("=" * 70)
print("\n🎉 CASH対応版が正常に動作しました！")
print("\n次のステップ:")
print("1. ✅ CASH比率が30〜40%に改善されたか確認")
print("2. ✅ コストドラッグが妥当な範囲（年率5〜10%）か確認")
print("3. ✅ 最大DDが-18%以下に改善されたか確認")
print("4. ✅ シャープレシオが1.5以上か確認")
print("5. 🚀 結果を踏まえて実運用開始を判断")
print()


完全修正版バックテスト：CASH完全対応 + 列名自動検出
初期資金: ¥10,000,000
TOP_N: 20
株価フィルタ: ¥800以上
売買代金上位: 70%
最低日次売買代金: ¥5,000,000
取引コスト: 往復0.40% + 信用金利2.8%
レバレッジ上限: 1.17x（追証安全率考慮）
価格ギャップバッファ: 5%

[1/8] 月次スナップショット読み込み中（列名自動検出）...
📋 検出された列（最初の10列）: ['MonthEnd', 'Code', 'AdjustedClose', 'Vo', 'MarketCap', 'BM_Ratio', 'ROE', 'INV_Growth', 'Month']
✅ 銘柄コード列を検出: 'Code' → 'Code'
✅ 列名正規化完了
✅ 期間: 2016-01-31 〜 2026-01-31
✅ データ: 502,101 行, 5303 銘柄

[2/8] 月次ポートフォリオ形成中（CASH対応版）...
✅ Vol3閾値（90%ile）: 0.064522
✅ ポートフォリオ形成完了: 1812 レコード
✅ リスクオフ月数: 32/121 (26.4%)
✅ CASH月数: 32/121 (26.4%)

[新] フィルタ統計（全期間平均）:
  初期銘柄数: 4150
  株価フィルタ後: 2055 (除外: 2095)
  財務データ後: 1645 (除外: 410)
  売買代金後: 1151 (除外: 494)
  流動性制約後: 1151 (除外: 0)
  最終選定: 15

[3/8] 日次データ読み込み中（列名自動検出）...
✅ 日次データの銘柄コード列: 'Code' → 'Code'
✅ 日次データ: 10,016,062 行, 5303 銘柄

[4/8] 日次ポートフォリオリターン計算中（改善版：部分リバランス+ターンオーバーコスト）...
✅ 日次ポートフォリオ（株式側）: 35,596 行

✅ 日次ポートフォリオリターン: 2430 日
✅ リバランス実行回数: 88 回
✅ リバランス見送り回数: 0 回
✅ レバレッジイベント: 318 回（改善版では参考）

[デバッグ] コスト累積（改善版）:
  取引コスト合計: 27.00%
  信用金利合

NameError: name 'cash_days' is not defined

In [2]:
"""
月次戦略 改善版：リアリスティック・バックテスト
（CASH完全対応 + 列名自動検出 + 月末引けで決定→翌月初寄りで執行）

改善点（従来版→改善版）
- リバランス執行前提を「②月末引けで決定→翌月初寄りで執行」に統一
- 取引コストを「月初固定」ではなく「ターンオーバー×片道コスト（現実的）」へ変更
- 部分リバランス（デッドバンド）を導入し売買回数/売買量を低減
- レバレッジを1.0固定（信用金利ゼロ化）でコスト削減を優先
- CASH月の扱いを明示（翌月 is_cash_month=True → 日次リターン0）
- 列名自動検出と正規化（Code/StockCode等、AdjustedClose/AdjustmentClose、Volume/Vo）

出力
- daily_portfolio_returns_monthly_improved.parquet
- leverage_events_monthly_improved.csv（参考：基本は1.0）
- daily_mdd_summary_monthly_improved.csv
- filter_statistics_monthly_improved.csv
"""

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


# ============================================================
# 基本設定
# ============================================================

OUTPUT_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks")
SNAP_PATH = OUTPUT_DIR / "factors" / "month_end_snapshot.parquet"
MERGED_PARTS_DIR = OUTPUT_DIR / "merged_parts"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_daily"
ANALYSIS_DIR.mkdir(exist_ok=True)


# ============================================================
# パラメータ（改善版）
# ============================================================

### 基本
INITIAL_CAPITAL = 10_000_000
TOP_N = 20
VO_TOP_PCT = 0.70
DEFAULT_TRADING_UNIT = 100

### 株価・財務フィルタ
MIN_PRICE = 800
MAX_PRICE = None
ROE_MIN = -0.5
ROE_MAX = 1.0
BM_RATIO_MIN = 0.1
BM_RATIO_MAX = 10.0

### リスクオフ判定（市場状態）
OFF_TH = -0.115
ON_TH = -0.085
TREND3_TH = -0.02
VOL3_Q = 0.90

### 取引コスト（片道）
COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.003
TOTAL_COST_PER_TRADE = COMMISSION_RATE + SLIPPAGE_RATE  # 片道0.4%

### 流動性制約
MAX_VOLUME_PARTICIPATION = 0.10
MIN_DAILY_TURNOVER = 5_000_000

### 価格ギャップ対策（執行可能性の安全側）
PRICE_GAP_BUFFER = 0.95  # 5%安全側
ENTRY_SLIPPAGE = 0.01

### 改善版：レバレッジ固定（信用金利ゼロ）
LEVERAGE_FIXED = 1.0
MARGIN_INTEREST_RATE = 0.028  # 参考（改善版では使わない）

### 改善版：部分リバランス（デッドバンド）と条件付きリバランス
WEIGHT_ADJUST_DEADBAND = 0.05          # 同一銘柄のウェイト差5%未満は調整しない
REBALANCE_MIN_TURNOVER_STOCKS = 5      # 入替銘柄数がこれ未満かつturnover小なら見送り候補
REBALANCE_MIN_TURNOVER_PCT = 0.10      # turnoverがこれ未満なら見送り候補

print("=" * 80)
print("月次戦略 改善版バックテスト（CASH完全対応 + 列名自動検出）")
print("前提：月末引けで決定 → 翌月初寄りで執行（コストは翌月第1営業日に計上）")
print("=" * 80)
print(f"初期資金: ¥{INITIAL_CAPITAL:,}")
print(f"TOP_N: {TOP_N}")
print(f"株価フィルタ: ¥{MIN_PRICE}以上")
print(f"売買代金上位: {VO_TOP_PCT*100:.0f}%")
print(f"最低日次売買代金: ¥{MIN_DAILY_TURNOVER:,}")
print(f"取引コスト（片道）: {TOTAL_COST_PER_TRADE*100:.2f}%（ターンオーバー比例）")
print(f"レバレッジ: {LEVERAGE_FIXED:.2f}x（改善版：信用取引なし）")
print(f"部分リバランス deadband: {WEIGHT_ADJUST_DEADBAND*100:.0f}%")
print("=" * 80)


# ============================================================
# 列名自動検出・正規化
# ============================================================

def detect_stock_code_column(df):
    candidates = ['Code', 'StockCode', 'Symbol', 'Ticker', 'stock_code', 'code']
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"銘柄コード列が見つかりません。利用可能な列: {list(df.columns)}")

def normalize_columns(df, stock_code_col=None):
    rename_map = {}
    if stock_code_col and stock_code_col != 'Code':
        rename_map[stock_code_col] = 'Code'
    if 'AdjustedClose' in df.columns and 'AdjustmentClose' not in df.columns:
        rename_map['AdjustedClose'] = 'AdjustmentClose'
    if 'Volume' in df.columns and 'Vo' not in df.columns:
        rename_map['Volume'] = 'Vo'
    return df.rename(columns=rename_map)


# ============================================================
# Step 1: 月次スナップショット読み込み
# ============================================================

print("\n[1/8] 月次スナップショット読み込み中（列名自動検出）...")
snap = pd.read_parquet(SNAP_PATH)
snap['MonthEnd'] = pd.to_datetime(snap['MonthEnd'])

print(f"📋 検出列（先頭10列）: {list(snap.columns[:10])}")

stock_code_col = detect_stock_code_column(snap)
print(f"✅ 銘柄コード列: '{stock_code_col}' → 'Code'")
snap = normalize_columns(snap, stock_code_col)

required_cols = ['MonthEnd', 'Code', 'AdjustmentClose', 'Vo', 'MarketCap', 'BM_Ratio', 'ROE', 'INV_Growth']
missing_cols = [c for c in required_cols if c not in snap.columns]
if missing_cols:
    print(f"⚠️ 不足列: {missing_cols}")
    if 'AdjustmentClose' not in snap.columns:
        if 'Close' in snap.columns:
            snap['AdjustmentClose'] = snap['Close']
            print("  代替: Close → AdjustmentClose")
        elif 'AdjustedClose' in snap.columns:
            snap['AdjustmentClose'] = snap['AdjustedClose']
            print("  代替: AdjustedClose → AdjustmentClose")
    if 'Vo' not in snap.columns and 'Volume' in snap.columns:
        snap['Vo'] = snap['Volume']
        print("  代替: Volume → Vo")
    missing_cols = [c for c in required_cols if c not in snap.columns]
    if missing_cols:
        raise ValueError(f"必須列が不足しており代替もできません: {missing_cols}")

snap = snap.sort_values(['Code', 'MonthEnd']).copy()
snap['MonthKey'] = snap['MonthEnd'].dt.to_period('M').astype(str)

print("✅ 列名正規化完了")
print(f"✅ 期間: {snap['MonthEnd'].min().date()} 〜 {snap['MonthEnd'].max().date()}")
print(f"✅ データ: {len(snap):,} 行, {snap['Code'].nunique():,} 銘柄")


# ============================================================
# Step 2: 月次ポートフォリオ形成（翌月のターゲットを決める）
# ============================================================

print("\n[2/8] 月次ポートフォリオ形成中（翌月ターゲット決定、CASH対応）...")

# 翌月リターン（参考：市場状態の計算用）
snap['ret_m_fwd'] = snap.groupby('Code')['AdjustmentClose'].pct_change().shift(-1)
snap['TurnoverValue'] = snap['Vo'] * snap['AdjustmentClose']

monthly = snap.groupby('MonthEnd').apply(
    lambda g: pd.Series({
        'mkt_vw': np.average(g['ret_m_fwd'].fillna(0), weights=g['MarketCap'].fillna(1)),
        'count': len(g)
    })
).reset_index()

monthly['cum_ret'] = (1 + monthly['mkt_vw']).cumprod()
monthly['peak'] = monthly['cum_ret'].cummax()
monthly['mkt_dd'] = (monthly['cum_ret'] / monthly['peak']) - 1
monthly['trend3'] = monthly['mkt_vw'].rolling(3, min_periods=1).mean()
monthly['vol3'] = monthly['mkt_vw'].rolling(3, min_periods=2).std()

vol3_th = monthly['vol3'].quantile(VOL3_Q)
print(f"✅ Vol3閾値（{VOL3_Q*100:.0f}%ile）: {vol3_th:.6f}")

snap = snap.merge(monthly[['MonthEnd', 'mkt_dd', 'trend3', 'vol3']], on='MonthEnd', how='left')
snap['risk_off'] = (
    (snap['mkt_dd'] <= OFF_TH) |
    (snap['trend3'] <= TREND3_TH) |
    (snap['vol3'] >= vol3_th)
)

portfolio_list = []
filter_stats = []
risk_off_months = 0

for month_end, df_month in snap.groupby('MonthEnd'):
    month_key = df_month['MonthKey'].iloc[0]
    risk_off = bool(df_month['risk_off'].iloc[0])

    stats = {
        'MonthEnd': month_end,
        'MonthKey': month_key,
        'initial_count': len(df_month),
        'risk_off': risk_off
    }

    # ★月末引けで決定 → 翌月に適用
    month_key_next = (pd.Timestamp(month_end) + pd.DateOffset(months=1)).to_period('M').strftime('%Y-%m')

    if risk_off:
        risk_off_months += 1
        portfolio_list.append({
            'MonthEnd': month_end,
            'MonthKey': month_key,
            'MonthKey_next': month_key_next,
            'Code': 'CASH',
            'Weight': 1.0,
            'risk_off': True
        })
        stats.update({
            'after_price': 0,
            'after_roe_bm': 0,
            'after_turnover': 0,
            'after_liquidity': 0,
            'final_count': 0,
            'selected_count': 0
        })
        filter_stats.append(stats)
        continue

    df_valid = df_month[df_month['AdjustmentClose'] >= MIN_PRICE].copy()
    if MAX_PRICE:
        df_valid = df_valid[df_valid['AdjustmentClose'] <= MAX_PRICE
                            ]
    stats['after_price'] = len(df_valid)

    df_valid = df_valid[
        (df_valid['BM_Ratio'].notna()) &
        (df_valid['ROE'].notna()) &
        (df_valid['ROE'] >= ROE_MIN) &
        (df_valid['ROE'] <= ROE_MAX) &
        (df_valid['BM_Ratio'] >= BM_RATIO_MIN) &
        (df_valid['BM_Ratio'] <= BM_RATIO_MAX)
    ]
    stats['after_roe_bm'] = len(df_valid)

    if 'TurnoverValue' in df_valid.columns and len(df_valid) > 0:
        turnover_th = df_valid['TurnoverValue'].quantile(1 - VO_TOP_PCT)
        df_valid = df_valid[df_valid['TurnoverValue'] >= turnover_th]
    stats['after_turnover'] = len(df_valid)

    df_valid = df_valid[df_valid['TurnoverValue'] >= MIN_DAILY_TURNOVER]

    df_valid['MaxBuyableAmount'] = df_valid['Vo'] * MAX_VOLUME_PARTICIPATION * df_valid['AdjustmentClose']

    # 改善版：レバレッジ固定1.0前提で流動性チェック
    target_capital = INITIAL_CAPITAL * LEVERAGE_FIXED * PRICE_GAP_BUFFER
    capital_per_stock = target_capital / TOP_N
    df_valid = df_valid[df_valid['MaxBuyableAmount'] >= capital_per_stock * 0.8]
    stats['after_liquidity'] = len(df_valid)

    if len(df_valid) < TOP_N:
        # 銘柄が揃わない月は翌月CASH
        portfolio_list.append({
            'MonthEnd': month_end,
            'MonthKey': month_key,
            'MonthKey_next': month_key_next,
            'Code': 'CASH',
            'Weight': 1.0,
            'risk_off': False
        })
        stats.update({
            'final_count': len(df_valid),
            'selected_count': 0
        })
        filter_stats.append(stats)
        continue

    # z-score
    for col in ['BM_Ratio', 'ROE', 'INV_Growth']:
        mean_val = df_valid[col].mean()
        std_val = df_valid[col].std()
        df_valid[f'{col}_z'] = (df_valid[col] - mean_val) / std_val if std_val and std_val > 0 else 0.0

    df_valid['composite_score'] = (
        df_valid['BM_Ratio_z'] +
        df_valid['ROE_z'] -
        df_valid['INV_Growth_z'].fillna(0)
    )

    top_n = df_valid.nlargest(TOP_N, 'composite_score')

    stats['final_count'] = len(df_valid)
    stats['selected_count'] = len(top_n)
    filter_stats.append(stats)

    weight = 1.0 / len(top_n)

    for _, row in top_n.iterrows():
        portfolio_list.append({
            'MonthEnd': month_end,
            'MonthKey': month_key,
            'MonthKey_next': month_key_next,  # 翌月に適用
            'Code': row['Code'],
            'Weight': weight,
            'risk_off': False
        })

df_portfolio = pd.DataFrame(portfolio_list)
df_filter_stats = pd.DataFrame(filter_stats)

total_months = len(df_filter_stats)
cash_months_cnt = df_portfolio[df_portfolio['Code'] == 'CASH']['MonthKey_next'].nunique()

print(f"✅ ポートフォリオ形成完了: {len(df_portfolio):,} レコード")
print(f"✅ リスクオフ月数: {risk_off_months}/{total_months} ({risk_off_months/total_months*100:.1f}%)")
print(f"✅ 翌月CASH月数: {cash_months_cnt}/{df_portfolio['MonthKey_next'].nunique()} ({cash_months_cnt/df_portfolio['MonthKey_next'].nunique()*100:.1f}%)")

print("\n[新] フィルタ統計（全期間平均）:")
print(f"  初期: {df_filter_stats['initial_count'].mean():.0f}")
print(f"  株価後: {df_filter_stats['after_price'].mean():.0f}")
print(f"  財務後: {df_filter_stats['after_roe_bm'].mean():.0f}")
print(f"  売買後: {df_filter_stats['after_turnover'].mean():.0f}")
print(f"  流動性後: {df_filter_stats['after_liquidity'].mean():.0f}")
print(f"  最終選定: {df_filter_stats['selected_count'].mean():.0f}")


# ============================================================
# Step 3: 日次データ読み込み
# ============================================================

print("\n[3/8] 日次データ読み込み中（列名自動検出）...")

daily_files = sorted(MERGED_PARTS_DIR.glob("merged-part-*.parquet"))
if not daily_files:
    raise FileNotFoundError(f"日次データが見つかりません: {MERGED_PARTS_DIR}")

sample_df = pd.read_parquet(daily_files[0])
daily_stock_code_col = detect_stock_code_column(sample_df)
print(f"✅ 日次データの銘柄コード列: '{daily_stock_code_col}' → 'Code'")

daily_list = []
for f in daily_files:
    df_part = pd.read_parquet(f)
    df_part = normalize_columns(df_part, daily_stock_code_col)
    daily_list.append(df_part)

daily = pd.concat(daily_list, ignore_index=True)
daily['Date'] = pd.to_datetime(daily['Date'])
daily = daily.sort_values(['Code', 'Date']).copy()

# 念のため
if 'AdjustedClose' in daily.columns and 'AdjustmentClose' not in daily.columns:
    daily = daily.rename(columns={'AdjustedClose': 'AdjustmentClose'})

daily['ret_d'] = daily.groupby('Code')['AdjustmentClose'].pct_change()
daily['MonthKey'] = daily['Date'].dt.to_period('M').astype(str)

print(f"✅ 日次データ: {len(daily):,} 行, {daily['Code'].nunique():,} 銘柄")


# ============================================================
# Step 4: 日次ポートフォリオリターン計算（改善版）
# 前提：月末で決定した MonthKey_next を、翌月の第1営業日で執行（コスト計上）
# ============================================================

print("\n[4/8] 日次ポートフォリオリターン計算中（改善版：月末決定→翌月初寄り執行）...")

# CASH月マスタ（翌月にCASHを適用）
cash_tbl = df_portfolio[df_portfolio['Code'] == 'CASH'][['MonthKey_next', 'risk_off']].copy()
cash_tbl = cash_tbl.rename(columns={'MonthKey_next': 'MonthKey'})
cash_tbl['is_cash_month'] = True
cash_month_set = set(cash_tbl['MonthKey'].unique())

# 株式ポート（翌月に適用）
stock_portfolio = df_portfolio[df_portfolio['Code'] != 'CASH'].copy()

# 月次ターゲット辞書（翌月）
monthly_target = {}
for mk, g in stock_portfolio.groupby('MonthKey_next'):
    monthly_target[mk] = dict(zip(g['Code'].astype(str), g['Weight'].astype(float)))

# 日次に月次ターゲットWeightを付与（株式ポートのみ）
daily_port = daily.merge(
    stock_portfolio[['MonthKey_next', 'Code', 'Weight']],
    left_on=['MonthKey', 'Code'],
    right_on=['MonthKey_next', 'Code'],
    how='inner'
)

daily_port = daily_port.sort_values('Date').copy()

# 翌月第1営業日（=そのMonthKeyに属する最初の日付）を求める
first_trade_day_of_month = daily.groupby('MonthKey')['Date'].min().to_dict()

def calc_turnover_and_new_hold(prev_w, target_w, deadband=WEIGHT_ADJUST_DEADBAND):
    prev = prev_w.copy()
    target = target_w.copy()
    all_codes = set(prev.keys()) | set(target.keys())
    after = {}

    for c in all_codes:
        w0 = prev.get(c, 0.0)
        w1 = target.get(c, 0.0)
        if (c in prev) and (c in target):
            if abs(w1 - w0) < deadband:
                after[c] = w0
            else:
                after[c] = w1
        else:
            after[c] = w1

    after = {c: w for c, w in after.items() if w > 0}
    s = sum(after.values())
    if s > 0:
        after = {c: w/s for c, w in after.items()}

    codes2 = set(prev.keys()) | set(after.keys())
    turnover = sum(abs(after.get(c, 0.0) - prev.get(c, 0.0)) for c in codes2)
    turnover_stocks = len(set(prev.keys()) ^ set(after.keys()))
    return turnover, turnover_stocks, after

global_wealth = INITIAL_CAPITAL
global_peak = INITIAL_CAPITAL
current_dd = 0.0

current_hold_weights = {}  # 実際に保持しているウェイト（近似）
daily_returns = []
leverage_events = []

rebalance_count = 0
skip_rebalance_count = 0
total_trading_cost = 0.0
total_interest_cost = 0.0  # 改善版は基本0

all_dates = sorted(daily['Date'].unique())

for date in all_dates:
    mk = pd.Timestamp(date).to_period('M').strftime('%Y-%m')

    # CASH判定（その月は現金か）
    is_cash_month = mk in cash_month_set
    is_risk_off = False
    if is_cash_month:
        tmp = cash_tbl[cash_tbl['MonthKey'] == mk]
        if len(tmp) > 0:
            is_risk_off = bool(tmp['risk_off'].iloc[0])
    is_cash = is_cash_month or is_risk_off

    # 当日がその月の第1営業日か（=翌月初寄り執行日として扱う）
    is_first_trade_day = (mk in first_trade_day_of_month) and (pd.Timestamp(date) == pd.Timestamp(first_trade_day_of_month[mk]))

    trading_cost = 0.0
    interest_cost_pct = 0.0
    leverage_used = LEVERAGE_FIXED

    if is_cash:
        # CASH：日次リターンはゼロ
        port_ret_gross = 0.0
        port_ret_net = 0.0

        # ただし、CASH → 株式に戻る月の初日には（ターゲットがあるなら）売買が発生する想定だが、
        # is_cash=Trueの月は“株式を持たない”ため、ここでは売買しない（翌月に株式ターゲットがある月で実行される）
    else:
        # 株式ターゲットの当日リターン（ターゲットWeight近似）
        df_day = daily_port[daily_port['Date'] == date].copy()
        if len(df_day) == 0:
            # データ欠損日はスキップ
            continue

        # 当日のターゲットベースの加重リターン
        df_day['weighted_ret'] = df_day['ret_d'].fillna(0) * df_day['Weight']
        port_ret_gross = df_day['weighted_ret'].sum() * leverage_used

        # ★翌月初（その月の第1営業日）にリバランス（コスト計上＋保有ウェイト更新）
        if is_first_trade_day:
            target_w = monthly_target.get(mk, {})
            if len(target_w) == 0:
                # ターゲットがない場合は実質CASH
                turnover = 0.0
                turnover_stocks = 0
                new_hold = {}
                trading_cost = 0.0
                skip_rebalance_count += 1
            else:
                turnover, turnover_stocks, new_hold = calc_turnover_and_new_hold(current_hold_weights, target_w)

                # 条件付きリバランス（抑制）
                if (turnover_stocks < REBALANCE_MIN_TURNOVER_STOCKS) and (turnover < REBALANCE_MIN_TURNOVER_PCT):
                    # 見送り：保有ウェイト維持（コスト0）
                    skip_rebalance_count += 1
                    trading_cost = 0.0
                    # 厳密には「保有ウェイトで当日リターン」を計算すべきだが、
                    # ここでは改善版の簡略化としてターゲットWeightでのリターンを維持（差は通常小さい想定）
                else:
                    # 実行：ターンオーバー比例でコスト
                    rebalance_count += 1
                    trading_cost = turnover * TOTAL_COST_PER_TRADE
                    total_trading_cost += trading_cost
                    current_hold_weights = new_hold

        port_ret_net = port_ret_gross - trading_cost - interest_cost_pct

    # wealth更新
    global_wealth *= (1 + port_ret_net)
    if global_wealth > global_peak:
        global_peak = global_wealth
        leverage_events.append({'Date': date, 'Event': 'NewPeak', 'Wealth': global_wealth, 'Leverage': leverage_used})

    current_dd = (global_wealth / global_peak) - 1

    daily_returns.append({
        'Date': date,
        'MonthKey': mk,
        'port_ret_gross': port_ret_gross,
        'port_ret_net': port_ret_net,
        'trading_cost': trading_cost,
        'interest_cost': interest_cost_pct,
        'leverage': leverage_used,
        'wealth': global_wealth,
        'peak': global_peak,
        'dd': current_dd,
        'is_cash': is_cash,
        'is_risk_off': is_risk_off,
        'is_cash_month': is_cash_month,
        'is_rebalance_day': bool(is_first_trade_day and (not is_cash))
    })

df_daily_returns = pd.DataFrame(daily_returns)

print(f"\n✅ 日次ポートフォリオリターン: {len(df_daily_returns):,} 日")
print(f"✅ リバランス実行回数: {rebalance_count} 回（翌月第1営業日ベース）")
print(f"✅ リバランス見送り回数: {skip_rebalance_count} 回")
print(f"✅ 信用金利合計（期待0）: {total_interest_cost*100:.4f}%")
print(f"✅ 取引コスト合計（ターンオーバー比例）: {total_trading_cost*100:.2f}%")


# ============================================================
# Step 5: 日次MDD計算とサマリ
# ============================================================

print("\n[5/8] 日次MDD計算中...")

df_daily_returns['cum_ret'] = df_daily_returns['port_ret_net'].add(1).cumprod().sub(1)
df_daily_returns['cum_wealth'] = INITIAL_CAPITAL * (1 + df_daily_returns['cum_ret'])

total_days = len(df_daily_returns)
final_wealth = df_daily_returns['cum_wealth'].iloc[-1]
cum_return = (final_wealth / INITIAL_CAPITAL) - 1

first_date = df_daily_returns['Date'].min()
last_date = df_daily_returns['Date'].max()
years = (last_date - first_date).days / 365.25 if last_date > first_date else 1.0

ann_return = (1 + cum_return) ** (1 / years) - 1
ann_vol = df_daily_returns['port_ret_net'].std() * np.sqrt(252)
sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0

daily_mdd = df_daily_returns['dd'].min()
worst_dd_date = df_daily_returns.loc[df_daily_returns['dd'].idxmin(), 'Date']

cash_days = int(df_daily_returns['is_cash'].sum())
cash_pct = cash_days / total_days * 100
avg_leverage = df_daily_returns['leverage'].mean()

# コストドラッグ（年率差分も併記）
total_gross_return = df_daily_returns['port_ret_gross'].add(1).cumprod().iloc[-1] - 1
total_net_return = cum_return
gross_ann = (1 + total_gross_return) ** (1 / years) - 1
net_ann = (1 + total_net_return) ** (1 / years) - 1
annual_cost_drag = gross_ann - net_ann

print("\n" + "=" * 80)
print("最終結果サマリー（月次改善版）")
print("=" * 80)
print(f"期間: {first_date.date()} 〜 {last_date.date()}")
print(f"総日数: {total_days}日 ({years:.2f}年)")
print(f"初期資金: ¥{INITIAL_CAPITAL:,}")
print(f"最終資産: ¥{final_wealth:,.0f}")
print(f"累積リターン: {cum_return*100:.2f}%")
print(f"年率リターン: {ann_return*100:.2f}%")
print(f"年率ボラティリティ: {ann_vol*100:.2f}%")
print(f"シャープレシオ: {sharpe:.4f}")
print(f"日次最大DD: {daily_mdd*100:.2f}% (日付: {worst_dd_date.date()})")
print(f"平均CASH比率: {cash_pct:.2f}%")
print(f"平均レバレッジ: {avg_leverage:.2f}x（改善版は原則1.0）")

print("\n[改善版] 取引コストの影響:")
print(f"  グロス累積リターン: {total_gross_return*100:.2f}%")
print(f"  ネット累積リターン: {total_net_return*100:.2f}%")
print(f"  年率コストドラッグ（年率差分）: {annual_cost_drag*100:.2f}%/年")
print(f"  取引コスト累積（ターンオーバー比例、足し上げ）: {total_trading_cost*100:.2f}%")
print("=" * 80)

print("\nワースト5ドローダウン日（CASH除外）:")
worst_days = df_daily_returns[~df_daily_returns['is_cash']].nsmallest(5, 'dd')
for _, row in worst_days.iterrows():
    print(f"  {row['Date'].date()}: dd {row['dd']:.6f}, port_ret {row['port_ret_net']:.6f}, rebalance_day={row['is_rebalance_day']}")


# ============================================================
# Step 6: ファイル保存
# ============================================================

print("\n[6/8] ファイル保存中...")

out_file1 = ANALYSIS_DIR / "daily_portfolio_returns_monthly_improved.parquet"
df_daily_returns.to_parquet(out_file1, index=False)
print(f"  ✅ {out_file1}")

if leverage_events:
    df_leverage_events = pd.DataFrame(leverage_events)
    out_file2 = ANALYSIS_DIR / "leverage_events_monthly_improved.csv"
    df_leverage_events.to_csv(out_file2, index=False, encoding='utf-8-sig')
    print(f"  ✅ {out_file2}")

summary_data = {
    'period_start': [first_date],
    'period_end': [last_date],
    'total_days': [total_days],
    'initial_capital': [INITIAL_CAPITAL],
    'final_wealth': [final_wealth],
    'cumulative_return': [cum_return],
    'annual_return': [ann_return],
    'annual_volatility': [ann_vol],
    'sharpe_ratio': [sharpe],
    'daily_max_dd': [daily_mdd],
    'worst_dd_date': [worst_dd_date],
    'cash_days': [cash_days],
    'cash_pct': [cash_pct],
    'avg_leverage': [avg_leverage],
    'gross_return': [total_gross_return],
    'net_return': [total_net_return],
    'annual_cost_drag': [annual_cost_drag],
    'rebalance_count': [rebalance_count],
    'skip_rebalance_count': [skip_rebalance_count],
    'total_trading_cost_sum': [total_trading_cost],
    'total_interest_cost_sum': [total_interest_cost],
}
df_summary = pd.DataFrame(summary_data)
out_file3 = ANALYSIS_DIR / "daily_mdd_summary_monthly_improved.csv"
df_summary.to_csv(out_file3, index=False, encoding='utf-8-sig')
print(f"  ✅ {out_file3}")

out_file4 = ANALYSIS_DIR / "filter_statistics_monthly_improved.csv"
df_filter_stats.to_csv(out_file4, index=False, encoding='utf-8-sig')
print(f"  ✅ {out_file4}")


# ============================================================
# Step 7: 比較表作成（簡易：今回の改善版のみ、必要なら旧版を追記可）
# ============================================================

print("\n[7/8] 戦略比較表作成中...")

comparison_data = {
    '指標': [
        '年率リターン',
        '年率ボラティリティ',
        'シャープレシオ',
        '最大DD',
        '累積リターン',
        'CASH比率',
        '平均レバレッジ',
        '年率コストドラッグ（年率差分）',
        '取引コスト累積（足し上げ）',
        'リバランス実行回数',
        'リバランス見送り回数'
    ],
    '月次改善版（今回）': [
        f'{ann_return*100:.2f}%',
        f'{ann_vol*100:.2f}%',
        f'{sharpe:.4f}',
        f'{daily_mdd*100:.2f}%',
        f'{cum_return*100:.2f}%',
        f'{cash_pct:.2f}%',
        f'{avg_leverage:.2f}x',
        f'{annual_cost_drag*100:.2f}%/年',
        f'{total_trading_cost*100:.2f}%',
        f'{rebalance_count}',
        f'{skip_rebalance_count}'
    ]
}
df_comparison = pd.DataFrame(comparison_data)

print("\n" + "=" * 80)
print("戦略比較表（今回：月次改善版）")
print("=" * 80)
print(df_comparison.to_string(index=False))
print("=" * 80)


# ============================================================
# Step 8: 完了メッセージ
# ============================================================

print("\n" + "=" * 80)
print("[8/8] 月次戦略 改善版バックテスト完了！ ✅")
print("=" * 80)

print("\n次のステップ（確認ポイント）:")
print("1) 取引コスト累積（total_trading_cost_sum）が旧月次より大幅に下がったか")
print("2) 年率コストドラッグ（年率差分）が 5%/年前後まで下がったか")
print("3) 最大DDが悪化していないか（-25%以内が目安）")
print("4) シャープが改善しているか（≥1.2〜1.4目標）")
print("5) 追加改善：Open列があれば『翌月初寄りリターン』を厳密化可能")
print()


月次戦略 改善版バックテスト（CASH完全対応 + 列名自動検出）
前提：月末引けで決定 → 翌月初寄りで執行（コストは翌月第1営業日に計上）
初期資金: ¥10,000,000
TOP_N: 20
株価フィルタ: ¥800以上
売買代金上位: 70%
最低日次売買代金: ¥5,000,000
取引コスト（片道）: 0.40%（ターンオーバー比例）
レバレッジ: 1.00x（改善版：信用取引なし）
部分リバランス deadband: 5%

[1/8] 月次スナップショット読み込み中（列名自動検出）...
📋 検出列（先頭10列）: ['MonthEnd', 'Code', 'AdjustedClose', 'Vo', 'MarketCap', 'BM_Ratio', 'ROE', 'INV_Growth', 'Month']
✅ 銘柄コード列: 'Code' → 'Code'
✅ 列名正規化完了
✅ 期間: 2016-01-31 〜 2026-01-31
✅ データ: 502,101 行, 5,303 銘柄

[2/8] 月次ポートフォリオ形成中（翌月ターゲット決定、CASH対応）...
✅ Vol3閾値（90%ile）: 0.064522
✅ ポートフォリオ形成完了: 1,812 レコード
✅ リスクオフ月数: 32/121 (26.4%)
✅ 翌月CASH月数: 32/121 (26.4%)

[新] フィルタ統計（全期間平均）:
  初期: 4150
  株価後: 2055
  財務後: 1645
  売買後: 1151
  流動性後: 1151
  最終選定: 15

[3/8] 日次データ読み込み中（列名自動検出）...
✅ 日次データの銘柄コード列: 'Code' → 'Code'
✅ 日次データ: 10,016,062 行, 5,303 銘柄

[4/8] 日次ポートフォリオリターン計算中（改善版：月末決定→翌月初寄り執行）...

✅ 日次ポートフォリオリターン: 2,430 日
✅ リバランス実行回数: 88 回（翌月第1営業日ベース）
✅ リバランス見送り回数: 0 回
✅ 信用金利合計（期待0）: 0.0000%
✅ 取引コスト合計（ターンオーバー比例）: 27.00%

[5/8] 日次MDD計算中...

最終結果サマリー（月次改善版）


In [6]:
"""
月次戦略 改善版（v2）：リアリスティック・バックテスト
（CASH完全対応 + 列名自動検出 + 月末引けで決定→翌月初寄りで執行）
※日次データがAdjustmentCloseのみの場合、寄り執行は厳密再現不可 → 「翌月第1営業日にコスト計上」で執行日整合（価格は日足近似）

v2（改善第2弾：最小差分で追加）
(1) TOP_N=20を満たすため流動性制約を 0.8 → 0.7 に緩和（LIQUIDITY_BUFFER_FACTOR）
(2) スコアを2ヶ月EWMAで平滑化（alpha=0.50）
(3) 銘柄重なり率(overlap)でリバランス見送り判定を追加（閾値0.80）
    ただし CASH↔株式の遷移は必ず実行（見送り禁止）
    既存の条件付きリバランス（turnover条件）と併用（C案）

出力（v2）
- daily_portfolio_returns_monthly_improved_v2.parquet
- leverage_events_monthly_improved_v2.csv（参考）
- daily_mdd_summary_monthly_improved_v2.csv
- filter_statistics_monthly_improved_v2.csv
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 基本設定
# ============================================================

OUTPUT_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks")
SNAP_PATH = OUTPUT_DIR / "factors" / "month_end_snapshot.parquet"
MERGED_PARTS_DIR = OUTPUT_DIR / "merged_parts"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_daily"
ANALYSIS_DIR.mkdir(exist_ok=True)


# ============================================================
# パラメータ（改善版 + v2）
# ============================================================

### 基本
INITIAL_CAPITAL = 10_000_000
TOP_N = 20
VO_TOP_PCT = 0.70
DEFAULT_TRADING_UNIT = 100

### 株価・財務フィルタ
MIN_PRICE = 800
MAX_PRICE = None
ROE_MIN = -0.5
ROE_MAX = 1.0
BM_RATIO_MIN = 0.1
BM_RATIO_MAX = 10.0

### リスクオフ判定（市場状態）※旧コード踏襲
OFF_TH = -0.115
ON_TH = -0.085
TREND3_TH = -0.02
VOL3_Q = 0.90

### 取引コスト（片道）
COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.003
TOTAL_COST_PER_TRADE = COMMISSION_RATE + SLIPPAGE_RATE  # 片道0.4%

### 流動性制約
MAX_VOLUME_PARTICIPATION = 0.10
MIN_DAILY_TURNOVER = 5_000_000

### 価格ギャップ対策（執行可能性の安全側）
PRICE_GAP_BUFFER = 0.95  # 5%安全側
ENTRY_SLIPPAGE = 0.01

### 改善版：レバレッジ固定（信用金利ゼロ）
LEVERAGE_FIXED = 1.0
MARGIN_INTEREST_RATE = 0.028  # 参考（改善版では使わない）

### 改善版：部分リバランス（デッドバンド）と条件付きリバランス（旧コード踏襲）
WEIGHT_ADJUST_DEADBAND = 0.05
REBALANCE_MIN_TURNOVER_STOCKS = 5
REBALANCE_MIN_TURNOVER_PCT = 0.10

# -------------------------
# v2（改善第2弾）追加パラメータ
# -------------------------
LIQUIDITY_BUFFER_FACTOR = 0.70     # (1) 流動性 0.8 -> 0.7
SCORE_EWMA_ALPHA = 0.50            # (2) スコアEWMA alpha
OVERLAP_SKIP_TH = 0.80             # (3) overlap見送り閾値
USE_OVERLAP_SKIP = True            # overlap見送りを有効化


print("=" * 80)
print("月次戦略 改善版バックテスト v2（CASH完全対応 + 列名自動検出）")
print("前提：月末引けで決定 → 翌月初寄りで執行（コストは翌月第1営業日に計上、価格は日足近似）")
print("=" * 80)
print(f"初期資金: ¥{INITIAL_CAPITAL:,}")
print(f"TOP_N: {TOP_N}")
print(f"株価フィルタ: ¥{MIN_PRICE}以上")
print(f"売買代金上位: {VO_TOP_PCT*100:.0f}%")
print(f"最低日次売買代金: ¥{MIN_DAILY_TURNOVER:,}")
print(f"取引コスト（片道）: {TOTAL_COST_PER_TRADE*100:.2f}%（ターンオーバー比例）")
print(f"レバレッジ: {LEVERAGE_FIXED:.2f}x（改善版：信用取引なし）")
print(f"部分リバランス deadband: {WEIGHT_ADJUST_DEADBAND*100:.0f}%")
print("-" * 80)
print("[v2] LIQUIDITY_BUFFER_FACTOR:", LIQUIDITY_BUFFER_FACTOR)
print("[v2] SCORE_EWMA_ALPHA:", SCORE_EWMA_ALPHA)
print("[v2] OVERLAP_SKIP_TH:", OVERLAP_SKIP_TH, "USE_OVERLAP_SKIP:", USE_OVERLAP_SKIP)
print("=" * 80)


# ============================================================
# 列名自動検出・正規化
# ============================================================

def detect_stock_code_column(df):
    candidates = ["Code", "StockCode", "Symbol", "Ticker", "stock_code", "code"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"銘柄コード列が見つかりません。利用可能な列: {list(df.columns)}")

def normalize_columns(df, stock_code_col=None):
    rename_map = {}
    if stock_code_col and stock_code_col != "Code":
        rename_map[stock_code_col] = "Code"
    if "AdjustedClose" in df.columns and "AdjustmentClose" not in df.columns:
        rename_map["AdjustedClose"] = "AdjustmentClose"
    if "Volume" in df.columns and "Vo" not in df.columns:
        rename_map["Volume"] = "Vo"
    return df.rename(columns=rename_map)


# ============================================================
# v2: EWMA平滑化ユーティリティ
# ============================================================

def add_ewma_score_to_snapshot(snap: pd.DataFrame, alpha: float) -> pd.DataFrame:
    """
    旧コードの composite_score（BM+ROE-INV）を month_end 時点で計算し、
    銘柄ごとの時系列でEWMA平滑化した composite_score_ewma を作る。
    フィルタ前に作っておくことで「最小差分」で Step2 の top_n 指標を差し替え可能にする。
    """
    snap = snap.sort_values(["Code", "MonthEnd"]).copy()

    # z-scoreは月内クロスセクション標準化（旧コード踏襲）→ 月内で計算した raw score を時系列EWMA
    def _calc_month_z_and_score(g):
        g = g.copy()
        for col in ["BM_Ratio", "ROE", "INV_Growth"]:
            mean_val = g[col].mean()
            std_val = g[col].std()
            g[f"{col}_z_all"] = (g[col] - mean_val) / std_val if (std_val is not None and std_val > 0) else 0.0
        g["composite_score_raw_all"] = (
            g["BM_Ratio_z_all"] +
            g["ROE_z_all"] -
            g["INV_Growth_z_all"].fillna(0)
        )
        return g

    snap = snap.groupby("MonthEnd", group_keys=False).apply(_calc_month_z_and_score)

    # 銘柄ごとのEWMA（時系列）
    snap["composite_score_ewma_all"] = (
        snap.groupby("Code")["composite_score_raw_all"]
            .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
    )

    return snap


def overlap_ratio(prev_codes, new_codes) -> float:
    prev_set = set(prev_codes)
    new_set = set(new_codes)
    if len(prev_set) == 0:
        return 0.0
    return len(prev_set & new_set) / max(len(prev_set), 1)


# ============================================================
# Step 1: 月次スナップショット読み込み
# ============================================================

print("\n[1/8] 月次スナップショット読み込み中（列名自動検出）...")
snap = pd.read_parquet(SNAP_PATH)
snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])

print(f"検出列（先頭10列）: {list(snap.columns[:10])}")

stock_code_col = detect_stock_code_column(snap)
print(f"銘柄コード列: '{stock_code_col}' → 'Code'")
snap = normalize_columns(snap, stock_code_col)

required_cols = ["MonthEnd", "Code", "AdjustmentClose", "Vo", "MarketCap", "BM_Ratio", "ROE", "INV_Growth"]
missing_cols = [c for c in required_cols if c not in snap.columns]
if missing_cols:
    print(f"不足列: {missing_cols}")
    if "AdjustmentClose" not in snap.columns:
        if "Close" in snap.columns:
            snap["AdjustmentClose"] = snap["Close"]
            print("  代替: Close → AdjustmentClose")
        elif "AdjustedClose" in snap.columns:
            snap["AdjustmentClose"] = snap["AdjustedClose"]
            print("  代替: AdjustedClose → AdjustmentClose")
    if "Vo" not in snap.columns and "Volume" in snap.columns:
        snap["Vo"] = snap["Volume"]
        print("  代替: Volume → Vo")
    missing_cols = [c for c in required_cols if c not in snap.columns]
    if missing_cols:
        raise ValueError(f"必須列が不足しており代替もできません: {missing_cols}")

snap = snap.sort_values(["Code", "MonthEnd"]).copy()
snap["MonthKey"] = snap["MonthEnd"].dt.to_period("M").astype(str)

print("列名正規化完了")
print(f"期間: {snap['MonthEnd'].min().date()} 〜 {snap['MonthEnd'].max().date()}")
print(f"データ: {len(snap):,} 行, {snap['Code'].nunique():,} 銘柄")


# ============================================================
# v2: スコアEWMAをスナップショットに付与（最小差分で追加）
# ============================================================
print("\n[v2] スコアEWMA（2ヶ月相当）を計算中...")
snap = add_ewma_score_to_snapshot(snap, alpha=SCORE_EWMA_ALPHA)
print("  composite_score_ewma_all を追加しました")


# ============================================================
# Step 2: 月次ポートフォリオ形成（翌月のターゲットを決める）
# ============================================================

print("\n[2/8] 月次ポートフォリオ形成中（翌月ターゲット決定、CASH対応）...")

# 翌月リターン（参考：市場状態の計算用）
snap["ret_m_fwd"] = snap.groupby("Code")["AdjustmentClose"].pct_change().shift(-1)
snap["TurnoverValue"] = snap["Vo"] * snap["AdjustmentClose"]

monthly = snap.groupby("MonthEnd").apply(
    lambda g: pd.Series({
        "mkt_vw": np.average(g["ret_m_fwd"].fillna(0), weights=g["MarketCap"].fillna(1)),
        "count": len(g)
    })
).reset_index()

monthly["cum_ret"] = (1 + monthly["mkt_vw"]).cumprod()
monthly["peak"] = monthly["cum_ret"].cummax()
monthly["mkt_dd"] = (monthly["cum_ret"] / monthly["peak"]) - 1
monthly["trend3"] = monthly["mkt_vw"].rolling(3, min_periods=1).mean()
monthly["vol3"] = monthly["mkt_vw"].rolling(3, min_periods=2).std()

vol3_th = monthly["vol3"].quantile(VOL3_Q)
print(f"Vol3閾値（{VOL3_Q*100:.0f}%ile）: {vol3_th:.6f}")

snap = snap.merge(monthly[["MonthEnd", "mkt_dd", "trend3", "vol3"]], on="MonthEnd", how="left")
snap["risk_off"] = (
    (snap["mkt_dd"] <= OFF_TH) |
    (snap["trend3"] <= TREND3_TH) |
    (snap["vol3"] >= vol3_th)
)

portfolio_list = []
filter_stats = []
risk_off_months = 0

for month_end, df_month in snap.groupby("MonthEnd"):
    month_key = df_month["MonthKey"].iloc[0]
    risk_off = bool(df_month["risk_off"].iloc[0])

    stats = {
        "MonthEnd": month_end,
        "MonthKey": month_key,
        "initial_count": len(df_month),
        "risk_off": risk_off
    }

    # 月末引けで決定 → 翌月適用
    month_key_next = (pd.Timestamp(month_end) + pd.DateOffset(months=1)).to_period("M").strftime("%Y-%m")

    if risk_off:
        risk_off_months += 1
        portfolio_list.append({
            "MonthEnd": month_end,
            "MonthKey": month_key,
            "MonthKey_next": month_key_next,
            "Code": "CASH",
            "Weight": 1.0,
            "risk_off": True
        })
        stats.update({
            "after_price": 0,
            "after_roe_bm": 0,
            "after_turnover": 0,
            "after_liquidity": 0,
            "final_count": 0,
            "selected_count": 0
        })
        filter_stats.append(stats)
        continue

    df_valid = df_month[df_month["AdjustmentClose"] >= MIN_PRICE].copy()
    if MAX_PRICE:
        df_valid = df_valid[df_valid["AdjustmentClose"] <= MAX_PRICE]
    stats["after_price"] = len(df_valid)

    df_valid = df_valid[
        (df_valid["BM_Ratio"].notna()) &
        (df_valid["ROE"].notna()) &
        (df_valid["ROE"] >= ROE_MIN) &
        (df_valid["ROE"] <= ROE_MAX) &
        (df_valid["BM_Ratio"] >= BM_RATIO_MIN) &
        (df_valid["BM_Ratio"] <= BM_RATIO_MAX)
    ]
    stats["after_roe_bm"] = len(df_valid)

    if "TurnoverValue" in df_valid.columns and len(df_valid) > 0:
        turnover_th = df_valid["TurnoverValue"].quantile(1 - VO_TOP_PCT)
        df_valid = df_valid[df_valid["TurnoverValue"] >= turnover_th]
    stats["after_turnover"] = len(df_valid)

    df_valid = df_valid[df_valid["TurnoverValue"] >= MIN_DAILY_TURNOVER]

    df_valid["MaxBuyableAmount"] = df_valid["Vo"] * MAX_VOLUME_PARTICIPATION * df_valid["AdjustmentClose"]

    # (1) v2: 流動性制約 0.8 -> 0.7
    target_capital = INITIAL_CAPITAL * LEVERAGE_FIXED * PRICE_GAP_BUFFER
    capital_per_stock = target_capital / TOP_N
    df_valid = df_valid[df_valid["MaxBuyableAmount"] >= capital_per_stock * LIQUIDITY_BUFFER_FACTOR]
    stats["after_liquidity"] = len(df_valid)

    if len(df_valid) < TOP_N:
        # 銘柄が揃わない月は翌月CASH
        portfolio_list.append({
            "MonthEnd": month_end,
            "MonthKey": month_key,
            "MonthKey_next": month_key_next,
            "Code": "CASH",
            "Weight": 1.0,
            "risk_off": False
        })
        stats.update({
            "final_count": len(df_valid),
            "selected_count": 0
        })
        filter_stats.append(stats)
        continue

    # (2) v2: スコアは EWMA を優先して使用（フィルタ後の df_valid から選ぶ）
    # ※EWMA列は snap全体で作ってあるので df_valid には composite_score_ewma_all が入っている
    score_col = "composite_score_ewma_all"

    top_n = df_valid.nlargest(TOP_N, score_col)

    stats["final_count"] = len(df_valid)
    stats["selected_count"] = len(top_n)
    filter_stats.append(stats)

    weight = 1.0 / len(top_n)
    for _, row in top_n.iterrows():
        portfolio_list.append({
            "MonthEnd": month_end,
            "MonthKey": month_key,
            "MonthKey_next": month_key_next,
            "Code": row["Code"],
            "Weight": weight,
            "risk_off": False
        })

df_portfolio = pd.DataFrame(portfolio_list)
df_filter_stats = pd.DataFrame(filter_stats)

total_months = len(df_filter_stats)
cash_months_cnt = df_portfolio[df_portfolio["Code"] == "CASH"]["MonthKey_next"].nunique()

print(f"ポートフォリオ形成完了: {len(df_portfolio):,} レコード")
print(f"リスクオフ月数: {risk_off_months}/{total_months} ({risk_off_months/total_months*100:.1f}%)")
print(f"翌月CASH月数: {cash_months_cnt}/{df_portfolio['MonthKey_next'].nunique()} ({cash_months_cnt/df_portfolio['MonthKey_next'].nunique()*100:.1f}%)")

print("\nフィルタ統計（全期間平均）:")
print(f"  初期: {df_filter_stats['initial_count'].mean():.0f}")
print(f"  株価後: {df_filter_stats['after_price'].mean():.0f}")
print(f"  財務後: {df_filter_stats['after_roe_bm'].mean():.0f}")
print(f"  売買後: {df_filter_stats['after_turnover'].mean():.0f}")
print(f"  流動性後: {df_filter_stats['after_liquidity'].mean():.0f}")
print(f"  最終選定: {df_filter_stats['selected_count'].mean():.0f}")


# ============================================================
# Step 3: 日次データ読み込み
# ============================================================

print("\n[3/8] 日次データ読み込み中（列名自動検出）...")

daily_files = sorted(MERGED_PARTS_DIR.glob("merged-part-*.parquet"))
if not daily_files:
    raise FileNotFoundError(f"日次データが見つかりません: {MERGED_PARTS_DIR}")

sample_df = pd.read_parquet(daily_files[0])
daily_stock_code_col = detect_stock_code_column(sample_df)
print(f"日次データの銘柄コード列: '{daily_stock_code_col}' → 'Code'")

daily_list = []
for f in daily_files:
    df_part = pd.read_parquet(f)
    df_part = normalize_columns(df_part, daily_stock_code_col)
    daily_list.append(df_part)

daily = pd.concat(daily_list, ignore_index=True)
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values(["Code", "Date"]).copy()

if "AdjustedClose" in daily.columns and "AdjustmentClose" not in daily.columns:
    daily = daily.rename(columns={"AdjustedClose": "AdjustmentClose"})

daily["ret_d"] = daily.groupby("Code")["AdjustmentClose"].pct_change()
daily["MonthKey"] = daily["Date"].dt.to_period("M").astype(str)

print(f"日次データ: {len(daily):,} 行, {daily['Code'].nunique():,} 銘柄")


# ============================================================
# Step 4: 日次ポートフォリオリターン計算（改善版 + v2 overlap）
# ============================================================

print("\n[4/8] 日次ポートフォリオリターン計算中（v2：月末決定→翌月初執行 + overlap見送り）...")

# CASH月マスタ（翌月にCASHを適用）
cash_tbl = df_portfolio[df_portfolio["Code"] == "CASH"][["MonthKey_next", "risk_off"]].copy()
cash_tbl = cash_tbl.rename(columns={"MonthKey_next": "MonthKey"})
cash_tbl["is_cash_month"] = True
cash_month_set = set(cash_tbl["MonthKey"].unique())

# 株式ポート（翌月に適用）
stock_portfolio = df_portfolio[df_portfolio["Code"] != "CASH"].copy()

# 月次ターゲット辞書（翌月）
monthly_target = {}
for mk, g in stock_portfolio.groupby("MonthKey_next"):
    monthly_target[mk] = dict(zip(g["Code"].astype(str), g["Weight"].astype(float)))

# 日次に月次ターゲットWeightを付与（株式ポートのみ）
daily_port = daily.merge(
    stock_portfolio[["MonthKey_next", "Code", "Weight"]],
    left_on=["MonthKey", "Code"],
    right_on=["MonthKey_next", "Code"],
    how="inner"
)
daily_port = daily_port.sort_values("Date").copy()

# 翌月第1営業日を求める
first_trade_day_of_month = daily.groupby("MonthKey")["Date"].min().to_dict()

def calc_turnover_and_new_hold(prev_w, target_w, deadband=WEIGHT_ADJUST_DEADBAND):
    prev = prev_w.copy()
    target = target_w.copy()
    all_codes = set(prev.keys()) | set(target.keys())
    after = {}

    for c in all_codes:
        w0 = prev.get(c, 0.0)
        w1 = target.get(c, 0.0)
        if (c in prev) and (c in target):
            if abs(w1 - w0) < deadband:
                after[c] = w0
            else:
                after[c] = w1
        else:
            after[c] = w1

    after = {c: w for c, w in after.items() if w > 0}
    s = sum(after.values())
    if s > 0:
        after = {c: w/s for c, w in after.items()}

    codes2 = set(prev.keys()) | set(after.keys())
    turnover = sum(abs(after.get(c, 0.0) - prev.get(c, 0.0)) for c in codes2)
    turnover_stocks = len(set(prev.keys()) ^ set(after.keys()))
    return turnover, turnover_stocks, after


global_wealth = INITIAL_CAPITAL
global_peak = INITIAL_CAPITAL
current_dd = 0.0

current_hold_weights = {}  # 実保有ウェイト（近似）
current_state = "CASH"     # "CASH" or "EQUITY"（CASH遷移の見送り禁止判定用）

daily_returns = []
leverage_events = []

rebalance_count = 0
skip_rebalance_count = 0
skip_rebalance_overlap_count = 0
skip_rebalance_turnover_count = 0

total_trading_cost = 0.0
total_interest_cost = 0.0  # 改善版は基本0

all_dates = sorted(daily["Date"].unique())

for date in all_dates:
    mk = pd.Timestamp(date).to_period("M").strftime("%Y-%m")

    # CASH判定（その月は現金か）
    is_cash_month = mk in cash_month_set
    is_risk_off = False
    if is_cash_month:
        tmp = cash_tbl[cash_tbl["MonthKey"] == mk]
        if len(tmp) > 0:
            is_risk_off = bool(tmp["risk_off"].iloc[0])
    is_cash = is_cash_month or is_risk_off

    # 当日がその月の第1営業日か（=執行日として扱う）
    is_first_trade_day = (mk in first_trade_day_of_month) and (pd.Timestamp(date) == pd.Timestamp(first_trade_day_of_month[mk]))

    trading_cost = 0.0
    interest_cost_pct = 0.0
    leverage_used = LEVERAGE_FIXED

    # ========== v2: リバランス日処理（CASH遷移は必ず実行） ==========
    if is_first_trade_day:
        target_w = monthly_target.get(mk, {})  # その月の株式ターゲット（なければ{}）

        # 状態遷移の定義
        target_state = "CASH" if is_cash else "EQUITY"
        is_cash_transition = (current_state != target_state)

        # overlap計算（株式月のときのみ意味がある）
        prev_codes = list(current_hold_weights.keys())
        new_codes = list(target_w.keys())
        ov = overlap_ratio(prev_codes, new_codes) if (current_state == "EQUITY" and target_state == "EQUITY") else np.nan

        # turnover計算（株式ターゲットがあるとき）
        turnover, turnover_stocks, new_hold = calc_turnover_and_new_hold(current_hold_weights, target_w)

        # ---- 見送り条件（C案：turnover条件 AND overlap条件。ただしCASH遷移は見送らない） ----
        do_skip = False
        skip_reason = ""

        if is_cash_transition:
            # CASH↔株式は必ず実行
            do_skip = False
        else:
            # 同じ状態（EQUITY→EQUITY または CASH→CASH）
            # CASH→CASH は実質何もしない
            if target_state == "CASH":
                do_skip = False
            else:
                # EQUITY→EQUITY の場合のみ、見送り判定
                turnover_skip = (turnover_stocks < REBALANCE_MIN_TURNOVER_STOCKS) and (turnover < REBALANCE_MIN_TURNOVER_PCT)
                overlap_skip = (USE_OVERLAP_SKIP and (ov >= OVERLAP_SKIP_TH))

                # C案：両方満たしたら見送り
                if turnover_skip and overlap_skip:
                    do_skip = True
                    skip_reason = f"SKIP_TURNOVER_AND_OVERLAP (ov={ov:.3f}, to={turnover:.3f}, nswap={turnover_stocks})"

        if do_skip:
            skip_rebalance_count += 1
            skip_rebalance_turnover_count += 1
            skip_rebalance_overlap_count += 1
            # 見送り：保有ウェイト維持、コスト0
            leverage_events.append({
                "Date": date,
                "Event": "RebalanceSkipped",
                "Reason": skip_reason,
                "MonthKey": mk,
                "PrevState": current_state,
                "TargetState": target_state,
                "Overlap": float(ov) if ov == ov else np.nan,
                "Turnover": float(turnover),
                "TurnoverStocks": int(turnover_stocks),
                "TradingCost": 0.0
            })
        else:
            # 実行
            # CASH月なら保有を空にする（株式を持たない）
            if target_state == "CASH":
                current_hold_weights = {}
                current_state = "CASH"
                rebalance_count += 1
                leverage_events.append({
                    "Date": date,
                    "Event": "RebalanceExecuted",
                    "Reason": "CASH_STATE",
                    "MonthKey": mk,
                    "PrevState": current_state,
                    "TargetState": target_state,
                    "Overlap": float(ov) if ov == ov else np.nan,
                    "Turnover": float(turnover),
                    "TurnoverStocks": int(turnover_stocks),
                    "TradingCost": 0.0
                })
            else:
                # 株式月：turnover比例でコスト
                rebalance_count += 1
                trading_cost = turnover * TOTAL_COST_PER_TRADE
                total_trading_cost += trading_cost
                current_hold_weights = new_hold
                current_state = "EQUITY"
                leverage_events.append({
                    "Date": date,
                    "Event": "RebalanceExecuted",
                    "Reason": "EQUITY_STATE",
                    "MonthKey": mk,
                    "PrevState": current_state,
                    "TargetState": target_state,
                    "Overlap": float(ov) if ov == ov else np.nan,
                    "Turnover": float(turnover),
                    "TurnoverStocks": int(turnover_stocks),
                    "TradingCost": float(trading_cost)
                })

    # ========== 日次リターン計算 ==========
    if is_cash:
        port_ret_gross = 0.0
        port_ret_net = 0.0
    else:
        df_day = daily_port[daily_port["Date"] == date].copy()
        if len(df_day) == 0:
            continue

        df_day["weighted_ret"] = df_day["ret_d"].fillna(0) * df_day["Weight"]
        port_ret_gross = df_day["weighted_ret"].sum() * leverage_used

        # v2：コストは「第1営業日」に計上（上でtrading_costが入っている）
        port_ret_net = port_ret_gross - trading_cost - interest_cost_pct

    global_wealth *= (1 + port_ret_net)
    if global_wealth > global_peak:
        global_peak = global_wealth

    current_dd = (global_wealth / global_peak) - 1

    daily_returns.append({
        "Date": date,
        "MonthKey": mk,
        "port_ret_gross": port_ret_gross,
        "port_ret_net": port_ret_net,
        "trading_cost": trading_cost,
        "interest_cost": interest_cost_pct,
        "leverage": leverage_used,
        "wealth": global_wealth,
        "peak": global_peak,
        "dd": current_dd,
        "is_cash": is_cash,
        "is_risk_off": is_risk_off,
        "is_cash_month": is_cash_month,
        "is_rebalance_day": bool(is_first_trade_day),
        "state": current_state
    })

df_daily_returns = pd.DataFrame(daily_returns)

print(f"\n日次ポートフォリオリターン: {len(df_daily_returns):,} 日")
print(f"リバランス実行回数: {rebalance_count} 回（翌月第1営業日ベース）")
print(f"リバランス見送り回数: {skip_rebalance_count} 回（turnover+overlapの両方満たす場合のみ）")
print(f"  - overlap起因見送り: {skip_rebalance_overlap_count} 回")
print(f"  - turnover起因見送り: {skip_rebalance_turnover_count} 回")
print(f"信用金利合計（期待0）: {total_interest_cost*100:.4f}%")
print(f"取引コスト合計（ターンオーバー比例）: {total_trading_cost*100:.2f}%")


# ============================================================
# Step 5: 日次MDD計算とサマリ
# ============================================================

print("\n[5/8] 日次MDD計算中...")

df_daily_returns["cum_ret"] = df_daily_returns["port_ret_net"].add(1).cumprod().sub(1)
df_daily_returns["cum_wealth"] = INITIAL_CAPITAL * (1 + df_daily_returns["cum_ret"])

total_days = len(df_daily_returns)
final_wealth = df_daily_returns["cum_wealth"].iloc[-1]
cum_return = (final_wealth / INITIAL_CAPITAL) - 1

first_date = df_daily_returns["Date"].min()
last_date = df_daily_returns["Date"].max()
years = (last_date - first_date).days / 365.25 if last_date > first_date else 1.0

ann_return = (1 + cum_return) ** (1 / years) - 1
ann_vol = df_daily_returns["port_ret_net"].std() * np.sqrt(252)
sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0

daily_mdd = df_daily_returns["dd"].min()
worst_dd_date = df_daily_returns.loc[df_daily_returns["dd"].idxmin(), "Date"]

cash_days = int(df_daily_returns["is_cash"].sum())
cash_pct = cash_days / total_days * 100
avg_leverage = df_daily_returns["leverage"].mean()

total_gross_return = df_daily_returns["port_ret_gross"].add(1).cumprod().iloc[-1] - 1
total_net_return = cum_return
gross_ann = (1 + total_gross_return) ** (1 / years) - 1
net_ann = (1 + total_net_return) ** (1 / years) - 1
annual_cost_drag = gross_ann - net_ann

print("\n" + "=" * 80)
print("最終結果サマリー（月次改善版 v2）")
print("=" * 80)
print(f"期間: {first_date.date()} 〜 {last_date.date()}")
print(f"総日数: {total_days}日 ({years:.2f}年)")
print(f"初期資金: ¥{INITIAL_CAPITAL:,}")
print(f"最終資産: ¥{final_wealth:,.0f}")
print(f"累積リターン: {cum_return*100:.2f}%")
print(f"年率リターン: {ann_return*100:.2f}%")
print(f"年率ボラティリティ: {ann_vol*100:.2f}%")
print(f"シャープレシオ: {sharpe:.4f}")
print(f"日次最大DD: {daily_mdd*100:.2f}% (日付: {worst_dd_date.date()})")
print(f"平均CASH比率: {cash_pct:.2f}%")
print(f"平均レバレッジ: {avg_leverage:.2f}x（改善版は原則1.0）")

print("\n[v2] 取引コストの影響:")
print(f"  グロス累積リターン: {total_gross_return*100:.2f}%")
print(f"  ネット累積リターン: {total_net_return*100:.2f}%")
print(f"  年率コストドラッグ（年率差分）: {annual_cost_drag*100:.2f}%/年")
print(f"  取引コスト累積（ターンオーバー比例、足し上げ）: {total_trading_cost*100:.2f}%")
print("=" * 80)


# ============================================================
# Step 6: ファイル保存（v2）
# ============================================================

print("\n[6/8] ファイル保存中...")

out_file1 = ANALYSIS_DIR / "daily_portfolio_returns_monthly_improved_v2.parquet"
df_daily_returns.to_parquet(out_file1, index=False)
print(f"  {out_file1}")

df_leverage_events = pd.DataFrame(leverage_events) if leverage_events else pd.DataFrame(columns=[
    "Date","Event","Reason","MonthKey","PrevState","TargetState","Overlap","Turnover","TurnoverStocks","TradingCost"
])
out_file2 = ANALYSIS_DIR / "leverage_events_monthly_improved_v2.csv"
df_leverage_events.to_csv(out_file2, index=False, encoding="utf-8-sig")
print(f"  {out_file2}")

summary_data = {
    "period_start": [first_date],
    "period_end": [last_date],
    "total_days": [total_days],
    "initial_capital": [INITIAL_CAPITAL],
    "final_wealth": [final_wealth],
    "cumulative_return": [cum_return],
    "annual_return": [ann_return],
    "annual_volatility": [ann_vol],
    "sharpe_ratio": [sharpe],
    "daily_max_dd": [daily_mdd],
    "worst_dd_date": [worst_dd_date],
    "cash_days": [cash_days],
    "cash_pct": [cash_pct],
    "avg_leverage": [avg_leverage],
    "gross_return": [total_gross_return],
    "net_return": [total_net_return],
    "annual_cost_drag": [annual_cost_drag],
    "rebalance_count": [rebalance_count],
    "skip_rebalance_count": [skip_rebalance_count],
    "skip_overlap_and_turnover_count": [skip_rebalance_count],
    "total_trading_cost_sum": [total_trading_cost],
    "total_interest_cost_sum": [total_interest_cost],
}
df_summary = pd.DataFrame(summary_data)
out_file3 = ANALYSIS_DIR / "daily_mdd_summary_monthly_improved_v2.csv"
df_summary.to_csv(out_file3, index=False, encoding="utf-8-sig")
print(f"  {out_file3}")

out_file4 = ANALYSIS_DIR / "filter_statistics_monthly_improved_v2.csv"
df_filter_stats.to_csv(out_file4, index=False, encoding="utf-8-sig")
print(f"  {out_file4}")


# ============================================================
# Step 7: 比較表（今回v2のみ）
# ============================================================

print("\n[7/8] 戦略比較表作成中...")

comparison_data = {
    "指標": [
        "年率リターン",
        "年率ボラティリティ",
        "シャープレシオ",
        "最大DD",
        "累積リターン",
        "CASH比率",
        "平均レバレッジ",
        "年率コストドラッグ（年率差分）",
        "取引コスト累積（足し上げ）",
        "リバランス実行回数",
        "リバランス見送り回数（turnover&overlap両方）"
    ],
    "月次改善版 v2": [
        f"{ann_return*100:.2f}%",
        f"{ann_vol*100:.2f}%",
        f"{sharpe:.4f}",
        f"{daily_mdd*100:.2f}%",
        f"{cum_return*100:.2f}%",
        f"{cash_pct:.2f}%",
        f"{avg_leverage:.2f}x",
        f"{annual_cost_drag*100:.2f}%/年",
        f"{total_trading_cost*100:.2f}%",
        f"{rebalance_count}",
        f"{skip_rebalance_count}"
    ]
}
df_comparison = pd.DataFrame(comparison_data)

print("\n" + "=" * 80)
print("戦略比較表（今回：月次改善版 v2）")
print("=" * 80)
print(df_comparison.to_string(index=False))
print("=" * 80)


# ============================================================
# Step 8: 完了メッセージ
# ============================================================

print("\n" + "=" * 80)
print("[8/8] 月次戦略 改善版 v2 バックテスト完了 ✅")
print("=" * 80)

print("\n次のチェック（必須）:")
print("1) risk_off月数・翌月CASH月数が旧コード（約26%）と同程度か")
print("2) フィルタ後の selected_count が平均で TOP_N=20 に近づいたか")
print("3) 取引コスト累積が旧改善版より下がったか（overlap見送りが効く）")
print("4) MaxDD が -25%近辺を維持できるか")
print("5) Sharpe が 1.2〜1.4を維持/改善できるか")


月次戦略 改善版バックテスト v2（CASH完全対応 + 列名自動検出）
前提：月末引けで決定 → 翌月初寄りで執行（コストは翌月第1営業日に計上、価格は日足近似）
初期資金: ¥10,000,000
TOP_N: 20
株価フィルタ: ¥800以上
売買代金上位: 70%
最低日次売買代金: ¥5,000,000
取引コスト（片道）: 0.40%（ターンオーバー比例）
レバレッジ: 1.00x（改善版：信用取引なし）
部分リバランス deadband: 5%
--------------------------------------------------------------------------------
[v2] LIQUIDITY_BUFFER_FACTOR: 0.7
[v2] SCORE_EWMA_ALPHA: 0.5
[v2] OVERLAP_SKIP_TH: 0.8 USE_OVERLAP_SKIP: True

[1/8] 月次スナップショット読み込み中（列名自動検出）...
検出列（先頭10列）: ['MonthEnd', 'Code', 'AdjustedClose', 'Vo', 'MarketCap', 'BM_Ratio', 'ROE', 'INV_Growth', 'Month']
銘柄コード列: 'Code' → 'Code'
列名正規化完了
期間: 2016-01-31 〜 2026-01-31
データ: 502,101 行, 5,303 銘柄

[v2] スコアEWMA（2ヶ月相当）を計算中...
  composite_score_ewma_all を追加しました

[2/8] 月次ポートフォリオ形成中（翌月ターゲット決定、CASH対応）...
Vol3閾値（90%ile）: 0.064522
ポートフォリオ形成完了: 1,812 レコード
リスクオフ月数: 32/121 (26.4%)
翌月CASH月数: 32/121 (26.4%)

フィルタ統計（全期間平均）:
  初期: 4150
  株価後: 2055
  財務後: 1645
  売買後: 1151
  流動性後: 1151
  最終選定: 15

[3/8] 日次データ読み込み中（列名自動検出）...
日次データの銘柄コード列: 'Code' → 'Code

In [7]:
"""
月次戦略 改善版（v2.1）：リアリスティック・バックテスト
（CASH完全対応 + 列名自動検出 + 月末引けで決定→翌月初寄りで執行）
※日次データがAdjustmentCloseのみの場合、寄り執行は厳密再現不可 → 「翌月第1営業日にコスト計上」で執行日整合（価格は日足近似）

v2（改善第2弾）
(1) 流動性制約を 0.8 → 0.7 に緩和（LIQUIDITY_BUFFER_FACTOR）
(2) スコアを2ヶ月EWMAで平滑化（alpha=0.50）
(3) overlapでリバランス見送り（閾値0.80）
    ただし CASH↔株式の遷移は必ず実行（見送り禁止）
    既存のturnover見送りと併用

v2.1（今回の変更：あなたの指示「1」）
- 見送り条件を「turnover_skip AND overlap_skip」→「turnover_skip OR overlap_skip」に変更
  → overlap見送りが0回だった問題を解消し、コスト削減を確実に発火させる

出力（v2.1）
- daily_portfolio_returns_monthly_improved_v2.parquet
- leverage_events_monthly_improved_v2.csv
- daily_mdd_summary_monthly_improved_v2.csv
- filter_statistics_monthly_improved_v2.csv
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 基本設定
# ============================================================

OUTPUT_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks")
SNAP_PATH = OUTPUT_DIR / "factors" / "month_end_snapshot.parquet"
MERGED_PARTS_DIR = OUTPUT_DIR / "merged_parts"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_daily"
ANALYSIS_DIR.mkdir(exist_ok=True)


# ============================================================
# パラメータ（改善版 + v2）
# ============================================================

### 基本
INITIAL_CAPITAL = 10_000_000
TOP_N = 20
VO_TOP_PCT = 0.70
DEFAULT_TRADING_UNIT = 100

### 株価・財務フィルタ
MIN_PRICE = 800
MAX_PRICE = None
ROE_MIN = -0.5
ROE_MAX = 1.0
BM_RATIO_MIN = 0.1
BM_RATIO_MAX = 10.0

### リスクオフ判定（市場状態）※旧コード踏襲
OFF_TH = -0.115
ON_TH = -0.085
TREND3_TH = -0.02
VOL3_Q = 0.90

### 取引コスト（片道）
COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.003
TOTAL_COST_PER_TRADE = COMMISSION_RATE + SLIPPAGE_RATE  # 片道0.4%

### 流動性制約
MAX_VOLUME_PARTICIPATION = 0.10
MIN_DAILY_TURNOVER = 5_000_000

### 価格ギャップ対策（執行可能性の安全側）
PRICE_GAP_BUFFER = 0.95  # 5%安全側
ENTRY_SLIPPAGE = 0.01

### 改善版：レバレッジ固定（信用金利ゼロ）
LEVERAGE_FIXED = 1.0
MARGIN_INTEREST_RATE = 0.028  # 参考（改善版では使わない）

### 改善版：部分リバランス（デッドバンド）と条件付きリバランス（旧コード踏襲）
WEIGHT_ADJUST_DEADBAND = 0.05
REBALANCE_MIN_TURNOVER_STOCKS = 5
REBALANCE_MIN_TURNOVER_PCT = 0.10

# -------------------------
# v2（改善第2弾）追加パラメータ
# -------------------------
LIQUIDITY_BUFFER_FACTOR = 0.70     # (1) 流動性 0.8 -> 0.7
SCORE_EWMA_ALPHA = 0.50            # (2) スコアEWMA alpha
OVERLAP_SKIP_TH = 0.80             # (3) overlap見送り閾値
USE_OVERLAP_SKIP = True            # overlap見送り有効化

# v2.1：見送り条件を OR にする
SKIP_MODE = "OR"  # "OR" or "AND"（検証用に残す）

print("=" * 80)
print("月次戦略 改善版バックテスト v2.1（CASH完全対応 + 列名自動検出）")
print("前提：月末引けで決定 → 翌月初寄りで執行（コストは翌月第1営業日に計上、価格は日足近似）")
print("=" * 80)
print(f"初期資金: ¥{INITIAL_CAPITAL:,}")
print(f"TOP_N: {TOP_N}")
print(f"株価フィルタ: ¥{MIN_PRICE}以上")
print(f"売買代金上位: {VO_TOP_PCT*100:.0f}%")
print(f"最低日次売買代金: ¥{MIN_DAILY_TURNOVER:,}")
print(f"取引コスト（片道）: {TOTAL_COST_PER_TRADE*100:.2f}%（ターンオーバー比例）")
print(f"レバレッジ: {LEVERAGE_FIXED:.2f}x（改善版：信用取引なし）")
print(f"部分リバランス deadband: {WEIGHT_ADJUST_DEADBAND*100:.0f}%")
print("-" * 80)
print("[v2] LIQUIDITY_BUFFER_FACTOR:", LIQUIDITY_BUFFER_FACTOR)
print("[v2] SCORE_EWMA_ALPHA:", SCORE_EWMA_ALPHA)
print("[v2] OVERLAP_SKIP_TH:", OVERLAP_SKIP_TH, "USE_OVERLAP_SKIP:", USE_OVERLAP_SKIP)
print("[v2.1] SKIP_MODE:", SKIP_MODE, "(turnover_skip OR overlap_skip)")
print("=" * 80)


# ============================================================
# 列名自動検出・正規化
# ============================================================

def detect_stock_code_column(df):
    candidates = ["Code", "StockCode", "Symbol", "Ticker", "stock_code", "code"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"銘柄コード列が見つかりません。利用可能な列: {list(df.columns)}")

def normalize_columns(df, stock_code_col=None):
    rename_map = {}
    if stock_code_col and stock_code_col != "Code":
        rename_map[stock_code_col] = "Code"
    if "AdjustedClose" in df.columns and "AdjustmentClose" not in df.columns:
        rename_map["AdjustedClose"] = "AdjustmentClose"
    if "Volume" in df.columns and "Vo" not in df.columns:
        rename_map["Volume"] = "Vo"
    return df.rename(columns=rename_map)


# ============================================================
# v2: EWMA平滑化ユーティリティ
# ============================================================

def add_ewma_score_to_snapshot(snap: pd.DataFrame, alpha: float) -> pd.DataFrame:
    snap = snap.sort_values(["Code", "MonthEnd"]).copy()

    def _calc_month_z_and_score(g):
        g = g.copy()
        for col in ["BM_Ratio", "ROE", "INV_Growth"]:
            mean_val = g[col].mean()
            std_val = g[col].std()
            g[f"{col}_z_all"] = (g[col] - mean_val) / std_val if (std_val is not None and std_val > 0) else 0.0
        g["composite_score_raw_all"] = (
            g["BM_Ratio_z_all"] +
            g["ROE_z_all"] -
            g["INV_Growth_z_all"].fillna(0)
        )
        return g

    snap = snap.groupby("MonthEnd", group_keys=False).apply(_calc_month_z_and_score)

    snap["composite_score_ewma_all"] = (
        snap.groupby("Code")["composite_score_raw_all"]
            .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
    )

    return snap


def overlap_ratio(prev_codes, new_codes) -> float:
    prev_set = set(prev_codes)
    new_set = set(new_codes)
    if len(prev_set) == 0:
        return 0.0
    return len(prev_set & new_set) / max(len(prev_set), 1)


# ============================================================
# Step 1: 月次スナップショット読み込み
# ============================================================

print("\n[1/8] 月次スナップショット読み込み中（列名自動検出）...")
snap = pd.read_parquet(SNAP_PATH)
snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])

print(f"検出列（先頭10列）: {list(snap.columns[:10])}")

stock_code_col = detect_stock_code_column(snap)
print(f"銘柄コード列: '{stock_code_col}' → 'Code'")
snap = normalize_columns(snap, stock_code_col)

required_cols = ["MonthEnd", "Code", "AdjustmentClose", "Vo", "MarketCap", "BM_Ratio", "ROE", "INV_Growth"]
missing_cols = [c for c in required_cols if c not in snap.columns]
if missing_cols:
    print(f"不足列: {missing_cols}")
    if "AdjustmentClose" not in snap.columns:
        if "Close" in snap.columns:
            snap["AdjustmentClose"] = snap["Close"]
            print("  代替: Close → AdjustmentClose")
        elif "AdjustedClose" in snap.columns:
            snap["AdjustmentClose"] = snap["AdjustedClose"]
            print("  代替: AdjustedClose → AdjustmentClose")
    if "Vo" not in snap.columns and "Volume" in snap.columns:
        snap["Vo"] = snap["Volume"]
        print("  代替: Volume → Vo")
    missing_cols = [c for c in required_cols if c not in snap.columns]
    if missing_cols:
        raise ValueError(f"必須列が不足しており代替もできません: {missing_cols}")

snap = snap.sort_values(["Code", "MonthEnd"]).copy()
snap["MonthKey"] = snap["MonthEnd"].dt.to_period("M").astype(str)

print("列名正規化完了")
print(f"期間: {snap['MonthEnd'].min().date()} 〜 {snap['MonthEnd'].max().date()}")
print(f"データ: {len(snap):,} 行, {snap['Code'].nunique():,} 銘柄")


# ============================================================
# v2: スコアEWMAをスナップショットに付与
# ============================================================
print("\n[v2] スコアEWMA（2ヶ月相当）を計算中...")
snap = add_ewma_score_to_snapshot(snap, alpha=SCORE_EWMA_ALPHA)
print("  composite_score_ewma_all を追加しました")


# ============================================================
# Step 2: 月次ポートフォリオ形成（翌月のターゲットを決める）
# ============================================================

print("\n[2/8] 月次ポートフォリオ形成中（翌月ターゲット決定、CASH対応）...")

snap["ret_m_fwd"] = snap.groupby("Code")["AdjustmentClose"].pct_change().shift(-1)
snap["TurnoverValue"] = snap["Vo"] * snap["AdjustmentClose"]

monthly = snap.groupby("MonthEnd").apply(
    lambda g: pd.Series({
        "mkt_vw": np.average(g["ret_m_fwd"].fillna(0), weights=g["MarketCap"].fillna(1)),
        "count": len(g)
    })
).reset_index()

monthly["cum_ret"] = (1 + monthly["mkt_vw"]).cumprod()
monthly["peak"] = monthly["cum_ret"].cummax()
monthly["mkt_dd"] = (monthly["cum_ret"] / monthly["peak"]) - 1
monthly["trend3"] = monthly["mkt_vw"].rolling(3, min_periods=1).mean()
monthly["vol3"] = monthly["mkt_vw"].rolling(3, min_periods=2).std()

vol3_th = monthly["vol3"].quantile(VOL3_Q)
print(f"Vol3閾値（{VOL3_Q*100:.0f}%ile）: {vol3_th:.6f}")

snap = snap.merge(monthly[["MonthEnd", "mkt_dd", "trend3", "vol3"]], on="MonthEnd", how="left")
snap["risk_off"] = (
    (snap["mkt_dd"] <= OFF_TH) |
    (snap["trend3"] <= TREND3_TH) |
    (snap["vol3"] >= vol3_th)
)

portfolio_list = []
filter_stats = []
risk_off_months = 0

for month_end, df_month in snap.groupby("MonthEnd"):
    month_key = df_month["MonthKey"].iloc[0]
    risk_off = bool(df_month["risk_off"].iloc[0])

    stats = {
        "MonthEnd": month_end,
        "MonthKey": month_key,
        "initial_count": len(df_month),
        "risk_off": risk_off
    }

    month_key_next = (pd.Timestamp(month_end) + pd.DateOffset(months=1)).to_period("M").strftime("%Y-%m")

    if risk_off:
        risk_off_months += 1
        portfolio_list.append({
            "MonthEnd": month_end,
            "MonthKey": month_key,
            "MonthKey_next": month_key_next,
            "Code": "CASH",
            "Weight": 1.0,
            "risk_off": True
        })
        stats.update({
            "after_price": 0,
            "after_roe_bm": 0,
            "after_turnover": 0,
            "after_liquidity": 0,
            "final_count": 0,
            "selected_count": 0
        })
        filter_stats.append(stats)
        continue

    df_valid = df_month[df_month["AdjustmentClose"] >= MIN_PRICE].copy()
    if MAX_PRICE:
        df_valid = df_valid[df_valid["AdjustmentClose"] <= MAX_PRICE]
    stats["after_price"] = len(df_valid)

    df_valid = df_valid[
        (df_valid["BM_Ratio"].notna()) &
        (df_valid["ROE"].notna()) &
        (df_valid["ROE"] >= ROE_MIN) &
        (df_valid["ROE"] <= ROE_MAX) &
        (df_valid["BM_Ratio"] >= BM_RATIO_MIN) &
        (df_valid["BM_Ratio"] <= BM_RATIO_MAX)
    ]
    stats["after_roe_bm"] = len(df_valid)

    if "TurnoverValue" in df_valid.columns and len(df_valid) > 0:
        turnover_th = df_valid["TurnoverValue"].quantile(1 - VO_TOP_PCT)
        df_valid = df_valid[df_valid["TurnoverValue"] >= turnover_th]
    stats["after_turnover"] = len(df_valid)

    df_valid = df_valid[df_valid["TurnoverValue"] >= MIN_DAILY_TURNOVER]

    df_valid["MaxBuyableAmount"] = df_valid["Vo"] * MAX_VOLUME_PARTICIPATION * df_valid["AdjustmentClose"]

    # (1) v2: 流動性制約 0.8 -> 0.7
    target_capital = INITIAL_CAPITAL * LEVERAGE_FIXED * PRICE_GAP_BUFFER
    capital_per_stock = target_capital / TOP_N
    df_valid = df_valid[df_valid["MaxBuyableAmount"] >= capital_per_stock * LIQUIDITY_BUFFER_FACTOR]
    stats["after_liquidity"] = len(df_valid)

    if len(df_valid) < TOP_N:
        portfolio_list.append({
            "MonthEnd": month_end,
            "MonthKey": month_key,
            "MonthKey_next": month_key_next,
            "Code": "CASH",
            "Weight": 1.0,
            "risk_off": False
        })
        stats.update({"final_count": len(df_valid), "selected_count": 0})
        filter_stats.append(stats)
        continue

    # (2) v2: スコアは EWMA を使用
    score_col = "composite_score_ewma_all"
    top_n = df_valid.nlargest(TOP_N, score_col)

    stats["final_count"] = len(df_valid)
    stats["selected_count"] = len(top_n)
    filter_stats.append(stats)

    weight = 1.0 / len(top_n)
    for _, row in top_n.iterrows():
        portfolio_list.append({
            "MonthEnd": month_end,
            "MonthKey": month_key,
            "MonthKey_next": month_key_next,
            "Code": row["Code"],
            "Weight": weight,
            "risk_off": False
        })

df_portfolio = pd.DataFrame(portfolio_list)
df_filter_stats = pd.DataFrame(filter_stats)

total_months = len(df_filter_stats)
cash_months_cnt = df_portfolio[df_portfolio["Code"] == "CASH"]["MonthKey_next"].nunique()

print(f"ポートフォリオ形成完了: {len(df_portfolio):,} レコード")
print(f"リスクオフ月数: {risk_off_months}/{total_months} ({risk_off_months/total_months*100:.1f}%)")
print(f"翌月CASH月数: {cash_months_cnt}/{df_portfolio['MonthKey_next'].nunique()} ({cash_months_cnt/df_portfolio['MonthKey_next'].nunique()*100:.1f}%)")

print("\nフィルタ統計（全期間平均）:")
print(f"  初期: {df_filter_stats['initial_count'].mean():.0f}")
print(f"  株価後: {df_filter_stats['after_price'].mean():.0f}")
print(f"  財務後: {df_filter_stats['after_roe_bm'].mean():.0f}")
print(f"  売買後: {df_filter_stats['after_turnover'].mean():.0f}")
print(f"  流動性後: {df_filter_stats['after_liquidity'].mean():.0f}")
print(f"  最終選定: {df_filter_stats['selected_count'].mean():.0f}")


# ============================================================
# Step 3: 日次データ読み込み
# ============================================================

print("\n[3/8] 日次データ読み込み中（列名自動検出）...")

daily_files = sorted(MERGED_PARTS_DIR.glob("merged-part-*.parquet"))
if not daily_files:
    raise FileNotFoundError(f"日次データが見つかりません: {MERGED_PARTS_DIR}")

sample_df = pd.read_parquet(daily_files[0])
daily_stock_code_col = detect_stock_code_column(sample_df)
print(f"日次データの銘柄コード列: '{daily_stock_code_col}' → 'Code'")

daily_list = []
for f in daily_files:
    df_part = pd.read_parquet(f)
    df_part = normalize_columns(df_part, daily_stock_code_col)
    daily_list.append(df_part)

daily = pd.concat(daily_list, ignore_index=True)
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values(["Code", "Date"]).copy()

if "AdjustedClose" in daily.columns and "AdjustmentClose" not in daily.columns:
    daily = daily.rename(columns={"AdjustedClose": "AdjustmentClose"})

daily["ret_d"] = daily.groupby("Code")["AdjustmentClose"].pct_change()
daily["MonthKey"] = daily["Date"].dt.to_period("M").astype(str)

print(f"日次データ: {len(daily):,} 行, {daily['Code'].nunique():,} 銘柄")


# ============================================================
# Step 4: 日次ポートフォリオリターン計算（v2.1：OR見送り）
# ============================================================

print("\n[4/8] 日次ポートフォリオリターン計算中（v2.1：月末決定→翌月初執行 + overlap見送りOR）...")

cash_tbl = df_portfolio[df_portfolio["Code"] == "CASH"][["MonthKey_next", "risk_off"]].copy()
cash_tbl = cash_tbl.rename(columns={"MonthKey_next": "MonthKey"})
cash_tbl["is_cash_month"] = True
cash_month_set = set(cash_tbl["MonthKey"].unique())

stock_portfolio = df_portfolio[df_portfolio["Code"] != "CASH"].copy()

monthly_target = {}
for mk, g in stock_portfolio.groupby("MonthKey_next"):
    monthly_target[mk] = dict(zip(g["Code"].astype(str), g["Weight"].astype(float)))

daily_port = daily.merge(
    stock_portfolio[["MonthKey_next", "Code", "Weight"]],
    left_on=["MonthKey", "Code"],
    right_on=["MonthKey_next", "Code"],
    how="inner"
).sort_values("Date").copy()

first_trade_day_of_month = daily.groupby("MonthKey")["Date"].min().to_dict()

def calc_turnover_and_new_hold(prev_w, target_w, deadband=WEIGHT_ADJUST_DEADBAND):
    prev = prev_w.copy()
    target = target_w.copy()
    all_codes = set(prev.keys()) | set(target.keys())
    after = {}

    for c in all_codes:
        w0 = prev.get(c, 0.0)
        w1 = target.get(c, 0.0)
        if (c in prev) and (c in target):
            if abs(w1 - w0) < deadband:
                after[c] = w0
            else:
                after[c] = w1
        else:
            after[c] = w1

    after = {c: w for c, w in after.items() if w > 0}
    s = sum(after.values())
    if s > 0:
        after = {c: w/s for c, w in after.items()}

    codes2 = set(prev.keys()) | set(after.keys())
    turnover = sum(abs(after.get(c, 0.0) - prev.get(c, 0.0)) for c in codes2)
    turnover_stocks = len(set(prev.keys()) ^ set(after.keys()))
    return turnover, turnover_stocks, after


global_wealth = INITIAL_CAPITAL
global_peak = INITIAL_CAPITAL

current_hold_weights = {}
current_state = "CASH"  # "CASH" or "EQUITY"

daily_returns = []
leverage_events = []

rebalance_count = 0
skip_rebalance_count = 0
skip_by_overlap = 0
skip_by_turnover = 0

total_trading_cost = 0.0
total_interest_cost = 0.0

all_dates = sorted(daily["Date"].unique())

for date in all_dates:
    mk = pd.Timestamp(date).to_period("M").strftime("%Y-%m")

    is_cash_month = mk in cash_month_set
    is_risk_off = False
    if is_cash_month:
        tmp = cash_tbl[cash_tbl["MonthKey"] == mk]
        if len(tmp) > 0:
            is_risk_off = bool(tmp["risk_off"].iloc[0])
    is_cash = is_cash_month or is_risk_off

    is_first_trade_day = (mk in first_trade_day_of_month) and (pd.Timestamp(date) == pd.Timestamp(first_trade_day_of_month[mk]))

    trading_cost = 0.0
    interest_cost_pct = 0.0
    leverage_used = LEVERAGE_FIXED

    # ---------- リバランス（執行日） ----------
    if is_first_trade_day:
        target_w = monthly_target.get(mk, {})

        target_state = "CASH" if is_cash else "EQUITY"
        is_cash_transition = (current_state != target_state)

        prev_codes = list(current_hold_weights.keys())
        new_codes = list(target_w.keys())
        ov = overlap_ratio(prev_codes, new_codes) if (current_state == "EQUITY" and target_state == "EQUITY") else np.nan

        turnover, turnover_stocks, new_hold = calc_turnover_and_new_hold(current_hold_weights, target_w)

        do_skip = False
        reasons = []

        if is_cash_transition:
            do_skip = False  # CASH遷移は必ず実行
        else:
            if target_state == "CASH":
                do_skip = False
            else:
                # (turnover) 既存見送り
                turnover_skip = (turnover_stocks < REBALANCE_MIN_TURNOVER_STOCKS) and (turnover < REBALANCE_MIN_TURNOVER_PCT)
                # (overlap) v2見送り
                overlap_skip = (USE_OVERLAP_SKIP and (ov >= OVERLAP_SKIP_TH))

                # v2.1：OR化（どちらか満たせば見送り）
                if turnover_skip or overlap_skip:
                    do_skip = True
                    if turnover_skip:
                        reasons.append(f"TURNOVER (to={turnover:.3f}, nswap={turnover_stocks})")
                    if overlap_skip:
                        reasons.append(f"OVERLAP (ov={ov:.3f})")

        if do_skip:
            skip_rebalance_count += 1
            if ("TURNOVER" in " ".join(reasons)):
                skip_by_turnover += 1
            if ("OVERLAP" in " ".join(reasons)):
                skip_by_overlap += 1

            leverage_events.append({
                "Date": date,
                "Event": "RebalanceSkipped",
                "Reason": " | ".join(reasons),
                "MonthKey": mk,
                "PrevState": current_state,
                "TargetState": target_state,
                "Overlap": float(ov) if ov == ov else np.nan,
                "Turnover": float(turnover),
                "TurnoverStocks": int(turnover_stocks),
                "TradingCost": 0.0
            })
        else:
            # 実行
            if target_state == "CASH":
                current_hold_weights = {}
                current_state = "CASH"
                rebalance_count += 1
                leverage_events.append({
                    "Date": date,
                    "Event": "RebalanceExecuted",
                    "Reason": "CASH_STATE",
                    "MonthKey": mk,
                    "PrevState": current_state,
                    "TargetState": target_state,
                    "Overlap": float(ov) if ov == ov else np.nan,
                    "Turnover": float(turnover),
                    "TurnoverStocks": int(turnover_stocks),
                    "TradingCost": 0.0
                })
            else:
                rebalance_count += 1
                trading_cost = turnover * TOTAL_COST_PER_TRADE
                total_trading_cost += trading_cost
                current_hold_weights = new_hold
                current_state = "EQUITY"
                leverage_events.append({
                    "Date": date,
                    "Event": "RebalanceExecuted",
                    "Reason": "EQUITY_STATE",
                    "MonthKey": mk,
                    "PrevState": current_state,
                    "TargetState": target_state,
                    "Overlap": float(ov) if ov == ov else np.nan,
                    "Turnover": float(turnover),
                    "TurnoverStocks": int(turnover_stocks),
                    "TradingCost": float(trading_cost)
                })

    # ---------- 日次リターン ----------
    if is_cash:
        port_ret_gross = 0.0
        port_ret_net = 0.0
    else:
        df_day = daily_port[daily_port["Date"] == date].copy()
        if len(df_day) == 0:
            continue
        df_day["weighted_ret"] = df_day["ret_d"].fillna(0) * df_day["Weight"]
        port_ret_gross = df_day["weighted_ret"].sum() * leverage_used
        port_ret_net = port_ret_gross - trading_cost - interest_cost_pct

    global_wealth *= (1 + port_ret_net)
    if global_wealth > global_peak:
        global_peak = global_wealth
    dd = (global_wealth / global_peak) - 1

    daily_returns.append({
        "Date": date,
        "MonthKey": mk,
        "port_ret_gross": port_ret_gross,
        "port_ret_net": port_ret_net,
        "trading_cost": trading_cost,
        "interest_cost": interest_cost_pct,
        "leverage": leverage_used,
        "wealth": global_wealth,
        "peak": global_peak,
        "dd": dd,
        "is_cash": is_cash,
        "is_risk_off": is_risk_off,
        "is_cash_month": is_cash_month,
        "is_rebalance_day": bool(is_first_trade_day),
        "state": current_state
    })

df_daily_returns = pd.DataFrame(daily_returns)

print(f"\n日次ポートフォリオリターン: {len(df_daily_returns):,} 日")
print(f"リバランス実行回数: {rebalance_count} 回（翌月第1営業日ベース）")
print(f"リバランス見送り回数: {skip_rebalance_count} 回（OR：turnover または overlap）")
print(f"  - overlapが理由に含まれた回数: {skip_by_overlap} 回")
print(f"  - turnoverが理由に含まれた回数: {skip_by_turnover} 回")
print(f"信用金利合計（期待0）: {total_interest_cost*100:.4f}%")
print(f"取引コスト合計（ターンオーバー比例）: {total_trading_cost*100:.2f}%")


# ============================================================
# Step 5: 日次MDD計算とサマリ
# ============================================================

print("\n[5/8] 日次MDD計算中...")

df_daily_returns["cum_ret"] = df_daily_returns["port_ret_net"].add(1).cumprod().sub(1)
df_daily_returns["cum_wealth"] = INITIAL_CAPITAL * (1 + df_daily_returns["cum_ret"])

total_days = len(df_daily_returns)
final_wealth = df_daily_returns["cum_wealth"].iloc[-1]
cum_return = (final_wealth / INITIAL_CAPITAL) - 1

first_date = df_daily_returns["Date"].min()
last_date = df_daily_returns["Date"].max()
years = (last_date - first_date).days / 365.25 if last_date > first_date else 1.0

ann_return = (1 + cum_return) ** (1 / years) - 1
ann_vol = df_daily_returns["port_ret_net"].std() * np.sqrt(252)
sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0

daily_mdd = df_daily_returns["dd"].min()
worst_dd_date = df_daily_returns.loc[df_daily_returns["dd"].idxmin(), "Date"]

cash_days = int(df_daily_returns["is_cash"].sum())
cash_pct = cash_days / total_days * 100
avg_leverage = df_daily_returns["leverage"].mean()

total_gross_return = df_daily_returns["port_ret_gross"].add(1).cumprod().iloc[-1] - 1
total_net_return = cum_return
gross_ann = (1 + total_gross_return) ** (1 / years) - 1
net_ann = (1 + total_net_return) ** (1 / years) - 1
annual_cost_drag = gross_ann - net_ann

print("\n" + "=" * 80)
print("最終結果サマリー（月次改善版 v2.1）")
print("=" * 80)
print(f"期間: {first_date.date()} 〜 {last_date.date()}")
print(f"総日数: {total_days}日 ({years:.2f}年)")
print(f"初期資金: ¥{INITIAL_CAPITAL:,}")
print(f"最終資産: ¥{final_wealth:,.0f}")
print(f"累積リターン: {cum_return*100:.2f}%")
print(f"年率リターン: {ann_return*100:.2f}%")
print(f"年率ボラティリティ: {ann_vol*100:.2f}%")
print(f"シャープレシオ: {sharpe:.4f}")
print(f"日次最大DD: {daily_mdd*100:.2f}% (日付: {worst_dd_date.date()})")
print(f"平均CASH比率: {cash_pct:.2f}%")
print(f"平均レバレッジ: {avg_leverage:.2f}x")

print("\n[v2.1] 取引コストの影響:")
print(f"  グロス累積リターン: {total_gross_return*100:.2f}%")
print(f"  ネット累積リターン: {total_net_return*100:.2f}%")
print(f"  年率コストドラッグ（年率差分）: {annual_cost_drag*100:.2f}%/年")
print(f"  取引コスト累積（ターンオーバー比例、足し上げ）: {total_trading_cost*100:.2f}%")
print("=" * 80)


# ============================================================
# Step 6: ファイル保存（v2.1：ファイル名はv2のまま）
# ============================================================

print("\n[6/8] ファイル保存中...")

out_file1 = ANALYSIS_DIR / "daily_portfolio_returns_monthly_improved_v2.parquet"
df_daily_returns.to_parquet(out_file1, index=False)
print(f"  {out_file1}")

df_leverage_events = pd.DataFrame(leverage_events) if leverage_events else pd.DataFrame(columns=[
    "Date","Event","Reason","MonthKey","PrevState","TargetState","Overlap","Turnover","TurnoverStocks","TradingCost"
])
out_file2 = ANALYSIS_DIR / "leverage_events_monthly_improved_v2.csv"
df_leverage_events.to_csv(out_file2, index=False, encoding="utf-8-sig")
print(f"  {out_file2}")

summary_data = {
    "period_start": [first_date],
    "period_end": [last_date],
    "total_days": [total_days],
    "initial_capital": [INITIAL_CAPITAL],
    "final_wealth": [final_wealth],
    "cumulative_return": [cum_return],
    "annual_return": [ann_return],
    "annual_volatility": [ann_vol],
    "sharpe_ratio": [sharpe],
    "daily_max_dd": [daily_mdd],
    "worst_dd_date": [worst_dd_date],
    "cash_days": [cash_days],
    "cash_pct": [cash_pct],
    "avg_leverage": [avg_leverage],
    "gross_return": [total_gross_return],
    "net_return": [total_net_return],
    "annual_cost_drag": [annual_cost_drag],
    "rebalance_count": [rebalance_count],
    "skip_rebalance_count": [skip_rebalance_count],
    "skip_by_overlap": [skip_by_overlap],
    "skip_by_turnover": [skip_by_turnover],
    "total_trading_cost_sum": [total_trading_cost],
}
df_summary = pd.DataFrame(summary_data)
out_file3 = ANALYSIS_DIR / "daily_mdd_summary_monthly_improved_v2.csv"
df_summary.to_csv(out_file3, index=False, encoding="utf-8-sig")
print(f"  {out_file3}")

out_file4 = ANALYSIS_DIR / "filter_statistics_monthly_improved_v2.csv"
df_filter_stats.to_csv(out_file4, index=False, encoding="utf-8-sig")
print(f"  {out_file4}")


# ============================================================
# Step 7: 比較表（今回v2.1のみ）
# ============================================================

print("\n[7/8] 戦略比較表作成中...")

comparison_data = {
    "指標": [
        "年率リターン",
        "年率ボラティリティ",
        "シャープレシオ",
        "最大DD",
        "累積リターン",
        "CASH比率",
        "平均レバレッジ",
        "年率コストドラッグ（年率差分）",
        "取引コスト累積（足し上げ）",
        "リバランス実行回数",
        "リバランス見送り回数（OR）",
        "見送り（overlap含む）回数",
        "見送り（turnover含む）回数",
    ],
    "月次改善版 v2.1": [
        f"{ann_return*100:.2f}%",
        f"{ann_vol*100:.2f}%",
        f"{sharpe:.4f}",
        f"{daily_mdd*100:.2f}%",
        f"{cum_return*100:.2f}%",
        f"{cash_pct:.2f}%",
        f"{avg_leverage:.2f}x",
        f"{annual_cost_drag*100:.2f}%/年",
        f"{total_trading_cost*100:.2f}%",
        f"{rebalance_count}",
        f"{skip_rebalance_count}",
        f"{skip_by_overlap}",
        f"{skip_by_turnover}",
    ]
}
df_comparison = pd.DataFrame(comparison_data)

print("\n" + "=" * 80)
print("戦略比較表（今回：月次改善版 v2.1）")
print("=" * 80)
print(df_comparison.to_string(index=False))
print("=" * 80)


# ============================================================
# Step 8: 完了
# ============================================================

print("\n" + "=" * 80)
print("[8/8] 月次戦略 改善版 v2.1 バックテスト完了 ✅")
print("=" * 80)

print("\n次に見るべきポイント:")
print("1) 見送り回数が 0 → 適度に増えるか（10〜40回程度が目安）")
print("2) 取引コスト累積（足し上げ）が v2(22.02%) からさらに下がるか")
print("3) Sharpe / MaxDD が劣化していないか")


月次戦略 改善版バックテスト v2.1（CASH完全対応 + 列名自動検出）
前提：月末引けで決定 → 翌月初寄りで執行（コストは翌月第1営業日に計上、価格は日足近似）
初期資金: ¥10,000,000
TOP_N: 20
株価フィルタ: ¥800以上
売買代金上位: 70%
最低日次売買代金: ¥5,000,000
取引コスト（片道）: 0.40%（ターンオーバー比例）
レバレッジ: 1.00x（改善版：信用取引なし）
部分リバランス deadband: 5%
--------------------------------------------------------------------------------
[v2] LIQUIDITY_BUFFER_FACTOR: 0.7
[v2] SCORE_EWMA_ALPHA: 0.5
[v2] OVERLAP_SKIP_TH: 0.8 USE_OVERLAP_SKIP: True
[v2.1] SKIP_MODE: OR (turnover_skip OR overlap_skip)

[1/8] 月次スナップショット読み込み中（列名自動検出）...
検出列（先頭10列）: ['MonthEnd', 'Code', 'AdjustedClose', 'Vo', 'MarketCap', 'BM_Ratio', 'ROE', 'INV_Growth', 'Month']
銘柄コード列: 'Code' → 'Code'
列名正規化完了
期間: 2016-01-31 〜 2026-01-31
データ: 502,101 行, 5,303 銘柄

[v2] スコアEWMA（2ヶ月相当）を計算中...
  composite_score_ewma_all を追加しました

[2/8] 月次ポートフォリオ形成中（翌月ターゲット決定、CASH対応）...
Vol3閾値（90%ile）: 0.064522
ポートフォリオ形成完了: 1,812 レコード
リスクオフ月数: 32/121 (26.4%)
翌月CASH月数: 32/121 (26.4%)

フィルタ統計（全期間平均）:
  初期: 4150
  株価後: 2055
  財務後: 1645
  売買後: 1151
  流動性後: 1151
  最終選定: 15

[

In [8]:
"""
月次戦略 改善版（v2.2）：リアリスティック・バックテスト
（CASH完全対応 + 列名自動検出 + 月末引けで決定→翌月初寄りで執行）
※日次データがAdjustmentCloseのみの場合、寄り執行は厳密再現不可 → 「翌月第1営業日にコスト計上」で執行日整合（価格は日足近似）

v2（改善第2弾）
(1) 流動性制約を 0.8 → 0.7 に緩和（LIQUIDITY_BUFFER_FACTOR）
(2) スコアを2ヶ月EWMAで平滑化（alpha=0.50）
(3) overlapでリバランス見送り（閾値0.80）

v2.1（前回）
- 見送り条件を AND → OR に変更（turnover_skip OR overlap_skip）
- CASH↔株式の遷移は必ず実行（見送り禁止）

v2.2（今回：あなたの指示「1」）
- MIN_DAILY_TURNOVER を 5,000,000 → 4,000,000 に緩和（TOP_N=20を満たしやすくする）

出力（v2.2：ファイル名は v2 を踏襲）
- daily_portfolio_returns_monthly_improved_v2.parquet
- leverage_events_monthly_improved_v2.csv
- daily_mdd_summary_monthly_improved_v2.csv
- filter_statistics_monthly_improved_v2.csv
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 基本設定
# ============================================================

OUTPUT_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks")
SNAP_PATH = OUTPUT_DIR / "factors" / "month_end_snapshot.parquet"
MERGED_PARTS_DIR = OUTPUT_DIR / "merged_parts"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_daily"
ANALYSIS_DIR.mkdir(exist_ok=True)


# ============================================================
# パラメータ（改善版 + v2.x）
# ============================================================

### 基本
INITIAL_CAPITAL = 10_000_000
TOP_N = 20
VO_TOP_PCT = 0.70
DEFAULT_TRADING_UNIT = 100

### 株価・財務フィルタ
MIN_PRICE = 800
MAX_PRICE = None
ROE_MIN = -0.5
ROE_MAX = 1.0
BM_RATIO_MIN = 0.1
BM_RATIO_MAX = 10.0

### リスクオフ判定（市場状態）※旧コード踏襲
OFF_TH = -0.115
ON_TH = -0.085
TREND3_TH = -0.02
VOL3_Q = 0.90

### 取引コスト（片道）
COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.003
TOTAL_COST_PER_TRADE = COMMISSION_RATE + SLIPPAGE_RATE  # 片道0.4%

### 流動性制約
MAX_VOLUME_PARTICIPATION = 0.10

# ★ v2.2 変更点（ここだけ）
MIN_DAILY_TURNOVER = 4_000_000   # 5_000_000 -> 4_000_000

### 価格ギャップ対策（執行可能性の安全側）
PRICE_GAP_BUFFER = 0.95
ENTRY_SLIPPAGE = 0.01

### 改善版：レバレッジ固定（信用金利ゼロ）
LEVERAGE_FIXED = 1.0
MARGIN_INTEREST_RATE = 0.028  # 参考（改善版では使わない）

### 改善版：部分リバランス（デッドバンド）と条件付きリバランス
WEIGHT_ADJUST_DEADBAND = 0.05
REBALANCE_MIN_TURNOVER_STOCKS = 5
REBALANCE_MIN_TURNOVER_PCT = 0.10

# v2（改善第2弾）
LIQUIDITY_BUFFER_FACTOR = 0.70
SCORE_EWMA_ALPHA = 0.50
OVERLAP_SKIP_TH = 0.80
USE_OVERLAP_SKIP = True

# v2.1：OR見送り
SKIP_MODE = "OR"


print("=" * 80)
print("月次戦略 改善版バックテスト v2.2（CASH完全対応 + 列名自動検出）")
print("前提：月末引けで決定 → 翌月初寄りで執行（コストは翌月第1営業日に計上、価格は日足近似）")
print("=" * 80)
print(f"初期資金: ¥{INITIAL_CAPITAL:,}")
print(f"TOP_N: {TOP_N}")
print(f"株価フィルタ: ¥{MIN_PRICE}以上")
print(f"売買代金上位: {VO_TOP_PCT*100:.0f}%")
print(f"最低日次売買代金: ¥{MIN_DAILY_TURNOVER:,}  ★v2.2で緩和")
print(f"取引コスト（片道）: {TOTAL_COST_PER_TRADE*100:.2f}%（ターンオーバー比例）")
print(f"レバレッジ: {LEVERAGE_FIXED:.2f}x（信用取引なし）")
print(f"部分リバランス deadband: {WEIGHT_ADJUST_DEADBAND*100:.0f}%")
print("-" * 80)
print("[v2] LIQUIDITY_BUFFER_FACTOR:", LIQUIDITY_BUFFER_FACTOR)
print("[v2] SCORE_EWMA_ALPHA:", SCORE_EWMA_ALPHA)
print("[v2] OVERLAP_SKIP_TH:", OVERLAP_SKIP_TH, "USE_OVERLAP_SKIP:", USE_OVERLAP_SKIP)
print("[v2.1] SKIP_MODE:", SKIP_MODE, "(turnover_skip OR overlap_skip)")
print("=" * 80)


# ============================================================
# 列名自動検出・正規化
# ============================================================

def detect_stock_code_column(df):
    candidates = ["Code", "StockCode", "Symbol", "Ticker", "stock_code", "code"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"銘柄コード列が見つかりません。利用可能な列: {list(df.columns)}")

def normalize_columns(df, stock_code_col=None):
    rename_map = {}
    if stock_code_col and stock_code_col != "Code":
        rename_map[stock_code_col] = "Code"
    if "AdjustedClose" in df.columns and "AdjustmentClose" not in df.columns:
        rename_map["AdjustedClose"] = "AdjustmentClose"
    if "Volume" in df.columns and "Vo" not in df.columns:
        rename_map["Volume"] = "Vo"
    return df.rename(columns=rename_map)


# ============================================================
# v2: EWMA平滑化ユーティリティ
# ============================================================

def add_ewma_score_to_snapshot(snap: pd.DataFrame, alpha: float) -> pd.DataFrame:
    snap = snap.sort_values(["Code", "MonthEnd"]).copy()

    def _calc_month_z_and_score(g):
        g = g.copy()
        for col in ["BM_Ratio", "ROE", "INV_Growth"]:
            mean_val = g[col].mean()
            std_val = g[col].std()
            g[f"{col}_z_all"] = (g[col] - mean_val) / std_val if (std_val is not None and std_val > 0) else 0.0
        g["composite_score_raw_all"] = (
            g["BM_Ratio_z_all"] +
            g["ROE_z_all"] -
            g["INV_Growth_z_all"].fillna(0)
        )
        return g

    snap = snap.groupby("MonthEnd", group_keys=False).apply(_calc_month_z_and_score)

    snap["composite_score_ewma_all"] = (
        snap.groupby("Code")["composite_score_raw_all"]
            .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
    )
    return snap


def overlap_ratio(prev_codes, new_codes) -> float:
    prev_set = set(prev_codes)
    new_set = set(new_codes)
    if len(prev_set) == 0:
        return 0.0
    return len(prev_set & new_set) / max(len(prev_set), 1)


# ============================================================
# Step 1: 月次スナップショット読み込み
# ============================================================

print("\n[1/8] 月次スナップショット読み込み中（列名自動検出）...")
snap = pd.read_parquet(SNAP_PATH)
snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])

print(f"検出列（先頭10列）: {list(snap.columns[:10])}")

stock_code_col = detect_stock_code_column(snap)
print(f"銘柄コード列: '{stock_code_col}' → 'Code'")
snap = normalize_columns(snap, stock_code_col)

required_cols = ["MonthEnd", "Code", "AdjustmentClose", "Vo", "MarketCap", "BM_Ratio", "ROE", "INV_Growth"]
missing_cols = [c for c in required_cols if c not in snap.columns]
if missing_cols:
    print(f"不足列: {missing_cols}")
    if "AdjustmentClose" not in snap.columns:
        if "Close" in snap.columns:
            snap["AdjustmentClose"] = snap["Close"]
            print("  代替: Close → AdjustmentClose")
        elif "AdjustedClose" in snap.columns:
            snap["AdjustmentClose"] = snap["AdjustedClose"]
            print("  代替: AdjustedClose → AdjustmentClose")
    if "Vo" not in snap.columns and "Volume" in snap.columns:
        snap["Vo"] = snap["Volume"]
        print("  代替: Volume → Vo")
    missing_cols = [c for c in required_cols if c not in snap.columns]
    if missing_cols:
        raise ValueError(f"必須列が不足しており代替もできません: {missing_cols}")

snap = snap.sort_values(["Code", "MonthEnd"]).copy()
snap["MonthKey"] = snap["MonthEnd"].dt.to_period("M").astype(str)

print("列名正規化完了")
print(f"期間: {snap['MonthEnd'].min().date()} 〜 {snap['MonthEnd'].max().date()}")
print(f"データ: {len(snap):,} 行, {snap['Code'].nunique():,} 銘柄")


# ============================================================
# v2: スコアEWMAをスナップショットに付与
# ============================================================
print("\n[v2] スコアEWMA（2ヶ月相当）を計算中...")
snap = add_ewma_score_to_snapshot(snap, alpha=SCORE_EWMA_ALPHA)
print("  composite_score_ewma_all を追加しました")


# ============================================================
# Step 2: 月次ポートフォリオ形成（翌月のターゲットを決める）
# ============================================================

print("\n[2/8] 月次ポートフォリオ形成中（翌月ターゲット決定、CASH対応）...")

snap["ret_m_fwd"] = snap.groupby("Code")["AdjustmentClose"].pct_change().shift(-1)
snap["TurnoverValue"] = snap["Vo"] * snap["AdjustmentClose"]

monthly = snap.groupby("MonthEnd").apply(
    lambda g: pd.Series({
        "mkt_vw": np.average(g["ret_m_fwd"].fillna(0), weights=g["MarketCap"].fillna(1)),
        "count": len(g)
    })
).reset_index()

monthly["cum_ret"] = (1 + monthly["mkt_vw"]).cumprod()
monthly["peak"] = monthly["cum_ret"].cummax()
monthly["mkt_dd"] = (monthly["cum_ret"] / monthly["peak"]) - 1
monthly["trend3"] = monthly["mkt_vw"].rolling(3, min_periods=1).mean()
monthly["vol3"] = monthly["mkt_vw"].rolling(3, min_periods=2).std()

vol3_th = monthly["vol3"].quantile(VOL3_Q)
print(f"Vol3閾値（{VOL3_Q*100:.0f}%ile）: {vol3_th:.6f}")

snap = snap.merge(monthly[["MonthEnd", "mkt_dd", "trend3", "vol3"]], on="MonthEnd", how="left")
snap["risk_off"] = (
    (snap["mkt_dd"] <= OFF_TH) |
    (snap["trend3"] <= TREND3_TH) |
    (snap["vol3"] >= vol3_th)
)

portfolio_list = []
filter_stats = []
risk_off_months = 0

for month_end, df_month in snap.groupby("MonthEnd"):
    month_key = df_month["MonthKey"].iloc[0]
    risk_off = bool(df_month["risk_off"].iloc[0])

    stats = {
        "MonthEnd": month_end,
        "MonthKey": month_key,
        "initial_count": len(df_month),
        "risk_off": risk_off
    }

    month_key_next = (pd.Timestamp(month_end) + pd.DateOffset(months=1)).to_period("M").strftime("%Y-%m")

    if risk_off:
        risk_off_months += 1
        portfolio_list.append({
            "MonthEnd": month_end,
            "MonthKey": month_key,
            "MonthKey_next": month_key_next,
            "Code": "CASH",
            "Weight": 1.0,
            "risk_off": True
        })
        stats.update({
            "after_price": 0,
            "after_roe_bm": 0,
            "after_turnover": 0,
            "after_liquidity": 0,
            "final_count": 0,
            "selected_count": 0
        })
        filter_stats.append(stats)
        continue

    df_valid = df_month[df_month["AdjustmentClose"] >= MIN_PRICE].copy()
    if MAX_PRICE:
        df_valid = df_valid[df_valid["AdjustmentClose"] <= MAX_PRICE]
    stats["after_price"] = len(df_valid)

    df_valid = df_valid[
        (df_valid["BM_Ratio"].notna()) &
        (df_valid["ROE"].notna()) &
        (df_valid["ROE"] >= ROE_MIN) &
        (df_valid["ROE"] <= ROE_MAX) &
        (df_valid["BM_Ratio"] >= BM_RATIO_MIN) &
        (df_valid["BM_Ratio"] <= BM_RATIO_MAX)
    ]
    stats["after_roe_bm"] = len(df_valid)

    if "TurnoverValue" in df_valid.columns and len(df_valid) > 0:
        turnover_th = df_valid["TurnoverValue"].quantile(1 - VO_TOP_PCT)
        df_valid = df_valid[df_valid["TurnoverValue"] >= turnover_th]
    stats["after_turnover"] = len(df_valid)

    # ★ v2.2: MIN_DAILY_TURNOVER が緩和されるのはここ
    df_valid = df_valid[df_valid["TurnoverValue"] >= MIN_DAILY_TURNOVER]

    df_valid["MaxBuyableAmount"] = df_valid["Vo"] * MAX_VOLUME_PARTICIPATION * df_valid["AdjustmentClose"]

    target_capital = INITIAL_CAPITAL * LEVERAGE_FIXED * PRICE_GAP_BUFFER
    capital_per_stock = target_capital / TOP_N
    df_valid = df_valid[df_valid["MaxBuyableAmount"] >= capital_per_stock * LIQUIDITY_BUFFER_FACTOR]
    stats["after_liquidity"] = len(df_valid)

    if len(df_valid) < TOP_N:
        portfolio_list.append({
            "MonthEnd": month_end,
            "MonthKey": month_key,
            "MonthKey_next": month_key_next,
            "Code": "CASH",
            "Weight": 1.0,
            "risk_off": False
        })
        stats.update({"final_count": len(df_valid), "selected_count": 0})
        filter_stats.append(stats)
        continue

    score_col = "composite_score_ewma_all"
    top_n = df_valid.nlargest(TOP_N, score_col)

    stats["final_count"] = len(df_valid)
    stats["selected_count"] = len(top_n)
    filter_stats.append(stats)

    weight = 1.0 / len(top_n)
    for _, row in top_n.iterrows():
        portfolio_list.append({
            "MonthEnd": month_end,
            "MonthKey": month_key,
            "MonthKey_next": month_key_next,
            "Code": row["Code"],
            "Weight": weight,
            "risk_off": False
        })

df_portfolio = pd.DataFrame(portfolio_list)
df_filter_stats = pd.DataFrame(filter_stats)

total_months = len(df_filter_stats)
cash_months_cnt = df_portfolio[df_portfolio["Code"] == "CASH"]["MonthKey_next"].nunique()

print(f"ポートフォリオ形成完了: {len(df_portfolio):,} レコード")
print(f"リスクオフ月数: {risk_off_months}/{total_months} ({risk_off_months/total_months*100:.1f}%)")
print(f"翌月CASH月数: {cash_months_cnt}/{df_portfolio['MonthKey_next'].nunique()} ({cash_months_cnt/df_portfolio['MonthKey_next'].nunique()*100:.1f}%)")

print("\nフィルタ統計（全期間平均）:")
print(f"  初期: {df_filter_stats['initial_count'].mean():.0f}")
print(f"  株価後: {df_filter_stats['after_price'].mean():.0f}")
print(f"  財務後: {df_filter_stats['after_roe_bm'].mean():.0f}")
print(f"  売買後: {df_filter_stats['after_turnover'].mean():.0f}")
print(f"  流動性後: {df_filter_stats['after_liquidity'].mean():.0f}")
print(f"  最終選定: {df_filter_stats['selected_count'].mean():.0f}")


# ============================================================
# Step 3: 日次データ読み込み
# ============================================================

print("\n[3/8] 日次データ読み込み中（列名自動検出）...")

daily_files = sorted(MERGED_PARTS_DIR.glob("merged-part-*.parquet"))
if not daily_files:
    raise FileNotFoundError(f"日次データが見つかりません: {MERGED_PARTS_DIR}")

sample_df = pd.read_parquet(daily_files[0])
daily_stock_code_col = detect_stock_code_column(sample_df)
print(f"日次データの銘柄コード列: '{daily_stock_code_col}' → 'Code'")

daily_list = []
for f in daily_files:
    df_part = pd.read_parquet(f)
    df_part = normalize_columns(df_part, daily_stock_code_col)
    daily_list.append(df_part)

daily = pd.concat(daily_list, ignore_index=True)
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values(["Code", "Date"]).copy()

if "AdjustedClose" in daily.columns and "AdjustmentClose" not in daily.columns:
    daily = daily.rename(columns={"AdjustedClose": "AdjustmentClose"})

daily["ret_d"] = daily.groupby("Code")["AdjustmentClose"].pct_change()
daily["MonthKey"] = daily["Date"].dt.to_period("M").astype(str)

print(f"日次データ: {len(daily):,} 行, {daily['Code'].nunique():,} 銘柄")


# ============================================================
# Step 4: 日次ポートフォリオリターン計算（v2.2：OR見送り）
# ============================================================

print("\n[4/8] 日次ポートフォリオリターン計算中（v2.2：月末決定→翌月初執行 + overlap見送りOR）...")

cash_tbl = df_portfolio[df_portfolio["Code"] == "CASH"][["MonthKey_next", "risk_off"]].copy()
cash_tbl = cash_tbl.rename(columns={"MonthKey_next": "MonthKey"})
cash_tbl["is_cash_month"] = True
cash_month_set = set(cash_tbl["MonthKey"].unique())

stock_portfolio = df_portfolio[df_portfolio["Code"] != "CASH"].copy()

monthly_target = {}
for mk, g in stock_portfolio.groupby("MonthKey_next"):
    monthly_target[mk] = dict(zip(g["Code"].astype(str), g["Weight"].astype(float)))

daily_port = daily.merge(
    stock_portfolio[["MonthKey_next", "Code", "Weight"]],
    left_on=["MonthKey", "Code"],
    right_on=["MonthKey_next", "Code"],
    how="inner"
).sort_values("Date").copy()

first_trade_day_of_month = daily.groupby("MonthKey")["Date"].min().to_dict()

def calc_turnover_and_new_hold(prev_w, target_w, deadband=WEIGHT_ADJUST_DEADBAND):
    prev = prev_w.copy()
    target = target_w.copy()
    all_codes = set(prev.keys()) | set(target.keys())
    after = {}

    for c in all_codes:
        w0 = prev.get(c, 0.0)
        w1 = target.get(c, 0.0)
        if (c in prev) and (c in target):
            if abs(w1 - w0) < deadband:
                after[c] = w0
            else:
                after[c] = w1
        else:
            after[c] = w1

    after = {c: w for c, w in after.items() if w > 0}
    s = sum(after.values())
    if s > 0:
        after = {c: w/s for c, w in after.items()}

    codes2 = set(prev.keys()) | set(after.keys())
    turnover = sum(abs(after.get(c, 0.0) - prev.get(c, 0.0)) for c in codes2)
    turnover_stocks = len(set(prev.keys()) ^ set(after.keys()))
    return turnover, turnover_stocks, after


global_wealth = INITIAL_CAPITAL
global_peak = INITIAL_CAPITAL

current_hold_weights = {}
current_state = "CASH"

daily_returns = []
leverage_events = []

rebalance_count = 0
skip_rebalance_count = 0
skip_by_overlap = 0
skip_by_turnover = 0

total_trading_cost = 0.0
total_interest_cost = 0.0

all_dates = sorted(daily["Date"].unique())

for date in all_dates:
    mk = pd.Timestamp(date).to_period("M").strftime("%Y-%m")

    is_cash_month = mk in cash_month_set
    is_risk_off = False
    if is_cash_month:
        tmp = cash_tbl[cash_tbl["MonthKey"] == mk]
        if len(tmp) > 0:
            is_risk_off = bool(tmp["risk_off"].iloc[0])
    is_cash = is_cash_month or is_risk_off

    is_first_trade_day = (mk in first_trade_day_of_month) and (pd.Timestamp(date) == pd.Timestamp(first_trade_day_of_month[mk]))

    trading_cost = 0.0
    interest_cost_pct = 0.0
    leverage_used = LEVERAGE_FIXED

    if is_first_trade_day:
        target_w = monthly_target.get(mk, {})

        target_state = "CASH" if is_cash else "EQUITY"
        is_cash_transition = (current_state != target_state)

        prev_codes = list(current_hold_weights.keys())
        new_codes = list(target_w.keys())
        ov = overlap_ratio(prev_codes, new_codes) if (current_state == "EQUITY" and target_state == "EQUITY") else np.nan

        turnover, turnover_stocks, new_hold = calc_turnover_and_new_hold(current_hold_weights, target_w)

        do_skip = False
        reasons = []

        if is_cash_transition:
            do_skip = False
        else:
            if target_state == "CASH":
                do_skip = False
            else:
                turnover_skip = (turnover_stocks < REBALANCE_MIN_TURNOVER_STOCKS) and (turnover < REBALANCE_MIN_TURNOVER_PCT)
                overlap_skip = (USE_OVERLAP_SKIP and (ov >= OVERLAP_SKIP_TH))

                if turnover_skip or overlap_skip:
                    do_skip = True
                    if turnover_skip:
                        reasons.append(f"TURNOVER (to={turnover:.3f}, nswap={turnover_stocks})")
                    if overlap_skip:
                        reasons.append(f"OVERLAP (ov={ov:.3f})")

        if do_skip:
            skip_rebalance_count += 1
            if any("TURNOVER" in x for x in reasons):
                skip_by_turnover += 1
            if any("OVERLAP" in x for x in reasons):
                skip_by_overlap += 1

            leverage_events.append({
                "Date": date,
                "Event": "RebalanceSkipped",
                "Reason": " | ".join(reasons),
                "MonthKey": mk,
                "PrevState": current_state,
                "TargetState": target_state,
                "Overlap": float(ov) if ov == ov else np.nan,
                "Turnover": float(turnover),
                "TurnoverStocks": int(turnover_stocks),
                "TradingCost": 0.0
            })
        else:
            if target_state == "CASH":
                current_hold_weights = {}
                current_state = "CASH"
                rebalance_count += 1
                leverage_events.append({
                    "Date": date,
                    "Event": "RebalanceExecuted",
                    "Reason": "CASH_STATE",
                    "MonthKey": mk,
                    "PrevState": current_state,
                    "TargetState": target_state,
                    "Overlap": float(ov) if ov == ov else np.nan,
                    "Turnover": float(turnover),
                    "TurnoverStocks": int(turnover_stocks),
                    "TradingCost": 0.0
                })
            else:
                rebalance_count += 1
                trading_cost = turnover * TOTAL_COST_PER_TRADE
                total_trading_cost += trading_cost
                current_hold_weights = new_hold
                current_state = "EQUITY"
                leverage_events.append({
                    "Date": date,
                    "Event": "RebalanceExecuted",
                    "Reason": "EQUITY_STATE",
                    "MonthKey": mk,
                    "PrevState": current_state,
                    "TargetState": target_state,
                    "Overlap": float(ov) if ov == ov else np.nan,
                    "Turnover": float(turnover),
                    "TurnoverStocks": int(turnover_stocks),
                    "TradingCost": float(trading_cost)
                })

    if is_cash:
        port_ret_gross = 0.0
        port_ret_net = 0.0
    else:
        df_day = daily_port[daily_port["Date"] == date].copy()
        if len(df_day) == 0:
            continue
        df_day["weighted_ret"] = df_day["ret_d"].fillna(0) * df_day["Weight"]
        port_ret_gross = df_day["weighted_ret"].sum() * leverage_used
        port_ret_net = port_ret_gross - trading_cost - interest_cost_pct

    global_wealth *= (1 + port_ret_net)
    if global_wealth > global_peak:
        global_peak = global_wealth
    dd = (global_wealth / global_peak) - 1

    daily_returns.append({
        "Date": date,
        "MonthKey": mk,
        "port_ret_gross": port_ret_gross,
        "port_ret_net": port_ret_net,
        "trading_cost": trading_cost,
        "interest_cost": interest_cost_pct,
        "leverage": leverage_used,
        "wealth": global_wealth,
        "peak": global_peak,
        "dd": dd,
        "is_cash": is_cash,
        "is_risk_off": is_risk_off,
        "is_cash_month": is_cash_month,
        "is_rebalance_day": bool(is_first_trade_day),
        "state": current_state
    })

df_daily_returns = pd.DataFrame(daily_returns)

print(f"\n日次ポートフォリオリターン: {len(df_daily_returns):,} 日")
print(f"リバランス実行回数: {rebalance_count} 回（翌月第1営業日ベース）")
print(f"リバランス見送り回数: {skip_rebalance_count} 回（OR：turnover または overlap）")
print(f"  - overlapが理由に含まれた回数: {skip_by_overlap} 回")
print(f"  - turnoverが理由に含まれた回数: {skip_by_turnover} 回")
print(f"信用金利合計（期待0）: {total_interest_cost*100:.4f}%")
print(f"取引コスト合計（ターンオーバー比例）: {total_trading_cost*100:.2f}%")


# ============================================================
# Step 5: 日次MDD計算とサマリ
# ============================================================

print("\n[5/8] 日次MDD計算中...")

df_daily_returns["cum_ret"] = df_daily_returns["port_ret_net"].add(1).cumprod().sub(1)
df_daily_returns["cum_wealth"] = INITIAL_CAPITAL * (1 + df_daily_returns["cum_ret"])

total_days = len(df_daily_returns)
final_wealth = df_daily_returns["cum_wealth"].iloc[-1]
cum_return = (final_wealth / INITIAL_CAPITAL) - 1

first_date = df_daily_returns["Date"].min()
last_date = df_daily_returns["Date"].max()
years = (last_date - first_date).days / 365.25 if last_date > first_date else 1.0

ann_return = (1 + cum_return) ** (1 / years) - 1
ann_vol = df_daily_returns["port_ret_net"].std() * np.sqrt(252)
sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0

daily_mdd = df_daily_returns["dd"].min()
worst_dd_date = df_daily_returns.loc[df_daily_returns["dd"].idxmin(), "Date"]

cash_days = int(df_daily_returns["is_cash"].sum())
cash_pct = cash_days / total_days * 100
avg_leverage = df_daily_returns["leverage"].mean()

total_gross_return = df_daily_returns["port_ret_gross"].add(1).cumprod().iloc[-1] - 1
total_net_return = cum_return
gross_ann = (1 + total_gross_return) ** (1 / years) - 1
net_ann = (1 + total_net_return) ** (1 / years) - 1
annual_cost_drag = gross_ann - net_ann

print("\n" + "=" * 80)
print("最終結果サマリー（月次改善版 v2.2）")
print("=" * 80)
print(f"期間: {first_date.date()} 〜 {last_date.date()}")
print(f"総日数: {total_days}日 ({years:.2f}年)")
print(f"初期資金: ¥{INITIAL_CAPITAL:,}")
print(f"最終資産: ¥{final_wealth:,.0f}")
print(f"累積リターン: {cum_return*100:.2f}%")
print(f"年率リターン: {ann_return*100:.2f}%")
print(f"年率ボラティリティ: {ann_vol*100:.2f}%")
print(f"シャープレシオ: {sharpe:.4f}")
print(f"日次最大DD: {daily_mdd*100:.2f}% (日付: {worst_dd_date.date()})")
print(f"平均CASH比率: {cash_pct:.2f}%")
print(f"平均レバレッジ: {avg_leverage:.2f}x")

print("\n[v2.2] 取引コストの影響:")
print(f"  グロス累積リターン: {total_gross_return*100:.2f}%")
print(f"  ネット累積リターン: {total_net_return*100:.2f}%")
print(f"  年率コストドラッグ（年率差分）: {annual_cost_drag*100:.2f}%/年")
print(f"  取引コスト累積（足し上げ）: {total_trading_cost*100:.2f}%")
print("=" * 80)


# ============================================================
# Step 6: ファイル保存（ファイル名は v2 のまま）
# ============================================================

print("\n[6/8] ファイル保存中...")

out_file1 = ANALYSIS_DIR / "daily_portfolio_returns_monthly_improved_v2.parquet"
df_daily_returns.to_parquet(out_file1, index=False)
print(f"  {out_file1}")

df_leverage_events = pd.DataFrame(leverage_events) if leverage_events else pd.DataFrame(columns=[
    "Date","Event","Reason","MonthKey","PrevState","TargetState","Overlap","Turnover","TurnoverStocks","TradingCost"
])
out_file2 = ANALYSIS_DIR / "leverage_events_monthly_improved_v2.csv"
df_leverage_events.to_csv(out_file2, index=False, encoding="utf-8-sig")
print(f"  {out_file2}")

summary_data = {
    "period_start": [first_date],
    "period_end": [last_date],
    "total_days": [total_days],
    "initial_capital": [INITIAL_CAPITAL],
    "final_wealth": [final_wealth],
    "cumulative_return": [cum_return],
    "annual_return": [ann_return],
    "annual_volatility": [ann_vol],
    "sharpe_ratio": [sharpe],
    "daily_max_dd": [daily_mdd],
    "worst_dd_date": [worst_dd_date],
    "cash_days": [cash_days],
    "cash_pct": [cash_pct],
    "avg_leverage": [avg_leverage],
    "gross_return": [total_gross_return],
    "net_return": [total_net_return],
    "annual_cost_drag": [annual_cost_drag],
    "rebalance_count": [rebalance_count],
    "skip_rebalance_count": [skip_rebalance_count],
    "skip_by_overlap": [skip_by_overlap],
    "skip_by_turnover": [skip_by_turnover],
    "total_trading_cost_sum": [total_trading_cost],
}
df_summary = pd.DataFrame(summary_data)
out_file3 = ANALYSIS_DIR / "daily_mdd_summary_monthly_improved_v2.csv"
df_summary.to_csv(out_file3, index=False, encoding="utf-8-sig")
print(f"  {out_file3}")

out_file4 = ANALYSIS_DIR / "filter_statistics_monthly_improved_v2.csv"
df_filter_stats.to_csv(out_file4, index=False, encoding="utf-8-sig")
print(f"  {out_file4}")


# ============================================================
# Step 7: 比較表
# ============================================================

print("\n[7/8] 戦略比較表作成中...")

comparison_data = {
    "指標": [
        "年率リターン",
        "年率ボラティリティ",
        "シャープレシオ",
        "最大DD",
        "累積リターン",
        "CASH比率",
        "平均レバレッジ",
        "年率コストドラッグ（年率差分）",
        "取引コスト累積（足し上げ）",
        "リバランス実行回数",
        "リバランス見送り回数（OR）",
        "見送り（overlap含む）回数",
        "見送り（turnover含む）回数",
    ],
    "月次改善版 v2.2": [
        f"{ann_return*100:.2f}%",
        f"{ann_vol*100:.2f}%",
        f"{sharpe:.4f}",
        f"{daily_mdd*100:.2f}%",
        f"{cum_return*100:.2f}%",
        f"{cash_pct:.2f}%",
        f"{avg_leverage:.2f}x",
        f"{annual_cost_drag*100:.2f}%/年",
        f"{total_trading_cost*100:.2f}%",
        f"{rebalance_count}",
        f"{skip_rebalance_count}",
        f"{skip_by_overlap}",
        f"{skip_by_turnover}",
    ]
}
df_comparison = pd.DataFrame(comparison_data)

print("\n" + "=" * 80)
print("戦略比較表（今回：月次改善版 v2.2）")
print("=" * 80)
print(df_comparison.to_string(index=False))
print("=" * 80)


# ============================================================
# Step 8: 完了
# ============================================================

print("\n" + "=" * 80)
print("[8/8] 月次戦略 改善版 v2.2 バックテスト完了 ✅")
print("=" * 80)

print("\n次に見るべきポイント（v2.2）:")
print("1) Step2の『最終選定』が 15 → 20 に近づくか")
print("2) 翌月CASH月数（26%前後）が維持されるか")
print("3) 取引コスト累積が v2.1(19.98%) からどう動くか")
print("4) Sharpe / MaxDD が維持されるか")


月次戦略 改善版バックテスト v2.2（CASH完全対応 + 列名自動検出）
前提：月末引けで決定 → 翌月初寄りで執行（コストは翌月第1営業日に計上、価格は日足近似）
初期資金: ¥10,000,000
TOP_N: 20
株価フィルタ: ¥800以上
売買代金上位: 70%
最低日次売買代金: ¥4,000,000  ★v2.2で緩和
取引コスト（片道）: 0.40%（ターンオーバー比例）
レバレッジ: 1.00x（信用取引なし）
部分リバランス deadband: 5%
--------------------------------------------------------------------------------
[v2] LIQUIDITY_BUFFER_FACTOR: 0.7
[v2] SCORE_EWMA_ALPHA: 0.5
[v2] OVERLAP_SKIP_TH: 0.8 USE_OVERLAP_SKIP: True
[v2.1] SKIP_MODE: OR (turnover_skip OR overlap_skip)

[1/8] 月次スナップショット読み込み中（列名自動検出）...
検出列（先頭10列）: ['MonthEnd', 'Code', 'AdjustedClose', 'Vo', 'MarketCap', 'BM_Ratio', 'ROE', 'INV_Growth', 'Month']
銘柄コード列: 'Code' → 'Code'
列名正規化完了
期間: 2016-01-31 〜 2026-01-31
データ: 502,101 行, 5,303 銘柄

[v2] スコアEWMA（2ヶ月相当）を計算中...
  composite_score_ewma_all を追加しました

[2/8] 月次ポートフォリオ形成中（翌月ターゲット決定、CASH対応）...
Vol3閾値（90%ile）: 0.064522
ポートフォリオ形成完了: 1,812 レコード
リスクオフ月数: 32/121 (26.4%)
翌月CASH月数: 32/121 (26.4%)

フィルタ統計（全期間平均）:
  初期: 4150
  株価後: 2055
  財務後: 1645
  売買後: 1151
  流動性後: 1151
  最終選定:

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 基本設定（あなたのv2.2と同じ）
# ============================================================
OUTPUT_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks")
SNAP_PATH = OUTPUT_DIR / "factors" / "month_end_snapshot.parquet"
MERGED_PARTS_DIR = OUTPUT_DIR / "merged_parts"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_daily"
ANALYSIS_DIR.mkdir(exist_ok=True)

# ============================================================
# パラメータ（あなたのv2.2と同じ）
# ============================================================
INITIAL_CAPITAL = 10_000_000
TOP_N = 20
VO_TOP_PCT = 0.70
MIN_PRICE = 800
MAX_PRICE = None
ROE_MIN = -0.5
ROE_MAX = 1.0
BM_RATIO_MIN = 0.1
BM_RATIO_MAX = 10.0

OFF_TH = -0.115
TREND3_TH = -0.02

COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.003
TOTAL_COST_PER_TRADE = COMMISSION_RATE + SLIPPAGE_RATE  # 片道0.4%

MAX_VOLUME_PARTICIPATION = 0.10
MIN_DAILY_TURNOVER = 4_000_000  # v2.2の設定（結果には影響しないことが確定済み）

PRICE_GAP_BUFFER = 0.95
LEVERAGE_FIXED = 1.0

WEIGHT_ADJUST_DEADBAND = 0.05
REBALANCE_MIN_TURNOVER_STOCKS = 5
REBALANCE_MIN_TURNOVER_PCT = 0.10

LIQUIDITY_BUFFER_FACTOR = 0.70
SCORE_EWMA_ALPHA = 0.50
OVERLAP_SKIP_TH = 0.80
USE_OVERLAP_SKIP = True

# 比較したいVOL3_Q
VOL3_Q_LIST = [0.90, 0.92, 0.94]

# ============================================================
# ユーティリティ（あなたのv2.2と同等）
# ============================================================
def detect_stock_code_column(df):
    candidates = ["Code", "StockCode", "Symbol", "Ticker", "stock_code", "code"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"銘柄コード列が見つかりません。利用可能な列: {list(df.columns)}")

def normalize_columns(df, stock_code_col=None):
    rename_map = {}
    if stock_code_col and stock_code_col != "Code":
        rename_map[stock_code_col] = "Code"
    if "AdjustedClose" in df.columns and "AdjustmentClose" not in df.columns:
        rename_map["AdjustedClose"] = "AdjustmentClose"
    if "Volume" in df.columns and "Vo" not in df.columns:
        rename_map["Volume"] = "Vo"
    return df.rename(columns=rename_map)

def add_ewma_score_to_snapshot(snap: pd.DataFrame, alpha: float) -> pd.DataFrame:
    snap = snap.sort_values(["Code", "MonthEnd"]).copy()

    def _calc_month_z_and_score(g):
        g = g.copy()
        for col in ["BM_Ratio", "ROE", "INV_Growth"]:
            mean_val = g[col].mean()
            std_val = g[col].std()
            g[f"{col}_z_all"] = (g[col] - mean_val) / std_val if (std_val is not None and std_val > 0) else 0.0
        g["composite_score_raw_all"] = (
            g["BM_Ratio_z_all"] +
            g["ROE_z_all"] -
            g["INV_Growth_z_all"].fillna(0)
        )
        return g

    snap = snap.groupby("MonthEnd", group_keys=False).apply(_calc_month_z_and_score)
    snap["composite_score_ewma_all"] = (
        snap.groupby("Code")["composite_score_raw_all"]
            .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
    )
    return snap

def overlap_ratio(prev_codes, new_codes) -> float:
    prev_set = set(prev_codes)
    new_set = set(new_codes)
    if len(prev_set) == 0:
        return 0.0
    return len(prev_set & new_set) / max(len(prev_set), 1)

def calc_turnover_and_new_hold(prev_w, target_w, deadband=WEIGHT_ADJUST_DEADBAND):
    prev = prev_w.copy()
    target = target_w.copy()
    all_codes = set(prev.keys()) | set(target.keys())
    after = {}

    for c in all_codes:
        w0 = prev.get(c, 0.0)
        w1 = target.get(c, 0.0)
        if (c in prev) and (c in target):
            if abs(w1 - w0) < deadband:
                after[c] = w0
            else:
                after[c] = w1
        else:
            after[c] = w1

    after = {c: w for c, w in after.items() if w > 0}
    s = sum(after.values())
    if s > 0:
        after = {c: w/s for c, w in after.items()}

    codes2 = set(prev.keys()) | set(after.keys())
    turnover = sum(abs(after.get(c, 0.0) - prev.get(c, 0.0)) for c in codes2)
    turnover_stocks = len(set(prev.keys()) ^ set(after.keys()))
    return turnover, turnover_stocks, after

def summarize_results(df_daily_returns: pd.DataFrame, total_trading_cost_sum: float):
    df = df_daily_returns.copy()
    df["cum_ret"] = df["port_ret_net"].add(1).cumprod().sub(1)
    df["cum_wealth"] = INITIAL_CAPITAL * (1 + df["cum_ret"])

    first_date = df["Date"].min()
    last_date = df["Date"].max()
    years = (last_date - first_date).days / 365.25 if last_date > first_date else 1.0

    final_wealth = df["cum_wealth"].iloc[-1]
    cum_return = final_wealth / INITIAL_CAPITAL - 1
    ann_return = (1 + cum_return) ** (1 / years) - 1
    ann_vol = df["port_ret_net"].std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0

    dd = df["dd"].min()
    dd_date = df.loc[df["dd"].idxmin(), "Date"]

    cash_pct = df["is_cash"].mean() * 100

    return {
        "期間開始": first_date.date(),
        "期間終了": last_date.date(),
        "CASH月数": None,  # 後で月次側で埋める
        "年率(%)": ann_return * 100,
        "Sharpe": sharpe,
        "最大DD(%)": dd * 100,
        "最大DD日": dd_date.date(),
        "取引コスト累積(%)": total_trading_cost_sum * 100,
        "平均CASH比率(%)": cash_pct,
        "最終資産": final_wealth
    }

# ============================================================
# データロード（1回だけ）
# ============================================================
print("スナップショット読込中...")
snap = pd.read_parquet(SNAP_PATH)
snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
snap = normalize_columns(snap, detect_stock_code_column(snap))
snap = snap.sort_values(["Code", "MonthEnd"]).copy()
snap["MonthKey"] = snap["MonthEnd"].dt.to_period("M").astype(str)

print("EWMAスコア計算中...")
snap = add_ewma_score_to_snapshot(snap, alpha=SCORE_EWMA_ALPHA)

print("日次データ読込中...")
daily_files = sorted(MERGED_PARTS_DIR.glob("merged-part-*.parquet"))
sample_df = pd.read_parquet(daily_files[0])
daily_stock_code_col = detect_stock_code_column(sample_df)

daily_list = []
for f in daily_files:
    df_part = pd.read_parquet(f)
    df_part = normalize_columns(df_part, daily_stock_code_col)
    daily_list.append(df_part)

daily = pd.concat(daily_list, ignore_index=True)
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values(["Code", "Date"]).copy()
if "AdjustedClose" in daily.columns and "AdjustmentClose" not in daily.columns:
    daily = daily.rename(columns={"AdjustedClose": "AdjustmentClose"})
daily["ret_d"] = daily.groupby("Code")["AdjustmentClose"].pct_change()
daily["MonthKey"] = daily["Date"].dt.to_period("M").astype(str)

first_trade_day_of_month = daily.groupby("MonthKey")["Date"].min().to_dict()
all_dates = sorted(daily["Date"].unique())

# ============================================================
# 1回分のバックテスト（VOL3_Qだけ可変）
# ============================================================
def run_backtest_once(vol3_q: float) -> dict:
    snap2 = snap.copy()

    # 月次市場状態
    snap2["ret_m_fwd"] = snap2.groupby("Code")["AdjustmentClose"].pct_change().shift(-1)
    snap2["TurnoverValue"] = snap2["Vo"] * snap2["AdjustmentClose"]

    monthly = snap2.groupby("MonthEnd").apply(
        lambda g: pd.Series({
            "mkt_vw": np.average(g["ret_m_fwd"].fillna(0), weights=g["MarketCap"].fillna(1)),
            "count": len(g)
        })
    ).reset_index()

    monthly["cum_ret"] = (1 + monthly["mkt_vw"]).cumprod()
    monthly["peak"] = monthly["cum_ret"].cummax()
    monthly["mkt_dd"] = (monthly["cum_ret"] / monthly["peak"]) - 1
    monthly["trend3"] = monthly["mkt_vw"].rolling(3, min_periods=1).mean()
    monthly["vol3"] = monthly["mkt_vw"].rolling(3, min_periods=2).std()

    vol3_th = monthly["vol3"].quantile(vol3_q)

    snap2 = snap2.merge(monthly[["MonthEnd", "mkt_dd", "trend3", "vol3"]], on="MonthEnd", how="left")
    snap2["risk_off"] = (
        (snap2["mkt_dd"] <= OFF_TH) |
        (snap2["trend3"] <= TREND3_TH) |
        (snap2["vol3"] >= vol3_th)
    )

    # 月次ターゲット（翌月適用）
    portfolio_list = []
    filter_stats = []
    risk_off_months = 0

    for month_end, df_month in snap2.groupby("MonthEnd"):
        risk_off = bool(df_month["risk_off"].iloc[0])
        month_key_next = (pd.Timestamp(month_end) + pd.DateOffset(months=1)).to_period("M").strftime("%Y-%m")

        if risk_off:
            risk_off_months += 1
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0, "risk_off": True})
            filter_stats.append({"MonthEnd": month_end, "risk_off": True, "selected_count": 0})
            continue

        df_valid = df_month[df_month["AdjustmentClose"] >= MIN_PRICE].copy()
        df_valid = df_valid[
            (df_valid["BM_Ratio"].notna()) & (df_valid["ROE"].notna()) &
            (df_valid["ROE"] >= ROE_MIN) & (df_valid["ROE"] <= ROE_MAX) &
            (df_valid["BM_Ratio"] >= BM_RATIO_MIN) & (df_valid["BM_Ratio"] <= BM_RATIO_MAX)
        ].copy()

        if len(df_valid) > 0:
            turnover_th = df_valid["TurnoverValue"].quantile(1 - VO_TOP_PCT)
            df_valid = df_valid[df_valid["TurnoverValue"] >= turnover_th]

        df_valid = df_valid[df_valid["TurnoverValue"] >= MIN_DAILY_TURNOVER]

        df_valid["MaxBuyableAmount"] = df_valid["Vo"] * MAX_VOLUME_PARTICIPATION * df_valid["AdjustmentClose"]
        target_capital = INITIAL_CAPITAL * LEVERAGE_FIXED * PRICE_GAP_BUFFER
        capital_per_stock = target_capital / TOP_N
        df_valid = df_valid[df_valid["MaxBuyableAmount"] >= capital_per_stock * LIQUIDITY_BUFFER_FACTOR]

        if len(df_valid) < TOP_N:
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0, "risk_off": False})
            filter_stats.append({"MonthEnd": month_end, "risk_off": False, "selected_count": 0})
            continue

        top_n = df_valid.nlargest(TOP_N, "composite_score_ewma_all")
        w = 1.0 / len(top_n)
        for _, r in top_n.iterrows():
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": r["Code"], "Weight": w, "risk_off": False})

        filter_stats.append({"MonthEnd": month_end, "risk_off": False, "selected_count": len(top_n)})

    df_portfolio = pd.DataFrame(portfolio_list)
    df_filter_stats = pd.DataFrame(filter_stats)

    cash_months_cnt = df_portfolio[df_portfolio["Code"] == "CASH"]["MonthKey_next"].nunique()

    cash_tbl = df_portfolio[df_portfolio["Code"] == "CASH"][["MonthKey_next"]].rename(columns={"MonthKey_next": "MonthKey"})
    cash_month_set = set(cash_tbl["MonthKey"].unique())

    stock_portfolio = df_portfolio[df_portfolio["Code"] != "CASH"].copy()
    monthly_target = {}
    for mk, g in stock_portfolio.groupby("MonthKey_next"):
        monthly_target[mk] = dict(zip(g["Code"].astype(str), g["Weight"].astype(float)))

    daily_port = daily.merge(
        stock_portfolio[["MonthKey_next", "Code", "Weight"]],
        left_on=["MonthKey", "Code"],
        right_on=["MonthKey_next", "Code"],
        how="inner"
    ).sort_values("Date").copy()

    # 日次バックテスト
    global_wealth = INITIAL_CAPITAL
    global_peak = INITIAL_CAPITAL
    current_hold_weights = {}
    current_state = "CASH"

    rebalance_exec = 0
    rebalance_skip = 0
    total_trading_cost = 0.0

    daily_returns = []

    for dt in all_dates:
        mk = pd.Timestamp(dt).to_period("M").strftime("%Y-%m")
        is_cash = mk in cash_month_set

        is_first = (mk in first_trade_day_of_month) and (pd.Timestamp(dt) == pd.Timestamp(first_trade_day_of_month[mk]))
        trading_cost = 0.0

        if is_first:
            target_w = monthly_target.get(mk, {})
            target_state = "CASH" if is_cash else "EQUITY"
            is_cash_transition = (current_state != target_state)

            prev_codes = list(current_hold_weights.keys())
            new_codes = list(target_w.keys())
            ov = overlap_ratio(prev_codes, new_codes) if (current_state == "EQUITY" and target_state == "EQUITY") else np.nan

            turnover, turnover_stocks, new_hold = calc_turnover_and_new_hold(current_hold_weights, target_w)

            if is_cash_transition:
                do_skip = False
            else:
                if target_state == "CASH":
                    do_skip = False
                else:
                    turnover_skip = (turnover_stocks < REBALANCE_MIN_TURNOVER_STOCKS) and (turnover < REBALANCE_MIN_TURNOVER_PCT)
                    overlap_skip = (USE_OVERLAP_SKIP and (ov >= OVERLAP_SKIP_TH))
                    do_skip = (turnover_skip or overlap_skip)

            if do_skip:
                rebalance_skip += 1
            else:
                rebalance_exec += 1
                if target_state == "CASH":
                    current_hold_weights = {}
                    current_state = "CASH"
                else:
                    trading_cost = turnover * TOTAL_COST_PER_TRADE
                    total_trading_cost += trading_cost
                    current_hold_weights = new_hold
                    current_state = "EQUITY"

        if is_cash:
            port_ret_gross = 0.0
            port_ret_net = 0.0
        else:
            df_day = daily_port[daily_port["Date"] == dt].copy()
            if len(df_day) == 0:
                continue
            df_day["weighted_ret"] = df_day["ret_d"].fillna(0) * df_day["Weight"]
            port_ret_gross = df_day["weighted_ret"].sum() * LEVERAGE_FIXED
            port_ret_net = port_ret_gross - trading_cost

        global_wealth *= (1 + port_ret_net)
        if global_wealth > global_peak:
            global_peak = global_wealth
        dd = global_wealth / global_peak - 1

        daily_returns.append({
            "Date": dt,
            "port_ret_net": port_ret_net,
            "port_ret_gross": port_ret_gross,
            "trading_cost": trading_cost,
            "dd": dd,
            "is_cash": is_cash
        })

    df_daily_returns = pd.DataFrame(daily_returns)
    out = summarize_results(df_daily_returns, total_trading_cost)
    out["CASH月数"] = int(cash_months_cnt)
    out["risk_off月数"] = int(risk_off_months)
    out["VOL3_Q"] = vol3_q
    out["Vol3閾値"] = float(vol3_th)
    out["リバランス実行回数"] = int(rebalance_exec)
    out["リバランス見送り回数"] = int(rebalance_skip)
    return out

# ============================================================
# 3パターン実行 → 日本語で比較表
# ============================================================
results = []
for q in VOL3_Q_LIST:
    print(f"\n実行中: VOL3_Q={q:.2f} ...")
    results.append(run_backtest_once(q))

df_cmp = pd.DataFrame(results)

# 見やすい列順
cols = [
    "VOL3_Q", "Vol3閾値",
    "risk_off月数", "CASH月数", "平均CASH比率(%)",
    "年率(%)", "Sharpe", "最大DD(%)", "最大DD日",
    "取引コスト累積(%)",
    "リバランス実行回数", "リバランス見送り回数",
    "最終資産",
]
df_cmp = df_cmp[cols].copy()

# 表示用の丸め
df_cmp["Vol3閾値"] = df_cmp["Vol3閾値"].round(6)
df_cmp["平均CASH比率(%)"] = df_cmp["平均CASH比率(%)"].round(2)
df_cmp["年率(%)"] = df_cmp["年率(%)"].round(2)
df_cmp["Sharpe"] = df_cmp["Sharpe"].round(4)
df_cmp["最大DD(%)"] = df_cmp["最大DD(%)"].round(2)
df_cmp["取引コスト累積(%)"] = df_cmp["取引コスト累積(%)"].round(2)
df_cmp["最終資産"] = df_cmp["最終資産"].round(0).astype(np.int64)

print("\n" + "="*90)
print("【VOL3_Q 3パターン比較（日本語）】")
print("="*90)
print(df_cmp.to_string(index=False))

out_path = ANALYSIS_DIR / "comparison_vol3q_ja.csv"
df_cmp.to_csv(out_path, index=False, encoding="utf-8-sig")
print("\n保存しました:", out_path)


スナップショット読込中...
EWMAスコア計算中...
日次データ読込中...

実行中: VOL3_Q=0.90 ...

実行中: VOL3_Q=0.92 ...

実行中: VOL3_Q=0.94 ...

【VOL3_Q 3パターン比較（日本語）】
 VOL3_Q   Vol3閾値  risk_off月数  CASH月数  平均CASH比率(%)  年率(%)  Sharpe  最大DD(%)      最大DD日  取引コスト累積(%)  リバランス実行回数  リバランス見送り回数     最終資産
   0.90 0.064522          32      32        26.67  20.59  1.3250   -21.93 2024-08-05       19.98        101          20 64297721
   0.92 0.064933          31      31        25.80  21.02  1.3468   -21.93 2024-08-05       20.38        101          20 66617489
   0.94 0.072170          31      31        25.80  21.02  1.3468   -21.93 2024-08-05       20.38        101          20 66617489

保存しました: C:\Users\yongr\Project\merged_data_all_stocks\analysis_daily\comparison_vol3q_ja.csv


In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 基本設定（あなたの環境と同じ）
# ============================================================
OUTPUT_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks")
SNAP_PATH = OUTPUT_DIR / "factors" / "month_end_snapshot.parquet"
MERGED_PARTS_DIR = OUTPUT_DIR / "merged_parts"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_daily"
ANALYSIS_DIR.mkdir(exist_ok=True)

# ============================================================
# 固定パラメータ（現状のv2.1/2.2系を踏襲）
# ============================================================
INITIAL_CAPITAL = 10_000_000
TOP_N = 20
VO_TOP_PCT = 0.70
MIN_PRICE = 800
ROE_MIN = -0.5
ROE_MAX = 1.0
BM_RATIO_MIN = 0.1
BM_RATIO_MAX = 10.0

OFF_TH = -0.115
VOL3_Q_FIXED = 0.92  # ★固定

COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.003
TOTAL_COST_PER_TRADE = COMMISSION_RATE + SLIPPAGE_RATE  # 片道0.4%

MAX_VOLUME_PARTICIPATION = 0.10
MIN_DAILY_TURNOVER = 5_000_000  # ここは結果に効いていないことが確定済みだが、基準コードに合わせる

PRICE_GAP_BUFFER = 0.95
LEVERAGE_FIXED = 1.0

WEIGHT_ADJUST_DEADBAND = 0.05
REBALANCE_MIN_TURNOVER_STOCKS = 5
REBALANCE_MIN_TURNOVER_PCT = 0.10

LIQUIDITY_BUFFER_FACTOR = 0.70
SCORE_EWMA_ALPHA = 0.50
OVERLAP_SKIP_TH = 0.80
USE_OVERLAP_SKIP = True

# 比較したいTREND3_TH
TREND3_TH_LIST = [-0.020, -0.025, -0.030]

# ============================================================
# ユーティリティ
# ============================================================
def detect_stock_code_column(df):
    candidates = ["Code", "StockCode", "Symbol", "Ticker", "stock_code", "code"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"銘柄コード列が見つかりません。利用可能な列: {list(df.columns)}")

def normalize_columns(df, stock_code_col=None):
    rename_map = {}
    if stock_code_col and stock_code_col != "Code":
        rename_map[stock_code_col] = "Code"
    if "AdjustedClose" in df.columns and "AdjustmentClose" not in df.columns:
        rename_map["AdjustedClose"] = "AdjustmentClose"
    if "Volume" in df.columns and "Vo" not in df.columns:
        rename_map["Volume"] = "Vo"
    return df.rename(columns=rename_map)

def add_ewma_score_to_snapshot(snap: pd.DataFrame, alpha: float) -> pd.DataFrame:
    snap = snap.sort_values(["Code", "MonthEnd"]).copy()

    def _calc_month_z_and_score(g):
        g = g.copy()
        for col in ["BM_Ratio", "ROE", "INV_Growth"]:
            mean_val = g[col].mean()
            std_val = g[col].std()
            g[f"{col}_z_all"] = (g[col] - mean_val) / std_val if (std_val is not None and std_val > 0) else 0.0
        g["composite_score_raw_all"] = (
            g["BM_Ratio_z_all"] +
            g["ROE_z_all"] -
            g["INV_Growth_z_all"].fillna(0)
        )
        return g

    snap = snap.groupby("MonthEnd", group_keys=False).apply(_calc_month_z_and_score)
    snap["composite_score_ewma_all"] = (
        snap.groupby("Code")["composite_score_raw_all"]
            .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
    )
    return snap

def overlap_ratio(prev_codes, new_codes) -> float:
    prev_set = set(prev_codes)
    new_set = set(new_codes)
    if len(prev_set) == 0:
        return 0.0
    return len(prev_set & new_set) / max(len(prev_set), 1)

def calc_turnover_and_new_hold(prev_w, target_w, deadband=WEIGHT_ADJUST_DEADBAND):
    prev = prev_w.copy()
    target = target_w.copy()
    all_codes = set(prev.keys()) | set(target.keys())
    after = {}

    for c in all_codes:
        w0 = prev.get(c, 0.0)
        w1 = target.get(c, 0.0)
        if (c in prev) and (c in target):
            if abs(w1 - w0) < deadband:
                after[c] = w0
            else:
                after[c] = w1
        else:
            after[c] = w1

    after = {c: w for c, w in after.items() if w > 0}
    s = sum(after.values())
    if s > 0:
        after = {c: w/s for c, w in after.items()}

    codes2 = set(prev.keys()) | set(after.keys())
    turnover = sum(abs(after.get(c, 0.0) - prev.get(c, 0.0)) for c in codes2)
    turnover_stocks = len(set(prev.keys()) ^ set(after.keys()))
    return turnover, turnover_stocks, after

def summarize_results(df_daily_returns: pd.DataFrame, total_trading_cost_sum: float):
    df = df_daily_returns.copy()
    df["cum_ret"] = df["port_ret_net"].add(1).cumprod().sub(1)
    df["cum_wealth"] = INITIAL_CAPITAL * (1 + df["cum_ret"])

    first_date = df["Date"].min()
    last_date = df["Date"].max()
    years = (last_date - first_date).days / 365.25 if last_date > first_date else 1.0

    final_wealth = df["cum_wealth"].iloc[-1]
    cum_return = final_wealth / INITIAL_CAPITAL - 1
    ann_return = (1 + cum_return) ** (1 / years) - 1
    ann_vol = df["port_ret_net"].std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0

    dd = df["dd"].min()
    dd_date = df.loc[df["dd"].idxmin(), "Date"]

    cash_pct = df["is_cash"].mean() * 100

    return {
        "期間開始": first_date.date(),
        "期間終了": last_date.date(),
        "年率(%)": ann_return * 100,
        "Sharpe": sharpe,
        "最大DD(%)": dd * 100,
        "最大DD日": dd_date.date(),
        "取引コスト累積(%)": total_trading_cost_sum * 100,
        "平均CASH比率(%)": cash_pct,
        "最終資産": final_wealth
    }

# ============================================================
# データロード（1回だけ）
# ============================================================
print("スナップショット読込中...")
snap = pd.read_parquet(SNAP_PATH)
snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
snap = normalize_columns(snap, detect_stock_code_column(snap))
snap = snap.sort_values(["Code", "MonthEnd"]).copy()
snap["MonthKey"] = snap["MonthEnd"].dt.to_period("M").astype(str)

print("EWMAスコア計算中...")
snap = add_ewma_score_to_snapshot(snap, alpha=SCORE_EWMA_ALPHA)

print("日次データ読込中...")
daily_files = sorted(MERGED_PARTS_DIR.glob("merged-part-*.parquet"))
sample_df = pd.read_parquet(daily_files[0])
daily_stock_code_col = detect_stock_code_column(sample_df)

daily_list = []
for f in daily_files:
    df_part = pd.read_parquet(f)
    df_part = normalize_columns(df_part, daily_stock_code_col)
    daily_list.append(df_part)

daily = pd.concat(daily_list, ignore_index=True)
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values(["Code", "Date"]).copy()
if "AdjustedClose" in daily.columns and "AdjustmentClose" not in daily.columns:
    daily = daily.rename(columns={"AdjustedClose": "AdjustmentClose"})
daily["ret_d"] = daily.groupby("Code")["AdjustmentClose"].pct_change()
daily["MonthKey"] = daily["Date"].dt.to_period("M").astype(str)

first_trade_day_of_month = daily.groupby("MonthKey")["Date"].min().to_dict()
all_dates = sorted(daily["Date"].unique())

# ============================================================
# 1回分のバックテスト（TREND3_THだけ可変）
# ============================================================
def run_backtest_once(trend3_th: float) -> dict:
    snap2 = snap.copy()

    # 月次市場状態
    snap2["ret_m_fwd"] = snap2.groupby("Code")["AdjustmentClose"].pct_change().shift(-1)
    snap2["TurnoverValue"] = snap2["Vo"] * snap2["AdjustmentClose"]

    monthly = snap2.groupby("MonthEnd").apply(
        lambda g: pd.Series({
            "mkt_vw": np.average(g["ret_m_fwd"].fillna(0), weights=g["MarketCap"].fillna(1)),
            "count": len(g)
        })
    ).reset_index()

    monthly["cum_ret"] = (1 + monthly["mkt_vw"]).cumprod()
    monthly["peak"] = monthly["cum_ret"].cummax()
    monthly["mkt_dd"] = (monthly["cum_ret"] / monthly["peak"]) - 1
    monthly["trend3"] = monthly["mkt_vw"].rolling(3, min_periods=1).mean()
    monthly["vol3"] = monthly["mkt_vw"].rolling(3, min_periods=2).std()

    vol3_th = monthly["vol3"].quantile(VOL3_Q_FIXED)

    snap2 = snap2.merge(monthly[["MonthEnd", "mkt_dd", "trend3", "vol3"]], on="MonthEnd", how="left")
    snap2["risk_off"] = (
        (snap2["mkt_dd"] <= OFF_TH) |
        (snap2["trend3"] <= trend3_th) |
        (snap2["vol3"] >= vol3_th)
    )

    # 月次ターゲット（翌月適用）
    portfolio_list = []
    filter_stats = []
    risk_off_months = 0

    for month_end, df_month in snap2.groupby("MonthEnd"):
        risk_off = bool(df_month["risk_off"].iloc[0])
        month_key_next = (pd.Timestamp(month_end) + pd.DateOffset(months=1)).to_period("M").strftime("%Y-%m")

        if risk_off:
            risk_off_months += 1
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0, "risk_off": True})
            filter_stats.append({"MonthEnd": month_end, "risk_off": True, "selected_count": 0})
            continue

        df_valid = df_month[df_month["AdjustmentClose"] >= MIN_PRICE].copy()
        df_valid = df_valid[
            (df_valid["BM_Ratio"].notna()) & (df_valid["ROE"].notna()) &
            (df_valid["ROE"] >= ROE_MIN) & (df_valid["ROE"] <= ROE_MAX) &
            (df_valid["BM_Ratio"] >= BM_RATIO_MIN) & (df_valid["BM_Ratio"] <= BM_RATIO_MAX)
        ].copy()

        if len(df_valid) > 0:
            turnover_th = df_valid["TurnoverValue"].quantile(1 - VO_TOP_PCT)
            df_valid = df_valid[df_valid["TurnoverValue"] >= turnover_th]

        df_valid = df_valid[df_valid["TurnoverValue"] >= MIN_DAILY_TURNOVER]

        df_valid["MaxBuyableAmount"] = df_valid["Vo"] * MAX_VOLUME_PARTICIPATION * df_valid["AdjustmentClose"]
        target_capital = INITIAL_CAPITAL * LEVERAGE_FIXED * PRICE_GAP_BUFFER
        capital_per_stock = target_capital / TOP_N
        df_valid = df_valid[df_valid["MaxBuyableAmount"] >= capital_per_stock * LIQUIDITY_BUFFER_FACTOR]

        if len(df_valid) < TOP_N:
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0, "risk_off": False})
            filter_stats.append({"MonthEnd": month_end, "risk_off": False, "selected_count": 0})
            continue

        top_n = df_valid.nlargest(TOP_N, "composite_score_ewma_all")
        w = 1.0 / len(top_n)
        for _, r in top_n.iterrows():
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": r["Code"], "Weight": w, "risk_off": False})

        filter_stats.append({"MonthEnd": month_end, "risk_off": False, "selected_count": len(top_n)})

    df_portfolio = pd.DataFrame(portfolio_list)
    df_filter_stats = pd.DataFrame(filter_stats)

    cash_months_cnt = df_portfolio[df_portfolio["Code"] == "CASH"]["MonthKey_next"].nunique()

    cash_tbl = df_portfolio[df_portfolio["Code"] == "CASH"][["MonthKey_next"]].rename(columns={"MonthKey_next": "MonthKey"})
    cash_month_set = set(cash_tbl["MonthKey"].unique())

    stock_portfolio = df_portfolio[df_portfolio["Code"] != "CASH"].copy()
    monthly_target = {}
    for mk, g in stock_portfolio.groupby("MonthKey_next"):
        monthly_target[mk] = dict(zip(g["Code"].astype(str), g["Weight"].astype(float)))

    daily_port = daily.merge(
        stock_portfolio[["MonthKey_next", "Code", "Weight"]],
        left_on=["MonthKey", "Code"],
        right_on=["MonthKey_next", "Code"],
        how="inner"
    ).sort_values("Date").copy()

    # 日次バックテスト
    global_wealth = INITIAL_CAPITAL
    global_peak = INITIAL_CAPITAL
    current_hold_weights = {}
    current_state = "CASH"

    rebalance_exec = 0
    rebalance_skip = 0
    total_trading_cost = 0.0

    daily_returns = []

    for dt in all_dates:
        mk = pd.Timestamp(dt).to_period("M").strftime("%Y-%m")
        is_cash = mk in cash_month_set

        is_first = (mk in first_trade_day_of_month) and (pd.Timestamp(dt) == pd.Timestamp(first_trade_day_of_month[mk]))
        trading_cost = 0.0

        if is_first:
            target_w = monthly_target.get(mk, {})
            target_state = "CASH" if is_cash else "EQUITY"
            is_cash_transition = (current_state != target_state)

            prev_codes = list(current_hold_weights.keys())
            new_codes = list(target_w.keys())
            ov = overlap_ratio(prev_codes, new_codes) if (current_state == "EQUITY" and target_state == "EQUITY") else np.nan

            turnover, turnover_stocks, new_hold = calc_turnover_and_new_hold(current_hold_weights, target_w)

            if is_cash_transition:
                do_skip = False
            else:
                if target_state == "CASH":
                    do_skip = False
                else:
                    turnover_skip = (turnover_stocks < REBALANCE_MIN_TURNOVER_STOCKS) and (turnover < REBALANCE_MIN_TURNOVER_PCT)
                    overlap_skip = (USE_OVERLAP_SKIP and (ov >= OVERLAP_SKIP_TH))
                    do_skip = (turnover_skip or overlap_skip)

            if do_skip:
                rebalance_skip += 1
            else:
                rebalance_exec += 1
                if target_state == "CASH":
                    current_hold_weights = {}
                    current_state = "CASH"
                else:
                    trading_cost = turnover * TOTAL_COST_PER_TRADE
                    total_trading_cost += trading_cost
                    current_hold_weights = new_hold
                    current_state = "EQUITY"

        if is_cash:
            port_ret_gross = 0.0
            port_ret_net = 0.0
        else:
            df_day = daily_port[daily_port["Date"] == dt].copy()
            if len(df_day) == 0:
                continue
            df_day["weighted_ret"] = df_day["ret_d"].fillna(0) * df_day["Weight"]
            port_ret_gross = df_day["weighted_ret"].sum() * LEVERAGE_FIXED
            port_ret_net = port_ret_gross - trading_cost

        global_wealth *= (1 + port_ret_net)
        if global_wealth > global_peak:
            global_peak = global_wealth
        dd = global_wealth / global_peak - 1

        daily_returns.append({
            "Date": dt,
            "port_ret_net": port_ret_net,
            "port_ret_gross": port_ret_gross,
            "trading_cost": trading_cost,
            "dd": dd,
            "is_cash": is_cash
        })

    df_daily_returns = pd.DataFrame(daily_returns)
    out = summarize_results(df_daily_returns, total_trading_cost)

    out["TREND3_TH"] = trend3_th
    out["VOL3_Q"] = VOL3_Q_FIXED
    out["Vol3閾値"] = float(vol3_th)
    out["risk_off月数"] = int(risk_off_months)
    out["CASH月数"] = int(cash_months_cnt)
    out["リバランス実行回数"] = int(rebalance_exec)
    out["リバランス見送り回数"] = int(rebalance_skip)

    return out

# ============================================================
# 3パターン実行 → 日本語で比較表
# ============================================================
results = []
for th in TREND3_TH_LIST:
    print(f"\n実行中: TREND3_TH={th:.3f}（VOL3_Q=0.92固定） ...")
    results.append(run_backtest_once(th))

df_cmp = pd.DataFrame(results)

cols = [
    "TREND3_TH", "VOL3_Q", "Vol3閾値",
    "risk_off月数", "CASH月数", "平均CASH比率(%)",
    "年率(%)", "Sharpe", "最大DD(%)", "最大DD日",
    "取引コスト累積(%)",
    "リバランス実行回数", "リバランス見送り回数",
    "最終資産",
]
df_cmp = df_cmp[cols].copy()

# 表示用丸め
df_cmp["Vol3閾値"] = df_cmp["Vol3閾値"].round(6)
df_cmp["平均CASH比率(%)"] = df_cmp["平均CASH比率(%)"].round(2)
df_cmp["年率(%)"] = df_cmp["年率(%)"].round(2)
df_cmp["Sharpe"] = df_cmp["Sharpe"].round(4)
df_cmp["最大DD(%)"] = df_cmp["最大DD(%)"].round(2)
df_cmp["取引コスト累積(%)"] = df_cmp["取引コスト累積(%)"].round(2)
df_cmp["最終資産"] = df_cmp["最終資産"].round(0).astype(np.int64)

print("\n" + "="*95)
print("【TREND3_TH 3パターン比較（日本語）※VOL3_Q=0.92固定】")
print("="*95)
print(df_cmp.to_string(index=False))

out_path = ANALYSIS_DIR / "comparison_trend3th_ja.csv"
df_cmp.to_csv(out_path, index=False, encoding="utf-8-sig")
print("\n保存しました:", out_path)


スナップショット読込中...
EWMAスコア計算中...
日次データ読込中...

実行中: TREND3_TH=-0.020（VOL3_Q=0.92固定） ...

実行中: TREND3_TH=-0.025（VOL3_Q=0.92固定） ...

実行中: TREND3_TH=-0.030（VOL3_Q=0.92固定） ...

【TREND3_TH 3パターン比較（日本語）※VOL3_Q=0.92固定】
 TREND3_TH  VOL3_Q   Vol3閾値  risk_off月数  CASH月数  平均CASH比率(%)  年率(%)  Sharpe  最大DD(%)      最大DD日  取引コスト累積(%)  リバランス実行回数  リバランス見送り回数     最終資産
    -0.020    0.92 0.064933          31      31        25.80  21.02  1.3468   -21.93 2024-08-05       20.38        101          20 66617489
    -0.025    0.92 0.064933          30      30        25.02  21.14  1.3476   -21.93 2024-08-05       20.46        101          20 67276441
    -0.030    0.92 0.064933          29      29        24.12  19.58  1.2211   -21.93 2024-08-05       20.70        101          20 59113830

保存しました: C:\Users\yongr\Project\merged_data_all_stocks\analysis_daily\comparison_trend3th_ja.csv


In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 基本設定（あなたの環境と同じ）
# ============================================================
OUTPUT_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks")
SNAP_PATH = OUTPUT_DIR / "factors" / "month_end_snapshot.parquet"
MERGED_PARTS_DIR = OUTPUT_DIR / "merged_parts"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_daily"
ANALYSIS_DIR.mkdir(exist_ok=True)

# ============================================================
# 固定パラメータ（現状のv2.1/2.2系を踏襲）
# ============================================================
INITIAL_CAPITAL = 10_000_000
TOP_N = 20
VO_TOP_PCT = 0.70
MIN_PRICE = 800
ROE_MIN = -0.5
ROE_MAX = 1.0
BM_RATIO_MIN = 0.1
BM_RATIO_MAX = 10.0

OFF_TH = -0.115
VOL3_Q_FIXED = 0.92  # ★固定

COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.003
TOTAL_COST_PER_TRADE = COMMISSION_RATE + SLIPPAGE_RATE  # 片道0.4%

MAX_VOLUME_PARTICIPATION = 0.10
MIN_DAILY_TURNOVER = 5_000_000  # 基準に合わせる（結果にはほぼ影響しないことが確認済み）

PRICE_GAP_BUFFER = 0.95
LEVERAGE_FIXED = 1.0

WEIGHT_ADJUST_DEADBAND = 0.05
REBALANCE_MIN_TURNOVER_STOCKS = 5
REBALANCE_MIN_TURNOVER_PCT = 0.10

LIQUIDITY_BUFFER_FACTOR = 0.70
SCORE_EWMA_ALPHA = 0.50
OVERLAP_SKIP_TH = 0.80
USE_OVERLAP_SKIP = True

# ★微調整するTREND3_TH（5点）
TREND3_TH_LIST = [-0.023, -0.024, -0.025, -0.026, -0.027]

# ============================================================
# ユーティリティ
# ============================================================
def detect_stock_code_column(df):
    candidates = ["Code", "StockCode", "Symbol", "Ticker", "stock_code", "code"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"銘柄コード列が見つかりません。利用可能な列: {list(df.columns)}")

def normalize_columns(df, stock_code_col=None):
    rename_map = {}
    if stock_code_col and stock_code_col != "Code":
        rename_map[stock_code_col] = "Code"
    if "AdjustedClose" in df.columns and "AdjustmentClose" not in df.columns:
        rename_map["AdjustedClose"] = "AdjustmentClose"
    if "Volume" in df.columns and "Vo" not in df.columns:
        rename_map["Volume"] = "Vo"
    return df.rename(columns=rename_map)

def add_ewma_score_to_snapshot(snap: pd.DataFrame, alpha: float) -> pd.DataFrame:
    snap = snap.sort_values(["Code", "MonthEnd"]).copy()

    def _calc_month_z_and_score(g):
        g = g.copy()
        for col in ["BM_Ratio", "ROE", "INV_Growth"]:
            mean_val = g[col].mean()
            std_val = g[col].std()
            g[f"{col}_z_all"] = (g[col] - mean_val) / std_val if (std_val is not None and std_val > 0) else 0.0
        g["composite_score_raw_all"] = (
            g["BM_Ratio_z_all"] +
            g["ROE_z_all"] -
            g["INV_Growth_z_all"].fillna(0)
        )
        return g

    snap = snap.groupby("MonthEnd", group_keys=False).apply(_calc_month_z_and_score)
    snap["composite_score_ewma_all"] = (
        snap.groupby("Code")["composite_score_raw_all"]
            .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
    )
    return snap

def overlap_ratio(prev_codes, new_codes) -> float:
    prev_set = set(prev_codes)
    new_set = set(new_codes)
    if len(prev_set) == 0:
        return 0.0
    return len(prev_set & new_set) / max(len(prev_set), 1)

def calc_turnover_and_new_hold(prev_w, target_w, deadband=WEIGHT_ADJUST_DEADBAND):
    prev = prev_w.copy()
    target = target_w.copy()
    all_codes = set(prev.keys()) | set(target.keys())
    after = {}

    for c in all_codes:
        w0 = prev.get(c, 0.0)
        w1 = target.get(c, 0.0)
        if (c in prev) and (c in target):
            if abs(w1 - w0) < deadband:
                after[c] = w0
            else:
                after[c] = w1
        else:
            after[c] = w1

    after = {c: w for c, w in after.items() if w > 0}
    s = sum(after.values())
    if s > 0:
        after = {c: w/s for c, w in after.items()}

    codes2 = set(prev.keys()) | set(after.keys())
    turnover = sum(abs(after.get(c, 0.0) - prev.get(c, 0.0)) for c in codes2)
    turnover_stocks = len(set(prev.keys()) ^ set(after.keys()))
    return turnover, turnover_stocks, after

def summarize_results(df_daily_returns: pd.DataFrame, total_trading_cost_sum: float):
    df = df_daily_returns.copy()
    df["cum_ret"] = df["port_ret_net"].add(1).cumprod().sub(1)
    df["cum_wealth"] = INITIAL_CAPITAL * (1 + df["cum_ret"])

    first_date = df["Date"].min()
    last_date = df["Date"].max()
    years = (last_date - first_date).days / 365.25 if last_date > first_date else 1.0

    final_wealth = df["cum_wealth"].iloc[-1]
    cum_return = final_wealth / INITIAL_CAPITAL - 1
    ann_return = (1 + cum_return) ** (1 / years) - 1
    ann_vol = df["port_ret_net"].std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0

    dd = df["dd"].min()
    dd_date = df.loc[df["dd"].idxmin(), "Date"]
    cash_pct = df["is_cash"].mean() * 100

    return {
        "期間開始": first_date.date(),
        "期間終了": last_date.date(),
        "年率(%)": ann_return * 100,
        "Sharpe": sharpe,
        "最大DD(%)": dd * 100,
        "最大DD日": dd_date.date(),
        "取引コスト累積(%)": total_trading_cost_sum * 100,
        "平均CASH比率(%)": cash_pct,
        "最終資産": final_wealth
    }

# ============================================================
# データロード（1回だけ）
# ============================================================
print("スナップショット読込中...")
snap = pd.read_parquet(SNAP_PATH)
snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
snap = normalize_columns(snap, detect_stock_code_column(snap))
snap = snap.sort_values(["Code", "MonthEnd"]).copy()
snap["MonthKey"] = snap["MonthEnd"].dt.to_period("M").astype(str)

print("EWMAスコア計算中...")
snap = add_ewma_score_to_snapshot(snap, alpha=SCORE_EWMA_ALPHA)

print("日次データ読込中...")
daily_files = sorted(MERGED_PARTS_DIR.glob("merged-part-*.parquet"))
sample_df = pd.read_parquet(daily_files[0])
daily_stock_code_col = detect_stock_code_column(sample_df)

daily_list = []
for f in daily_files:
    df_part = pd.read_parquet(f)
    df_part = normalize_columns(df_part, daily_stock_code_col)
    daily_list.append(df_part)

daily = pd.concat(daily_list, ignore_index=True)
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values(["Code", "Date"]).copy()
if "AdjustedClose" in daily.columns and "AdjustmentClose" not in daily.columns:
    daily = daily.rename(columns={"AdjustedClose": "AdjustmentClose"})
daily["ret_d"] = daily.groupby("Code")["AdjustmentClose"].pct_change()
daily["MonthKey"] = daily["Date"].dt.to_period("M").astype(str)

first_trade_day_of_month = daily.groupby("MonthKey")["Date"].min().to_dict()
all_dates = sorted(daily["Date"].unique())

# ============================================================
# 1回分のバックテスト（TREND3_THのみ可変）
# ============================================================
def run_backtest_once(trend3_th: float) -> dict:
    snap2 = snap.copy()

    snap2["ret_m_fwd"] = snap2.groupby("Code")["AdjustmentClose"].pct_change().shift(-1)
    snap2["TurnoverValue"] = snap2["Vo"] * snap2["AdjustmentClose"]

    monthly = snap2.groupby("MonthEnd").apply(
        lambda g: pd.Series({
            "mkt_vw": np.average(g["ret_m_fwd"].fillna(0), weights=g["MarketCap"].fillna(1)),
            "count": len(g)
        })
    ).reset_index()

    monthly["cum_ret"] = (1 + monthly["mkt_vw"]).cumprod()
    monthly["peak"] = monthly["cum_ret"].cummax()
    monthly["mkt_dd"] = (monthly["cum_ret"] / monthly["peak"]) - 1
    monthly["trend3"] = monthly["mkt_vw"].rolling(3, min_periods=1).mean()
    monthly["vol3"] = monthly["mkt_vw"].rolling(3, min_periods=2).std()

    vol3_th = monthly["vol3"].quantile(VOL3_Q_FIXED)

    snap2 = snap2.merge(monthly[["MonthEnd", "mkt_dd", "trend3", "vol3"]], on="MonthEnd", how="left")
    snap2["risk_off"] = (
        (snap2["mkt_dd"] <= OFF_TH) |
        (snap2["trend3"] <= trend3_th) |
        (snap2["vol3"] >= vol3_th)
    )

    portfolio_list = []
    risk_off_months = 0

    for month_end, df_month in snap2.groupby("MonthEnd"):
        risk_off = bool(df_month["risk_off"].iloc[0])
        month_key_next = (pd.Timestamp(month_end) + pd.DateOffset(months=1)).to_period("M").strftime("%Y-%m")

        if risk_off:
            risk_off_months += 1
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0})
            continue

        df_valid = df_month[df_month["AdjustmentClose"] >= MIN_PRICE].copy()
        df_valid = df_valid[
            (df_valid["BM_Ratio"].notna()) & (df_valid["ROE"].notna()) &
            (df_valid["ROE"] >= ROE_MIN) & (df_valid["ROE"] <= ROE_MAX) &
            (df_valid["BM_Ratio"] >= BM_RATIO_MIN) & (df_valid["BM_Ratio"] <= BM_RATIO_MAX)
        ].copy()

        if len(df_valid) > 0:
            turnover_th = df_valid["TurnoverValue"].quantile(1 - VO_TOP_PCT)
            df_valid = df_valid[df_valid["TurnoverValue"] >= turnover_th]

        df_valid = df_valid[df_valid["TurnoverValue"] >= MIN_DAILY_TURNOVER]

        df_valid["MaxBuyableAmount"] = df_valid["Vo"] * MAX_VOLUME_PARTICIPATION * df_valid["AdjustmentClose"]
        target_capital = INITIAL_CAPITAL * LEVERAGE_FIXED * PRICE_GAP_BUFFER
        capital_per_stock = target_capital / TOP_N
        df_valid = df_valid[df_valid["MaxBuyableAmount"] >= capital_per_stock * LIQUIDITY_BUFFER_FACTOR]

        if len(df_valid) < TOP_N:
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0})
            continue

        top_n = df_valid.nlargest(TOP_N, "composite_score_ewma_all")
        w = 1.0 / len(top_n)
        for _, r in top_n.iterrows():
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": r["Code"], "Weight": w})

    df_portfolio = pd.DataFrame(portfolio_list)
    cash_months_cnt = df_portfolio[df_portfolio["Code"] == "CASH"]["MonthKey_next"].nunique()

    cash_tbl = df_portfolio[df_portfolio["Code"] == "CASH"][["MonthKey_next"]].rename(columns={"MonthKey_next": "MonthKey"})
    cash_month_set = set(cash_tbl["MonthKey"].unique())

    stock_portfolio = df_portfolio[df_portfolio["Code"] != "CASH"].copy()
    monthly_target = {}
    for mk, g in stock_portfolio.groupby("MonthKey_next"):
        monthly_target[mk] = dict(zip(g["Code"].astype(str), g["Weight"].astype(float)))

    daily_port = daily.merge(
        stock_portfolio[["MonthKey_next", "Code", "Weight"]],
        left_on=["MonthKey", "Code"],
        right_on=["MonthKey_next", "Code"],
        how="inner"
    ).sort_values("Date").copy()

    global_wealth = INITIAL_CAPITAL
    global_peak = INITIAL_CAPITAL
    current_hold_weights = {}
    current_state = "CASH"

    rebalance_exec = 0
    rebalance_skip = 0
    total_trading_cost = 0.0

    daily_returns = []

    for dt in all_dates:
        mk = pd.Timestamp(dt).to_period("M").strftime("%Y-%m")
        is_cash = mk in cash_month_set

        is_first = (mk in first_trade_day_of_month) and (pd.Timestamp(dt) == pd.Timestamp(first_trade_day_of_month[mk]))
        trading_cost = 0.0

        if is_first:
            target_w = monthly_target.get(mk, {})
            target_state = "CASH" if is_cash else "EQUITY"
            is_cash_transition = (current_state != target_state)

            prev_codes = list(current_hold_weights.keys())
            new_codes = list(target_w.keys())
            ov = overlap_ratio(prev_codes, new_codes) if (current_state == "EQUITY" and target_state == "EQUITY") else np.nan

            turnover, turnover_stocks, new_hold = calc_turnover_and_new_hold(current_hold_weights, target_w)

            if is_cash_transition:
                do_skip = False
            else:
                if target_state == "CASH":
                    do_skip = False
                else:
                    turnover_skip = (turnover_stocks < REBALANCE_MIN_TURNOVER_STOCKS) and (turnover < REBALANCE_MIN_TURNOVER_PCT)
                    overlap_skip = (USE_OVERLAP_SKIP and (ov >= OVERLAP_SKIP_TH))
                    do_skip = (turnover_skip or overlap_skip)

            if do_skip:
                rebalance_skip += 1
            else:
                rebalance_exec += 1
                if target_state == "CASH":
                    current_hold_weights = {}
                    current_state = "CASH"
                else:
                    trading_cost = turnover * TOTAL_COST_PER_TRADE
                    total_trading_cost += trading_cost
                    current_hold_weights = new_hold
                    current_state = "EQUITY"

        if is_cash:
            port_ret_net = 0.0
            port_ret_gross = 0.0
        else:
            df_day = daily_port[daily_port["Date"] == dt].copy()
            if len(df_day) == 0:
                continue
            df_day["weighted_ret"] = df_day["ret_d"].fillna(0) * df_day["Weight"]
            port_ret_gross = df_day["weighted_ret"].sum() * LEVERAGE_FIXED
            port_ret_net = port_ret_gross - trading_cost

        global_wealth *= (1 + port_ret_net)
        if global_wealth > global_peak:
            global_peak = global_wealth
        dd = global_wealth / global_peak - 1

        daily_returns.append({
            "Date": dt,
            "port_ret_net": port_ret_net,
            "port_ret_gross": port_ret_gross,
            "trading_cost": trading_cost,
            "dd": dd,
            "is_cash": is_cash
        })

    df_daily_returns = pd.DataFrame(daily_returns)
    out = summarize_results(df_daily_returns, total_trading_cost)

    out["TREND3_TH"] = trend3_th
    out["VOL3_Q"] = VOL3_Q_FIXED
    out["Vol3閾値"] = float(vol3_th)
    out["risk_off月数"] = int(risk_off_months)
    out["CASH月数"] = int(cash_months_cnt)
    out["リバランス実行回数"] = int(rebalance_exec)
    out["リバランス見送り回数"] = int(rebalance_skip)
    return out

# ============================================================
# 5パターン実行 → 日本語比較表
# ============================================================
results = []
for th in TREND3_TH_LIST:
    print(f"\n実行中: TREND3_TH={th:.3f}（VOL3_Q=0.92固定） ...")
    results.append(run_backtest_once(th))

df_cmp = pd.DataFrame(results)

cols = [
    "TREND3_TH", "VOL3_Q", "Vol3閾値",
    "risk_off月数", "CASH月数", "平均CASH比率(%)",
    "年率(%)", "Sharpe", "最大DD(%)", "最大DD日",
    "取引コスト累積(%)",
    "リバランス実行回数", "リバランス見送り回数",
    "最終資産",
]
df_cmp = df_cmp[cols].copy()

df_cmp["Vol3閾値"] = df_cmp["Vol3閾値"].round(6)
df_cmp["平均CASH比率(%)"] = df_cmp["平均CASH比率(%)"].round(2)
df_cmp["年率(%)"] = df_cmp["年率(%)"].round(2)
df_cmp["Sharpe"] = df_cmp["Sharpe"].round(4)
df_cmp["最大DD(%)"] = df_cmp["最大DD(%)"].round(2)
df_cmp["取引コスト累積(%)"] = df_cmp["取引コスト累積(%)"].round(2)
df_cmp["最終資産"] = df_cmp["最終資産"].round(0).astype(np.int64)

print("\n" + "="*110)
print("【TREND3_TH 微調整 5パターン比較（日本語）※VOL3_Q=0.92固定】")
print("="*110)
print(df_cmp.to_string(index=False))

out_path = ANALYSIS_DIR / "comparison_trend3th_fine_ja.csv"
df_cmp.to_csv(out_path, index=False, encoding="utf-8-sig")
print("\n保存しました:", out_path)


スナップショット読込中...
EWMAスコア計算中...
日次データ読込中...

実行中: TREND3_TH=-0.023（VOL3_Q=0.92固定） ...

実行中: TREND3_TH=-0.024（VOL3_Q=0.92固定） ...

実行中: TREND3_TH=-0.025（VOL3_Q=0.92固定） ...

実行中: TREND3_TH=-0.026（VOL3_Q=0.92固定） ...

実行中: TREND3_TH=-0.027（VOL3_Q=0.92固定） ...

【TREND3_TH 微調整 5パターン比較（日本語）※VOL3_Q=0.92固定】
 TREND3_TH  VOL3_Q   Vol3閾値  risk_off月数  CASH月数  平均CASH比率(%)  年率(%)  Sharpe  最大DD(%)      最大DD日  取引コスト累積(%)  リバランス実行回数  リバランス見送り回数     最終資産
    -0.023    0.92 0.064933          31      31        25.80  21.02  1.3468   -21.93 2024-08-05       20.38        101          20 66617489
    -0.024    0.92 0.064933          30      30        25.02  21.14  1.3476   -21.93 2024-08-05       20.46        101          20 67276441
    -0.025    0.92 0.064933          30      30        25.02  21.14  1.3476   -21.93 2024-08-05       20.46        101          20 67276441
    -0.026    0.92 0.064933          30      30        25.02  21.14  1.3476   -21.93 2024-08-05       20.46        101          20 67276441
    -

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 固定パラメータ（指定）
# ============================================================
VOL3_Q_FIXED = 0.92
TREND3_TH_FIXED = -0.025

# ============================================================
# 基本設定（あなたの環境）
# ============================================================
OUTPUT_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks")
SNAP_PATH = OUTPUT_DIR / "factors" / "month_end_snapshot.parquet"
MERGED_PARTS_DIR = OUTPUT_DIR / "merged_parts"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_daily"
ANALYSIS_DIR.mkdir(exist_ok=True)

# ============================================================
# その他パラメータ（現行v2.1系を踏襲：固定）
# ============================================================
INITIAL_CAPITAL = 10_000_000
TOP_N = 20
VO_TOP_PCT = 0.70
MIN_PRICE = 800

ROE_MIN = -0.5
ROE_MAX = 1.0
BM_RATIO_MIN = 0.1
BM_RATIO_MAX = 10.0

OFF_TH = -0.115

COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.003
TOTAL_COST_PER_TRADE = COMMISSION_RATE + SLIPPAGE_RATE  # 片道0.4%

MAX_VOLUME_PARTICIPATION = 0.10
MIN_DAILY_TURNOVER = 5_000_000  # ここは大勢に影響薄い（確認済み）だが基準に合わせる

PRICE_GAP_BUFFER = 0.95
LEVERAGE_FIXED = 1.0

WEIGHT_ADJUST_DEADBAND = 0.05
REBALANCE_MIN_TURNOVER_STOCKS = 5
REBALANCE_MIN_TURNOVER_PCT = 0.10

LIQUIDITY_BUFFER_FACTOR = 0.70
SCORE_EWMA_ALPHA = 0.50
OVERLAP_SKIP_TH = 0.80
USE_OVERLAP_SKIP = True


# ============================================================
# 期間分割（指定）
# ============================================================
IS_START = pd.Timestamp("2016-02-01")
IS_END   = pd.Timestamp("2021-12-31")

OOS_START = pd.Timestamp("2022-01-01")
# OOS_END は日次データの最終日に合わせて自動で決める


# ============================================================
# Utilities
# ============================================================
def detect_stock_code_column(df):
    candidates = ["Code", "StockCode", "Symbol", "Ticker", "stock_code", "code"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"銘柄コード列が見つかりません。利用可能な列: {list(df.columns)}")

def normalize_columns(df, stock_code_col=None):
    rename_map = {}
    if stock_code_col and stock_code_col != "Code":
        rename_map[stock_code_col] = "Code"
    if "AdjustedClose" in df.columns and "AdjustmentClose" not in df.columns:
        rename_map["AdjustedClose"] = "AdjustmentClose"
    if "Volume" in df.columns and "Vo" not in df.columns:
        rename_map["Volume"] = "Vo"
    return df.rename(columns=rename_map)

def add_ewma_score_to_snapshot(snap: pd.DataFrame, alpha: float) -> pd.DataFrame:
    snap = snap.sort_values(["Code", "MonthEnd"]).copy()

    def _calc_month_z_and_score(g):
        g = g.copy()
        for col in ["BM_Ratio", "ROE", "INV_Growth"]:
            mean_val = g[col].mean()
            std_val = g[col].std()
            g[f"{col}_z_all"] = (g[col] - mean_val) / std_val if (std_val is not None and std_val > 0) else 0.0
        g["composite_score_raw_all"] = (
            g["BM_Ratio_z_all"] +
            g["ROE_z_all"] -
            g["INV_Growth_z_all"].fillna(0)
        )
        return g

    snap = snap.groupby("MonthEnd", group_keys=False).apply(_calc_month_z_and_score)
    snap["composite_score_ewma_all"] = (
        snap.groupby("Code")["composite_score_raw_all"]
            .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
    )
    return snap

def overlap_ratio(prev_codes, new_codes) -> float:
    prev_set = set(prev_codes)
    new_set = set(new_codes)
    if len(prev_set) == 0:
        return 0.0
    return len(prev_set & new_set) / max(len(prev_set), 1)

def calc_turnover_and_new_hold(prev_w, target_w, deadband=WEIGHT_ADJUST_DEADBAND):
    prev = prev_w.copy()
    target = target_w.copy()
    all_codes = set(prev.keys()) | set(target.keys())
    after = {}

    for c in all_codes:
        w0 = prev.get(c, 0.0)
        w1 = target.get(c, 0.0)
        if (c in prev) and (c in target):
            if abs(w1 - w0) < deadband:
                after[c] = w0
            else:
                after[c] = w1
        else:
            after[c] = w1

    after = {c: w for c, w in after.items() if w > 0}
    s = sum(after.values())
    if s > 0:
        after = {c: w/s for c, w in after.items()}

    codes2 = set(prev.keys()) | set(after.keys())
    turnover = sum(abs(after.get(c, 0.0) - prev.get(c, 0.0)) for c in codes2)
    turnover_stocks = len(set(prev.keys()) ^ set(after.keys()))
    return turnover, turnover_stocks, after

def summarize_daily(df_daily_returns: pd.DataFrame, total_trading_cost_sum: float) -> dict:
    df = df_daily_returns.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    df["cum_ret"] = df["port_ret_net"].add(1).cumprod().sub(1)
    df["cum_wealth"] = INITIAL_CAPITAL * (1 + df["cum_ret"])

    first_date = df["Date"].min()
    last_date = df["Date"].max()
    years = (last_date - first_date).days / 365.25 if last_date > first_date else 1.0

    final_wealth = df["cum_wealth"].iloc[-1]
    cum_return = final_wealth / INITIAL_CAPITAL - 1
    ann_return = (1 + cum_return) ** (1 / years) - 1
    ann_vol = df["port_ret_net"].std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0

    # drawdown
    wealth = df["cum_wealth"]
    peak = wealth.cummax()
    dd = wealth / peak - 1
    maxdd = float(dd.min())
    maxdd_date = df.loc[dd.idxmin(), "Date"].date()

    avg_cash_pct = df["is_cash"].mean() * 100

    return {
        "期間開始": first_date.date(),
        "期間終了": last_date.date(),
        "年率(%)": ann_return * 100,
        "Sharpe": sharpe,
        "最大DD(%)": maxdd * 100,
        "最大DD日": maxdd_date,
        "取引コスト累積(%)": total_trading_cost_sum * 100,
        "平均CASH比率(%)": avg_cash_pct,
        "最終資産": float(final_wealth),
        "日数": int(len(df))
    }


# ============================================================
# Load snapshot + daily once
# ============================================================
print("スナップショット読込中...")
snap = pd.read_parquet(SNAP_PATH)
snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
snap = normalize_columns(snap, detect_stock_code_column(snap))
snap = snap.sort_values(["Code", "MonthEnd"]).copy()
snap["MonthKey"] = snap["MonthEnd"].dt.to_period("M").astype(str)

print("EWMAスコア計算中...")
snap = add_ewma_score_to_snapshot(snap, alpha=SCORE_EWMA_ALPHA)

print("日次データ読込中...")
daily_files = sorted(MERGED_PARTS_DIR.glob("merged-part-*.parquet"))
sample_df = pd.read_parquet(daily_files[0])
daily_stock_code_col = detect_stock_code_column(sample_df)

daily_list = []
for f in daily_files:
    df_part = pd.read_parquet(f)
    df_part = normalize_columns(df_part, daily_stock_code_col)
    daily_list.append(df_part)

daily = pd.concat(daily_list, ignore_index=True)
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values(["Code", "Date"]).copy()
if "AdjustedClose" in daily.columns and "AdjustmentClose" not in daily.columns:
    daily = daily.rename(columns={"AdjustedClose": "AdjustmentClose"})
daily["ret_d"] = daily.groupby("Code")["AdjustmentClose"].pct_change()
daily["MonthKey"] = daily["Date"].dt.to_period("M").astype(str)

OOS_END = daily["Date"].max()
print("日次データ期間:", daily["Date"].min().date(), "〜", OOS_END.date())

first_trade_day_of_month = daily.groupby("MonthKey")["Date"].min().to_dict()
all_dates = sorted(daily["Date"].unique())


# ============================================================
# Build monthly targets (common for whole period, then we slice by months for IS/OOS)
# ============================================================
def build_monthly_targets(vol3_q: float, trend3_th: float):
    snap2 = snap.copy()

    snap2["ret_m_fwd"] = snap2.groupby("Code")["AdjustmentClose"].pct_change().shift(-1)
    snap2["TurnoverValue"] = snap2["Vo"] * snap2["AdjustmentClose"]

    monthly = snap2.groupby("MonthEnd").apply(
        lambda g: pd.Series({
            "mkt_vw": np.average(g["ret_m_fwd"].fillna(0), weights=g["MarketCap"].fillna(1)),
            "count": len(g)
        })
    ).reset_index()

    monthly["cum_ret"] = (1 + monthly["mkt_vw"]).cumprod()
    monthly["peak"] = monthly["cum_ret"].cummax()
    monthly["mkt_dd"] = (monthly["cum_ret"] / monthly["peak"]) - 1
    monthly["trend3"] = monthly["mkt_vw"].rolling(3, min_periods=1).mean()
    monthly["vol3"] = monthly["mkt_vw"].rolling(3, min_periods=2).std()

    vol3_th = monthly["vol3"].quantile(vol3_q)

    snap2 = snap2.merge(monthly[["MonthEnd", "mkt_dd", "trend3", "vol3"]], on="MonthEnd", how="left")
    snap2["risk_off"] = (
        (snap2["mkt_dd"] <= OFF_TH) |
        (snap2["trend3"] <= trend3_th) |
        (snap2["vol3"] >= vol3_th)
    )

    portfolio_list = []
    risk_off_months = 0

    for month_end, df_month in snap2.groupby("MonthEnd"):
        risk_off = bool(df_month["risk_off"].iloc[0])
        month_key_next = (pd.Timestamp(month_end) + pd.DateOffset(months=1)).to_period("M").strftime("%Y-%m")

        if risk_off:
            risk_off_months += 1
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0, "risk_off": True, "MonthEnd": month_end})
            continue

        df_valid = df_month[df_month["AdjustmentClose"] >= MIN_PRICE].copy()
        df_valid = df_valid[
            (df_valid["BM_Ratio"].notna()) & (df_valid["ROE"].notna()) &
            (df_valid["ROE"] >= ROE_MIN) & (df_valid["ROE"] <= ROE_MAX) &
            (df_valid["BM_Ratio"] >= BM_RATIO_MIN) & (df_valid["BM_Ratio"] <= BM_RATIO_MAX)
        ].copy()

        if len(df_valid) > 0:
            turnover_th = df_valid["TurnoverValue"].quantile(1 - VO_TOP_PCT)
            df_valid = df_valid[df_valid["TurnoverValue"] >= turnover_th]

        df_valid = df_valid[df_valid["TurnoverValue"] >= MIN_DAILY_TURNOVER]

        df_valid["MaxBuyableAmount"] = df_valid["Vo"] * MAX_VOLUME_PARTICIPATION * df_valid["AdjustmentClose"]
        target_capital = INITIAL_CAPITAL * LEVERAGE_FIXED * PRICE_GAP_BUFFER
        capital_per_stock = target_capital / TOP_N
        df_valid = df_valid[df_valid["MaxBuyableAmount"] >= capital_per_stock * LIQUIDITY_BUFFER_FACTOR]

        if len(df_valid) < TOP_N:
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0, "risk_off": False, "MonthEnd": month_end})
            continue

        top_n = df_valid.nlargest(TOP_N, "composite_score_ewma_all")
        w = 1.0 / len(top_n)
        for _, r in top_n.iterrows():
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": r["Code"], "Weight": w, "risk_off": False, "MonthEnd": month_end})

    df_portfolio = pd.DataFrame(portfolio_list)
    return df_portfolio, float(vol3_th)


df_portfolio_all, vol3_th_used = build_monthly_targets(VOL3_Q_FIXED, TREND3_TH_FIXED)

# monthly CASH months count helper per period
def count_cash_months_in_range(df_portfolio, start_date, end_date):
    # MonthKey_next are months where portfolio applies. We count those within [start,end] based on first trading day.
    # Convert MonthKey_next to month start date for filtering.
    mk = pd.to_datetime(df_portfolio["MonthKey_next"] + "-01")
    in_range = (mk >= pd.Timestamp(start_date).to_period("M").to_timestamp()) & (mk <= pd.Timestamp(end_date).to_period("M").to_timestamp())
    dfp = df_portfolio.loc[in_range].copy()
    return int(dfp[dfp["Code"] == "CASH"]["MonthKey_next"].nunique())

# Build sets and mapping for daily backtest (we will slice daily by period)
cash_tbl_all = df_portfolio_all[df_portfolio_all["Code"] == "CASH"][["MonthKey_next"]].rename(columns={"MonthKey_next": "MonthKey"})
cash_month_set_all = set(cash_tbl_all["MonthKey"].unique())

stock_portfolio_all = df_portfolio_all[df_portfolio_all["Code"] != "CASH"].copy()
monthly_target_all = {}
for mk, g in stock_portfolio_all.groupby("MonthKey_next"):
    monthly_target_all[mk] = dict(zip(g["Code"].astype(str), g["Weight"].astype(float)))

daily_port_all = daily.merge(
    stock_portfolio_all[["MonthKey_next", "Code", "Weight"]],
    left_on=["MonthKey", "Code"],
    right_on=["MonthKey_next", "Code"],
    how="inner"
).sort_values("Date").copy()


# ============================================================
# Daily backtest for a given period
# ============================================================
def run_daily_period(start_dt: pd.Timestamp, end_dt: pd.Timestamp) -> tuple[pd.DataFrame, float, int, int]:
    period_dates = [d for d in all_dates if (d >= start_dt) and (d <= end_dt)]
    if not period_dates:
        raise ValueError("指定期間に日次データがありません")

    # IMPORTANT: keep state within the period (reset) to avoid leakage across IS/OOS
    global_wealth = INITIAL_CAPITAL
    global_peak = INITIAL_CAPITAL
    current_hold_weights = {}
    current_state = "CASH"

    rebalance_exec = 0
    rebalance_skip = 0
    total_trading_cost = 0.0

    daily_rows = []

    for dt in period_dates:
        mk = pd.Timestamp(dt).to_period("M").strftime("%Y-%m")
        is_cash = mk in cash_month_set_all

        is_first = (mk in first_trade_day_of_month) and (pd.Timestamp(dt) == pd.Timestamp(first_trade_day_of_month[mk]))
        trading_cost = 0.0

        if is_first:
            target_w = monthly_target_all.get(mk, {})
            target_state = "CASH" if is_cash else "EQUITY"
            is_cash_transition = (current_state != target_state)

            prev_codes = list(current_hold_weights.keys())
            new_codes = list(target_w.keys())
            ov = overlap_ratio(prev_codes, new_codes) if (current_state == "EQUITY" and target_state == "EQUITY") else np.nan

            turnover, turnover_stocks, new_hold = calc_turnover_and_new_hold(current_hold_weights, target_w)

            if is_cash_transition:
                do_skip = False
            else:
                if target_state == "CASH":
                    do_skip = False
                else:
                    turnover_skip = (turnover_stocks < REBALANCE_MIN_TURNOVER_STOCKS) and (turnover < REBALANCE_MIN_TURNOVER_PCT)
                    overlap_skip = (USE_OVERLAP_SKIP and (ov >= OVERLAP_SKIP_TH))
                    do_skip = (turnover_skip or overlap_skip)

            if do_skip:
                rebalance_skip += 1
            else:
                rebalance_exec += 1
                if target_state == "CASH":
                    current_hold_weights = {}
                    current_state = "CASH"
                else:
                    trading_cost = turnover * TOTAL_COST_PER_TRADE
                    total_trading_cost += trading_cost
                    current_hold_weights = new_hold
                    current_state = "EQUITY"

        if is_cash:
            port_ret_gross = 0.0
            port_ret_net = 0.0
        else:
            df_day = daily_port_all[daily_port_all["Date"] == dt].copy()
            if len(df_day) == 0:
                continue
            df_day["weighted_ret"] = df_day["ret_d"].fillna(0) * df_day["Weight"]
            port_ret_gross = df_day["weighted_ret"].sum() * LEVERAGE_FIXED
            port_ret_net = port_ret_gross - trading_cost

        global_wealth *= (1 + port_ret_net)
        if global_wealth > global_peak:
            global_peak = global_wealth
        dd = global_wealth / global_peak - 1

        daily_rows.append({
            "Date": dt,
            "port_ret_net": port_ret_net,
            "port_ret_gross": port_ret_gross,
            "trading_cost": trading_cost,
            "dd": dd,
            "is_cash": is_cash
        })

    df_daily = pd.DataFrame(daily_rows)
    return df_daily, total_trading_cost, rebalance_exec, rebalance_skip


# ============================================================
# Run IS / OOS
# ============================================================
print("\n実行パラメータ固定:")
print(f"  VOL3_Q={VOL3_Q_FIXED}, TREND3_TH={TREND3_TH_FIXED}, Vol3閾値={vol3_th_used:.6f}")
print("期間分割:")
print(f"  IS : {IS_START.date()} 〜 {IS_END.date()}")
print(f"  OOS: {OOS_START.date()} 〜 {OOS_END.date()}")

# IS
df_is, cost_is, re_is, rs_is = run_daily_period(IS_START, IS_END)
is_cash_months = count_cash_months_in_range(df_portfolio_all, IS_START, IS_END)
is_summ = summarize_daily(df_is, cost_is)
is_summ["区分"] = "IS (2016-2021)"
is_summ["CASH月数"] = is_cash_months
is_summ["リバランス実行回数"] = re_is
is_summ["リバランス見送り回数"] = rs_is

# OOS
df_oos, cost_oos, re_oos, rs_oos = run_daily_period(OOS_START, OOS_END)
oos_cash_months = count_cash_months_in_range(df_portfolio_all, OOS_START, OOS_END)
oos_summ = summarize_daily(df_oos, cost_oos)
oos_summ["区分"] = "OOS (2022-2026)"
oos_summ["CASH月数"] = oos_cash_months
oos_summ["リバランス実行回数"] = re_oos
oos_summ["リバランス見送り回数"] = rs_oos

# ============================================================
# 日本語比較表
# ============================================================
df_cmp = pd.DataFrame([is_summ, oos_summ])

# 表示列
cols = [
    "区分",
    "期間開始", "期間終了", "日数",
    "年率(%)", "Sharpe", "最大DD(%)", "最大DD日",
    "CASH月数", "平均CASH比率(%)",
    "取引コスト累積(%)",
    "リバランス実行回数", "リバランス見送り回数",
    "最終資産",
]
df_cmp = df_cmp[cols].copy()

# 表示用丸め
df_cmp["年率(%)"] = df_cmp["年率(%)"].round(2)
df_cmp["Sharpe"] = df_cmp["Sharpe"].round(4)
df_cmp["最大DD(%)"] = df_cmp["最大DD(%)"].round(2)
df_cmp["平均CASH比率(%)"] = df_cmp["平均CASH比率(%)"].round(2)
df_cmp["取引コスト累積(%)"] = df_cmp["取引コスト累積(%)"].round(2)
df_cmp["最終資産"] = df_cmp["最終資産"].round(0).astype(np.int64)

print("\n" + "="*110)
print("【期間分割バックテスト比較（日本語）】VOL3_Q=0.92, TREND3_TH=-0.025 固定")
print("="*110)
print(df_cmp.to_string(index=False))

out_path = ANALYSIS_DIR / "comparison_is_oos_ja.csv"
df_cmp.to_csv(out_path, index=False, encoding="utf-8-sig")
print("\n保存しました:", out_path)


スナップショット読込中...
EWMAスコア計算中...
日次データ読込中...
日次データ期間: 2016-01-15 〜 2026-01-09

実行パラメータ固定:
  VOL3_Q=0.92, TREND3_TH=-0.025, Vol3閾値=0.064933
期間分割:
  IS : 2016-02-01 〜 2021-12-31
  OOS: 2022-01-01 〜 2026-01-09

【期間分割バックテスト比較（日本語）】VOL3_Q=0.92, TREND3_TH=-0.025 固定
             区分       期間開始       期間終了   日数  年率(%)  Sharpe  最大DD(%)      最大DD日  CASH月数  平均CASH比率(%)  取引コスト累積(%)  リバランス実行回数  リバランス見送り回数     最終資産
 IS (2016-2021) 2016-02-01 2021-12-30 1447  13.59  1.1327   -16.79 2021-11-30      29        40.64        8.44         57          14 21239484
OOS (2022-2026) 2022-01-04 2026-01-09  983  33.22  1.6695   -21.93 2024-08-05       1         2.03       12.18         43           6 31625166

保存しました: C:\Users\yongr\Project\merged_data_all_stocks\analysis_daily\comparison_is_oos_ja.csv


In [15]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 固定パラメータ（指定）
# ============================================================
VOL3_Q_FIXED = 0.92
TREND3_TH_FIXED = -0.025

# ============================================================
# 基本設定（あなたの環境）
# ============================================================
OUTPUT_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks")
SNAP_PATH = OUTPUT_DIR / "factors" / "month_end_snapshot.parquet"
MERGED_PARTS_DIR = OUTPUT_DIR / "merged_parts"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_daily"
ANALYSIS_DIR.mkdir(exist_ok=True)

# ============================================================
# その他パラメータ（現行v2.1系を踏襲：固定）
# ============================================================
INITIAL_CAPITAL = 10_000_000
TOP_N = 20
VO_TOP_PCT = 0.70
MIN_PRICE = 800

ROE_MIN = -0.5
ROE_MAX = 1.0
BM_RATIO_MIN = 0.1
BM_RATIO_MAX = 10.0

OFF_TH = -0.115

COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.003
TOTAL_COST_PER_TRADE = COMMISSION_RATE + SLIPPAGE_RATE  # 片道0.4%

MAX_VOLUME_PARTICIPATION = 0.10
MIN_DAILY_TURNOVER = 5_000_000  # ここは大勢に影響薄い（確認済み）だが基準に合わせる

PRICE_GAP_BUFFER = 0.95
LEVERAGE_FIXED = 1.0

WEIGHT_ADJUST_DEADBAND = 0.05
REBALANCE_MIN_TURNOVER_STOCKS = 5
REBALANCE_MIN_TURNOVER_PCT = 0.10

LIQUIDITY_BUFFER_FACTOR = 0.70
SCORE_EWMA_ALPHA = 0.50
OVERLAP_SKIP_TH = 0.80
USE_OVERLAP_SKIP = True


# ============================================================
# 期間分割（指定）
# ============================================================
IS_START = pd.Timestamp("2016-02-01")
IS_END   = pd.Timestamp("2021-12-31")

OOS_START = pd.Timestamp("2022-01-01")
# OOS_END は日次データの最終日に合わせて自動で決める


# ============================================================
# Utilities
# ============================================================
def detect_stock_code_column(df):
    candidates = ["Code", "StockCode", "Symbol", "Ticker", "stock_code", "code"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"銘柄コード列が見つかりません。利用可能な列: {list(df.columns)}")

def normalize_columns(df, stock_code_col=None):
    rename_map = {}
    if stock_code_col and stock_code_col != "Code":
        rename_map[stock_code_col] = "Code"
    if "AdjustedClose" in df.columns and "AdjustmentClose" not in df.columns:
        rename_map["AdjustedClose"] = "AdjustmentClose"
    if "Volume" in df.columns and "Vo" not in df.columns:
        rename_map["Volume"] = "Vo"
    return df.rename(columns=rename_map)

def add_ewma_score_to_snapshot(snap: pd.DataFrame, alpha: float) -> pd.DataFrame:
    snap = snap.sort_values(["Code", "MonthEnd"]).copy()

    def _calc_month_z_and_score(g):
        g = g.copy()
        for col in ["BM_Ratio", "ROE", "INV_Growth"]:
            mean_val = g[col].mean()
            std_val = g[col].std()
            g[f"{col}_z_all"] = (g[col] - mean_val) / std_val if (std_val is not None and std_val > 0) else 0.0
        g["composite_score_raw_all"] = (
            g["BM_Ratio_z_all"] +
            g["ROE_z_all"] -
            g["INV_Growth_z_all"].fillna(0)
        )
        return g

    snap = snap.groupby("MonthEnd", group_keys=False).apply(_calc_month_z_and_score)
    snap["composite_score_ewma_all"] = (
        snap.groupby("Code")["composite_score_raw_all"]
            .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
    )
    return snap

def overlap_ratio(prev_codes, new_codes) -> float:
    prev_set = set(prev_codes)
    new_set = set(new_codes)
    if len(prev_set) == 0:
        return 0.0
    return len(prev_set & new_set) / max(len(prev_set), 1)

def calc_turnover_and_new_hold(prev_w, target_w, deadband=WEIGHT_ADJUST_DEADBAND):
    prev = prev_w.copy()
    target = target_w.copy()
    all_codes = set(prev.keys()) | set(target.keys())
    after = {}

    for c in all_codes:
        w0 = prev.get(c, 0.0)
        w1 = target.get(c, 0.0)
        if (c in prev) and (c in target):
            if abs(w1 - w0) < deadband:
                after[c] = w0
            else:
                after[c] = w1
        else:
            after[c] = w1

    after = {c: w for c, w in after.items() if w > 0}
    s = sum(after.values())
    if s > 0:
        after = {c: w/s for c, w in after.items()}

    codes2 = set(prev.keys()) | set(after.keys())
    turnover = sum(abs(after.get(c, 0.0) - prev.get(c, 0.0)) for c in codes2)
    turnover_stocks = len(set(prev.keys()) ^ set(after.keys()))
    return turnover, turnover_stocks, after

def summarize_daily(df_daily_returns: pd.DataFrame, total_trading_cost_sum: float) -> dict:
    df = df_daily_returns.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    df["cum_ret"] = df["port_ret_net"].add(1).cumprod().sub(1)
    df["cum_wealth"] = INITIAL_CAPITAL * (1 + df["cum_ret"])

    first_date = df["Date"].min()
    last_date = df["Date"].max()
    years = (last_date - first_date).days / 365.25 if last_date > first_date else 1.0

    final_wealth = df["cum_wealth"].iloc[-1]
    cum_return = final_wealth / INITIAL_CAPITAL - 1
    ann_return = (1 + cum_return) ** (1 / years) - 1
    ann_vol = df["port_ret_net"].std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0

    # drawdown
    wealth = df["cum_wealth"]
    peak = wealth.cummax()
    dd = wealth / peak - 1
    maxdd = float(dd.min())
    maxdd_date = df.loc[dd.idxmin(), "Date"].date()

    avg_cash_pct = df["is_cash"].mean() * 100

    return {
        "期間開始": first_date.date(),
        "期間終了": last_date.date(),
        "年率(%)": ann_return * 100,
        "Sharpe": sharpe,
        "最大DD(%)": maxdd * 100,
        "最大DD日": maxdd_date,
        "取引コスト累積(%)": total_trading_cost_sum * 100,
        "平均CASH比率(%)": avg_cash_pct,
        "最終資産": float(final_wealth),
        "日数": int(len(df))
    }


# ============================================================
# Load snapshot + daily once
# ============================================================
print("スナップショット読込中...")
snap = pd.read_parquet(SNAP_PATH)
snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
snap = normalize_columns(snap, detect_stock_code_column(snap))
snap = snap.sort_values(["Code", "MonthEnd"]).copy()
snap["MonthKey"] = snap["MonthEnd"].dt.to_period("M").astype(str)

print("EWMAスコア計算中...")
snap = add_ewma_score_to_snapshot(snap, alpha=SCORE_EWMA_ALPHA)

print("日次データ読込中...")
daily_files = sorted(MERGED_PARTS_DIR.glob("merged-part-*.parquet"))
sample_df = pd.read_parquet(daily_files[0])
daily_stock_code_col = detect_stock_code_column(sample_df)

daily_list = []
for f in daily_files:
    df_part = pd.read_parquet(f)
    df_part = normalize_columns(df_part, daily_stock_code_col)
    daily_list.append(df_part)

daily = pd.concat(daily_list, ignore_index=True)
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values(["Code", "Date"]).copy()
if "AdjustedClose" in daily.columns and "AdjustmentClose" not in daily.columns:
    daily = daily.rename(columns={"AdjustedClose": "AdjustmentClose"})
daily["ret_d"] = daily.groupby("Code")["AdjustmentClose"].pct_change()
daily["MonthKey"] = daily["Date"].dt.to_period("M").astype(str)

OOS_END = daily["Date"].max()
print("日次データ期間:", daily["Date"].min().date(), "〜", OOS_END.date())

first_trade_day_of_month = daily.groupby("MonthKey")["Date"].min().to_dict()
all_dates = sorted(daily["Date"].unique())


# ============================================================
# Build monthly targets (common for whole period, then we slice by months for IS/OOS)
# ============================================================
def build_monthly_targets(vol3_q: float, trend3_th: float):
    snap2 = snap.copy()

    snap2["ret_m_fwd"] = snap2.groupby("Code")["AdjustmentClose"].pct_change().shift(-1)
    snap2["TurnoverValue"] = snap2["Vo"] * snap2["AdjustmentClose"]

    monthly = snap2.groupby("MonthEnd").apply(
        lambda g: pd.Series({
            "mkt_vw": np.average(g["ret_m_fwd"].fillna(0), weights=g["MarketCap"].fillna(1)),
            "count": len(g)
        })
    ).reset_index()

    monthly["cum_ret"] = (1 + monthly["mkt_vw"]).cumprod()
    monthly["peak"] = monthly["cum_ret"].cummax()
    monthly["mkt_dd"] = (monthly["cum_ret"] / monthly["peak"]) - 1
    monthly["trend3"] = monthly["mkt_vw"].rolling(3, min_periods=1).mean()
    monthly["vol3"] = monthly["mkt_vw"].rolling(3, min_periods=2).std()

    vol3_th = monthly["vol3"].quantile(vol3_q)

    snap2 = snap2.merge(monthly[["MonthEnd", "mkt_dd", "trend3", "vol3"]], on="MonthEnd", how="left")
    snap2["risk_off"] = (
        (snap2["mkt_dd"] <= OFF_TH) |
        (snap2["trend3"] <= trend3_th) |
        (snap2["vol3"] >= vol3_th)
    )

    portfolio_list = []
    risk_off_months = 0

    for month_end, df_month in snap2.groupby("MonthEnd"):
        risk_off = bool(df_month["risk_off"].iloc[0])
        month_key_next = (pd.Timestamp(month_end) + pd.DateOffset(months=1)).to_period("M").strftime("%Y-%m")

        if risk_off:
            risk_off_months += 1
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0, "risk_off": True, "MonthEnd": month_end})
            continue

        df_valid = df_month[df_month["AdjustmentClose"] >= MIN_PRICE].copy()
        df_valid = df_valid[
            (df_valid["BM_Ratio"].notna()) & (df_valid["ROE"].notna()) &
            (df_valid["ROE"] >= ROE_MIN) & (df_valid["ROE"] <= ROE_MAX) &
            (df_valid["BM_Ratio"] >= BM_RATIO_MIN) & (df_valid["BM_Ratio"] <= BM_RATIO_MAX)
        ].copy()

        if len(df_valid) > 0:
            turnover_th = df_valid["TurnoverValue"].quantile(1 - VO_TOP_PCT)
            df_valid = df_valid[df_valid["TurnoverValue"] >= turnover_th]

        df_valid = df_valid[df_valid["TurnoverValue"] >= MIN_DAILY_TURNOVER]

        df_valid["MaxBuyableAmount"] = df_valid["Vo"] * MAX_VOLUME_PARTICIPATION * df_valid["AdjustmentClose"]
        target_capital = INITIAL_CAPITAL * LEVERAGE_FIXED * PRICE_GAP_BUFFER
        capital_per_stock = target_capital / TOP_N
        df_valid = df_valid[df_valid["MaxBuyableAmount"] >= capital_per_stock * LIQUIDITY_BUFFER_FACTOR]

        if len(df_valid) < TOP_N:
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0, "risk_off": False, "MonthEnd": month_end})
            continue

        top_n = df_valid.nlargest(TOP_N, "composite_score_ewma_all")
        w = 1.0 / len(top_n)
        for _, r in top_n.iterrows():
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": r["Code"], "Weight": w, "risk_off": False, "MonthEnd": month_end})

    df_portfolio = pd.DataFrame(portfolio_list)
    return df_portfolio, float(vol3_th)


df_portfolio_all, vol3_th_used = build_monthly_targets(VOL3_Q_FIXED, TREND3_TH_FIXED)

# monthly CASH months count helper per period
def count_cash_months_in_range(df_portfolio, start_date, end_date):
    # MonthKey_next are months where portfolio applies. We count those within [start,end] based on first trading day.
    # Convert MonthKey_next to month start date for filtering.
    mk = pd.to_datetime(df_portfolio["MonthKey_next"] + "-01")
    in_range = (mk >= pd.Timestamp(start_date).to_period("M").to_timestamp()) & (mk <= pd.Timestamp(end_date).to_period("M").to_timestamp())
    dfp = df_portfolio.loc[in_range].copy()
    return int(dfp[dfp["Code"] == "CASH"]["MonthKey_next"].nunique())

# Build sets and mapping for daily backtest (we will slice daily by period)
cash_tbl_all = df_portfolio_all[df_portfolio_all["Code"] == "CASH"][["MonthKey_next"]].rename(columns={"MonthKey_next": "MonthKey"})
cash_month_set_all = set(cash_tbl_all["MonthKey"].unique())

stock_portfolio_all = df_portfolio_all[df_portfolio_all["Code"] != "CASH"].copy()
monthly_target_all = {}
for mk, g in stock_portfolio_all.groupby("MonthKey_next"):
    monthly_target_all[mk] = dict(zip(g["Code"].astype(str), g["Weight"].astype(float)))

daily_port_all = daily.merge(
    stock_portfolio_all[["MonthKey_next", "Code", "Weight"]],
    left_on=["MonthKey", "Code"],
    right_on=["MonthKey_next", "Code"],
    how="inner"
).sort_values("Date").copy()


# ============================================================
# Daily backtest for a given period
# ============================================================
def run_daily_period(start_dt: pd.Timestamp, end_dt: pd.Timestamp) -> tuple[pd.DataFrame, float, int, int]:
    period_dates = [d for d in all_dates if (d >= start_dt) and (d <= end_dt)]
    if not period_dates:
        raise ValueError("指定期間に日次データがありません")

    # IMPORTANT: keep state within the period (reset) to avoid leakage across IS/OOS
    global_wealth = INITIAL_CAPITAL
    global_peak = INITIAL_CAPITAL
    current_hold_weights = {}
    current_state = "CASH"

    rebalance_exec = 0
    rebalance_skip = 0
    total_trading_cost = 0.0

    daily_rows = []

    for dt in period_dates:
        mk = pd.Timestamp(dt).to_period("M").strftime("%Y-%m")
        is_cash = mk in cash_month_set_all

        is_first = (mk in first_trade_day_of_month) and (pd.Timestamp(dt) == pd.Timestamp(first_trade_day_of_month[mk]))
        trading_cost = 0.0

        if is_first:
            target_w = monthly_target_all.get(mk, {})
            target_state = "CASH" if is_cash else "EQUITY"
            is_cash_transition = (current_state != target_state)

            prev_codes = list(current_hold_weights.keys())
            new_codes = list(target_w.keys())
            ov = overlap_ratio(prev_codes, new_codes) if (current_state == "EQUITY" and target_state == "EQUITY") else np.nan

            turnover, turnover_stocks, new_hold = calc_turnover_and_new_hold(current_hold_weights, target_w)

            if is_cash_transition:
                do_skip = False
            else:
                if target_state == "CASH":
                    do_skip = False
                else:
                    turnover_skip = (turnover_stocks < REBALANCE_MIN_TURNOVER_STOCKS) and (turnover < REBALANCE_MIN_TURNOVER_PCT)
                    overlap_skip = (USE_OVERLAP_SKIP and (ov >= OVERLAP_SKIP_TH))
                    do_skip = (turnover_skip or overlap_skip)

            if do_skip:
                rebalance_skip += 1
            else:
                rebalance_exec += 1
                if target_state == "CASH":
                    current_hold_weights = {}
                    current_state = "CASH"
                else:
                    trading_cost = turnover * TOTAL_COST_PER_TRADE
                    total_trading_cost += trading_cost
                    current_hold_weights = new_hold
                    current_state = "EQUITY"

        if is_cash:
            port_ret_gross = 0.0
            port_ret_net = 0.0
        else:
            df_day = daily_port_all[daily_port_all["Date"] == dt].copy()
            if len(df_day) == 0:
                continue
            df_day["weighted_ret"] = df_day["ret_d"].fillna(0) * df_day["Weight"]
            port_ret_gross = df_day["weighted_ret"].sum() * LEVERAGE_FIXED
            port_ret_net = port_ret_gross - trading_cost

        global_wealth *= (1 + port_ret_net)
        if global_wealth > global_peak:
            global_peak = global_wealth
        dd = global_wealth / global_peak - 1

        daily_rows.append({
            "Date": dt,
            "port_ret_net": port_ret_net,
            "port_ret_gross": port_ret_gross,
            "trading_cost": trading_cost,
            "dd": dd,
            "is_cash": is_cash
        })

    df_daily = pd.DataFrame(daily_rows)
    return df_daily, total_trading_cost, rebalance_exec, rebalance_skip


# ============================================================
# Run IS / OOS
# ============================================================
print("\n実行パラメータ固定:")
print(f"  VOL3_Q={VOL3_Q_FIXED}, TREND3_TH={TREND3_TH_FIXED}, Vol3閾値={vol3_th_used:.6f}")
print("期間分割:")
print(f"  IS : {IS_START.date()} 〜 {IS_END.date()}")
print(f"  OOS: {OOS_START.date()} 〜 {OOS_END.date()}")

# IS
df_is, cost_is, re_is, rs_is = run_daily_period(IS_START, IS_END)
is_cash_months = count_cash_months_in_range(df_portfolio_all, IS_START, IS_END)
is_summ = summarize_daily(df_is, cost_is)
is_summ["区分"] = "IS (2016-2021)"
is_summ["CASH月数"] = is_cash_months
is_summ["リバランス実行回数"] = re_is
is_summ["リバランス見送り回数"] = rs_is

# OOS
df_oos, cost_oos, re_oos, rs_oos = run_daily_period(OOS_START, OOS_END)
oos_cash_months = count_cash_months_in_range(df_portfolio_all, OOS_START, OOS_END)
oos_summ = summarize_daily(df_oos, cost_oos)
oos_summ["区分"] = "OOS (2022-2026)"
oos_summ["CASH月数"] = oos_cash_months
oos_summ["リバランス実行回数"] = re_oos
oos_summ["リバランス見送り回数"] = rs_oos

# ============================================================
# 日本語比較表
# ============================================================
df_cmp = pd.DataFrame([is_summ, oos_summ])

# 表示列
cols = [
    "区分",
    "期間開始", "期間終了", "日数",
    "年率(%)", "Sharpe", "最大DD(%)", "最大DD日",
    "CASH月数", "平均CASH比率(%)",
    "取引コスト累積(%)",
    "リバランス実行回数", "リバランス見送り回数",
    "最終資産",
]
df_cmp = df_cmp[cols].copy()

# 表示用丸め
df_cmp["年率(%)"] = df_cmp["年率(%)"].round(2)
df_cmp["Sharpe"] = df_cmp["Sharpe"].round(4)
df_cmp["最大DD(%)"] = df_cmp["最大DD(%)"].round(2)
df_cmp["平均CASH比率(%)"] = df_cmp["平均CASH比率(%)"].round(2)
df_cmp["取引コスト累積(%)"] = df_cmp["取引コスト累積(%)"].round(2)
df_cmp["最終資産"] = df_cmp["最終資産"].round(0).astype(np.int64)

print("\n" + "="*110)
print("【期間分割バックテスト比較（日本語）】VOL3_Q=0.92, TREND3_TH=-0.025 固定")
print("="*110)
print(df_cmp.to_string(index=False))

out_path = ANALYSIS_DIR / "comparison_is_oos_ja.csv"
df_cmp.to_csv(out_path, index=False, encoding="utf-8-sig")
print("\n保存しました:", out_path)


スナップショット読込中...
EWMAスコア計算中...
日次データ読込中...
日次データ期間: 2016-01-15 〜 2026-01-09

実行パラメータ固定:
  VOL3_Q=0.92, TREND3_TH=-0.025, Vol3閾値=0.064933
期間分割:
  IS : 2016-02-01 〜 2021-12-31
  OOS: 2022-01-01 〜 2026-01-09

【期間分割バックテスト比較（日本語）】VOL3_Q=0.92, TREND3_TH=-0.025 固定
             区分       期間開始       期間終了   日数  年率(%)  Sharpe  最大DD(%)      最大DD日  CASH月数  平均CASH比率(%)  取引コスト累積(%)  リバランス実行回数  リバランス見送り回数     最終資産
 IS (2016-2021) 2016-02-01 2021-12-30 1447  13.59  1.1327   -16.79 2021-11-30      29        40.64        8.44         57          14 21239484
OOS (2022-2026) 2022-01-04 2026-01-09  983  33.22  1.6695   -21.93 2024-08-05       1         2.03       12.18         43           6 31625166

保存しました: C:\Users\yongr\Project\merged_data_all_stocks\analysis_daily\comparison_is_oos_ja.csv


In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 固定パラメータ（あなたの指定）
# ============================================================
VOL3_Q_FIXED = 0.92
TREND3_TH_FIXED = -0.025

# ============================================================
# 基本設定（あなたの環境）
# ============================================================
OUTPUT_DIR = Path(r"C:\Users\yongr\Project\merged_data_all_stocks")
SNAP_PATH = OUTPUT_DIR / "factors" / "month_end_snapshot.parquet"
MERGED_PARTS_DIR = OUTPUT_DIR / "merged_parts"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_daily"
ANALYSIS_DIR.mkdir(exist_ok=True)

# ============================================================
# その他パラメータ（現行v2.1系を踏襲：固定）
# ============================================================
INITIAL_CAPITAL = 10_000_000
TOP_N = 20
VO_TOP_PCT = 0.70
MIN_PRICE = 800

ROE_MIN = -0.5
ROE_MAX = 1.0
BM_RATIO_MIN = 0.1
BM_RATIO_MAX = 10.0

OFF_TH = -0.115

COMMISSION_RATE = 0.001
SLIPPAGE_RATE = 0.003
TOTAL_COST_PER_TRADE = COMMISSION_RATE + SLIPPAGE_RATE  # 片道0.4%

MAX_VOLUME_PARTICIPATION = 0.10
MIN_DAILY_TURNOVER = 5_000_000

PRICE_GAP_BUFFER = 0.95
LEVERAGE_FIXED = 1.0

WEIGHT_ADJUST_DEADBAND = 0.05
REBALANCE_MIN_TURNOVER_STOCKS = 5
REBALANCE_MIN_TURNOVER_PCT = 0.10

LIQUIDITY_BUFFER_FACTOR = 0.70
SCORE_EWMA_ALPHA = 0.50
OVERLAP_SKIP_TH = 0.80
USE_OVERLAP_SKIP = True

# ============================================================
# ローリング設定：3年IS → 次の1年OOS
# ============================================================
IS_YEARS = 3

# 例：2016–2018→2019, 2017–2019→2020...
ROLL_START_YEAR = 2016
ROLL_END_OOS_YEAR = 2025  # 2025までOOSを作れる想定（データ終端により自動調整）


# ============================================================
# Utilities
# ============================================================
def detect_stock_code_column(df):
    candidates = ["Code", "StockCode", "Symbol", "Ticker", "stock_code", "code"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"銘柄コード列が見つかりません。利用可能な列: {list(df.columns)}")

def normalize_columns(df, stock_code_col=None):
    rename_map = {}
    if stock_code_col and stock_code_col != "Code":
        rename_map[stock_code_col] = "Code"
    if "AdjustedClose" in df.columns and "AdjustmentClose" not in df.columns:
        rename_map["AdjustedClose"] = "AdjustmentClose"
    if "Volume" in df.columns and "Vo" not in df.columns:
        rename_map["Volume"] = "Vo"
    return df.rename(columns=rename_map)

def add_ewma_score_to_snapshot(snap: pd.DataFrame, alpha: float) -> pd.DataFrame:
    snap = snap.sort_values(["Code", "MonthEnd"]).copy()

    def _calc_month_z_and_score(g):
        g = g.copy()
        for col in ["BM_Ratio", "ROE", "INV_Growth"]:
            mean_val = g[col].mean()
            std_val = g[col].std()
            g[f"{col}_z_all"] = (g[col] - mean_val) / std_val if (std_val is not None and std_val > 0) else 0.0
        g["composite_score_raw_all"] = (
            g["BM_Ratio_z_all"] +
            g["ROE_z_all"] -
            g["INV_Growth_z_all"].fillna(0)
        )
        return g

    snap = snap.groupby("MonthEnd", group_keys=False).apply(_calc_month_z_and_score)
    snap["composite_score_ewma_all"] = (
        snap.groupby("Code")["composite_score_raw_all"]
            .transform(lambda s: s.ewm(alpha=alpha, adjust=False).mean())
    )
    return snap

def overlap_ratio(prev_codes, new_codes) -> float:
    prev_set = set(prev_codes)
    new_set = set(new_codes)
    if len(prev_set) == 0:
        return 0.0
    return len(prev_set & new_set) / max(len(prev_set), 1)

def calc_turnover_and_new_hold(prev_w, target_w, deadband=WEIGHT_ADJUST_DEADBAND):
    prev = prev_w.copy()
    target = target_w.copy()
    all_codes = set(prev.keys()) | set(target.keys())
    after = {}

    for c in all_codes:
        w0 = prev.get(c, 0.0)
        w1 = target.get(c, 0.0)
        if (c in prev) and (c in target):
            if abs(w1 - w0) < deadband:
                after[c] = w0
            else:
                after[c] = w1
        else:
            after[c] = w1

    after = {c: w for c, w in after.items() if w > 0}
    s = sum(after.values())
    if s > 0:
        after = {c: w/s for c, w in after.items()}

    codes2 = set(prev.keys()) | set(after.keys())
    turnover = sum(abs(after.get(c, 0.0) - prev.get(c, 0.0)) for c in codes2)
    turnover_stocks = len(set(prev.keys()) ^ set(after.keys()))
    return turnover, turnover_stocks, after

def summarize_oos(df_daily_returns: pd.DataFrame, total_trading_cost_sum: float) -> dict:
    df = df_daily_returns.sort_values("Date").reset_index(drop=True).copy()

    # wealth from INITIAL_CAPITAL (period-start reset)
    wealth = INITIAL_CAPITAL * (1 + df["port_ret_net"]).cumprod()
    peak = wealth.cummax()
    dd = wealth / peak - 1

    first_date = df["Date"].min()
    last_date = df["Date"].max()
    years = (last_date - first_date).days / 365.25 if last_date > first_date else 1.0

    final_wealth = float(wealth.iloc[-1])
    cum_return = final_wealth / INITIAL_CAPITAL - 1
    ann_return = (1 + cum_return) ** (1 / years) - 1
    ann_vol = df["port_ret_net"].std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0.0

    maxdd = float(dd.min())
    maxdd_date = df.loc[dd.idxmin(), "Date"].date()

    return {
        "OOS_期間開始": first_date.date(),
        "OOS_期間終了": last_date.date(),
        "OOS_日数": int(len(df)),
        "OOS_年率(%)": ann_return * 100,
        "OOS_Sharpe": sharpe,
        "OOS_最大DD(%)": maxdd * 100,
        "OOS_最大DD日": maxdd_date,
        "OOS_取引コスト累積(%)": total_trading_cost_sum * 100,
        "OOS_最終資産": int(round(final_wealth))
    }


# ============================================================
# Load snapshot + daily once
# ============================================================
print("スナップショット読込中...")
snap = pd.read_parquet(SNAP_PATH)
snap["MonthEnd"] = pd.to_datetime(snap["MonthEnd"])
snap = normalize_columns(snap, detect_stock_code_column(snap))
snap = snap.sort_values(["Code", "MonthEnd"]).copy()
snap["MonthKey"] = snap["MonthEnd"].dt.to_period("M").astype(str)

print("EWMAスコア計算中...")
snap = add_ewma_score_to_snapshot(snap, alpha=SCORE_EWMA_ALPHA)

print("日次データ読込中...")
daily_files = sorted(MERGED_PARTS_DIR.glob("merged-part-*.parquet"))
sample_df = pd.read_parquet(daily_files[0])
daily_stock_code_col = detect_stock_code_column(sample_df)

daily_list = []
for f in daily_files:
    df_part = pd.read_parquet(f)
    df_part = normalize_columns(df_part, daily_stock_code_col)
    daily_list.append(df_part)

daily = pd.concat(daily_list, ignore_index=True)
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values(["Code", "Date"]).copy()
if "AdjustedClose" in daily.columns and "AdjustmentClose" not in daily.columns:
    daily = daily.rename(columns={"AdjustedClose": "AdjustmentClose"})
daily["ret_d"] = daily.groupby("Code")["AdjustmentClose"].pct_change()
daily["MonthKey"] = daily["Date"].dt.to_period("M").astype(str)

daily_start = daily["Date"].min()
daily_end = daily["Date"].max()
print(f"日次データ期間: {daily_start.date()} 〜 {daily_end.date()}")

first_trade_day_of_month = daily.groupby("MonthKey")["Date"].min().to_dict()


# ============================================================
# Core: build monthly targets using IS-only vol3_th and then apply to OOS
# ============================================================
def build_monthly_targets_with_is_vol3th(is_start: pd.Timestamp, is_end: pd.Timestamp, vol3_q: float, trend3_th: float):
    """
    重要：
    - vol3_thはIS期間の monthly['vol3'] 分布で quantile を計算（未来情報を使わない）
    - risk_off判定自体は全期間に適用（ただしvol3_thはIS由来）
    """
    snap2 = snap.copy()

    snap2["ret_m_fwd"] = snap2.groupby("Code")["AdjustmentClose"].pct_change().shift(-1)
    snap2["TurnoverValue"] = snap2["Vo"] * snap2["AdjustmentClose"]

    monthly = snap2.groupby("MonthEnd").apply(
        lambda g: pd.Series({
            "mkt_vw": np.average(g["ret_m_fwd"].fillna(0), weights=g["MarketCap"].fillna(1)),
            "count": len(g)
        })
    ).reset_index()

    monthly["cum_ret"] = (1 + monthly["mkt_vw"]).cumprod()
    monthly["peak"] = monthly["cum_ret"].cummax()
    monthly["mkt_dd"] = (monthly["cum_ret"] / monthly["peak"]) - 1
    monthly["trend3"] = monthly["mkt_vw"].rolling(3, min_periods=1).mean()
    monthly["vol3"] = monthly["mkt_vw"].rolling(3, min_periods=2).std()

    # IS期間のMonthEndでフィルタ（IS年の月末だけ）
    is_monthly = monthly[(monthly["MonthEnd"] >= is_start) & (monthly["MonthEnd"] <= is_end)].copy()
    if len(is_monthly) < 12:
        raise ValueError("IS期間の月数が少なすぎます")

    vol3_th_is = float(is_monthly["vol3"].quantile(vol3_q))

    # 全期間にmkt_dd/trend3/vol3を付与（vol3閾値はIS由来で固定）
    snap2 = snap2.merge(monthly[["MonthEnd", "mkt_dd", "trend3", "vol3"]], on="MonthEnd", how="left")
    snap2["risk_off"] = (
        (snap2["mkt_dd"] <= OFF_TH) |
        (snap2["trend3"] <= trend3_th) |
        (snap2["vol3"] >= vol3_th_is)
    )

    # 月次ターゲット（翌月適用）
    portfolio_list = []
    for month_end, df_month in snap2.groupby("MonthEnd"):
        risk_off = bool(df_month["risk_off"].iloc[0])
        month_key_next = (pd.Timestamp(month_end) + pd.DateOffset(months=1)).to_period("M").strftime("%Y-%m")

        if risk_off:
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0, "MonthEnd": month_end})
            continue

        df_valid = df_month[df_month["AdjustmentClose"] >= MIN_PRICE].copy()
        df_valid = df_valid[
            (df_valid["BM_Ratio"].notna()) & (df_valid["ROE"].notna()) &
            (df_valid["ROE"] >= ROE_MIN) & (df_valid["ROE"] <= ROE_MAX) &
            (df_valid["BM_Ratio"] >= BM_RATIO_MIN) & (df_valid["BM_Ratio"] <= BM_RATIO_MAX)
        ].copy()

        if len(df_valid) > 0:
            turnover_th = df_valid["TurnoverValue"].quantile(1 - VO_TOP_PCT)
            df_valid = df_valid[df_valid["TurnoverValue"] >= turnover_th]

        df_valid = df_valid[df_valid["TurnoverValue"] >= MIN_DAILY_TURNOVER]

        df_valid["MaxBuyableAmount"] = df_valid["Vo"] * MAX_VOLUME_PARTICIPATION * df_valid["AdjustmentClose"]
        target_capital = INITIAL_CAPITAL * LEVERAGE_FIXED * PRICE_GAP_BUFFER
        capital_per_stock = target_capital / TOP_N
        df_valid = df_valid[df_valid["MaxBuyableAmount"] >= capital_per_stock * LIQUIDITY_BUFFER_FACTOR]

        if len(df_valid) < TOP_N:
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": "CASH", "Weight": 1.0, "MonthEnd": month_end})
            continue

        top_n = df_valid.nlargest(TOP_N, "composite_score_ewma_all")
        w = 1.0 / len(top_n)
        for _, r in top_n.iterrows():
            portfolio_list.append({"MonthKey_next": month_key_next, "Code": r["Code"], "Weight": w, "MonthEnd": month_end})

    df_portfolio = pd.DataFrame(portfolio_list)
    return df_portfolio, vol3_th_is


def run_oos_year(df_portfolio_all: pd.DataFrame, oos_start: pd.Timestamp, oos_end: pd.Timestamp):
    # OOSに該当するMonthKey集合
    oos_monthkeys = set(pd.date_range(oos_start, oos_end, freq="MS").to_period("M").astype(str).tolist())

    # CASH月数（OOS期間内のMonthKey_nextでCASHになっている月数）
    dfp = df_portfolio_all[df_portfolio_all["MonthKey_next"].isin(oos_monthkeys)].copy()
    cash_months = int(dfp[dfp["Code"] == "CASH"]["MonthKey_next"].nunique())

    # 日次OOS期間
    period_dates = sorted([d for d in daily["Date"].unique() if (d >= oos_start) and (d <= oos_end)])
    if len(period_dates) == 0:
        return None

    # cash set
    cash_tbl = dfp[dfp["Code"] == "CASH"][["MonthKey_next"]].rename(columns={"MonthKey_next": "MonthKey"})
    cash_month_set = set(cash_tbl["MonthKey"].unique())

    # stock targets
    stock_portfolio = dfp[dfp["Code"] != "CASH"].copy()
    monthly_target = {}
    for mk, g in stock_portfolio.groupby("MonthKey_next"):
        monthly_target[mk] = dict(zip(g["Code"].astype(str), g["Weight"].astype(float)))

    daily_port = daily.merge(
        stock_portfolio[["MonthKey_next", "Code", "Weight"]],
        left_on=["MonthKey", "Code"],
        right_on=["MonthKey_next", "Code"],
        how="inner"
    )

    # state reset per OOS year (公平比較)
    current_hold_weights = {}
    current_state = "CASH"

    reb_exec = 0
    reb_skip = 0
    total_cost = 0.0

    rows = []
    for dt in period_dates:
        mk = pd.Timestamp(dt).to_period("M").strftime("%Y-%m")
        is_cash = mk in cash_month_set

        is_first = (mk in first_trade_day_of_month) and (pd.Timestamp(dt) == pd.Timestamp(first_trade_day_of_month[mk]))
        trading_cost = 0.0

        if is_first:
            target_w = monthly_target.get(mk, {})
            target_state = "CASH" if is_cash else "EQUITY"
            is_cash_transition = (current_state != target_state)

            prev_codes = list(current_hold_weights.keys())
            new_codes = list(target_w.keys())
            ov = overlap_ratio(prev_codes, new_codes) if (current_state == "EQUITY" and target_state == "EQUITY") else np.nan

            turnover, turnover_stocks, new_hold = calc_turnover_and_new_hold(current_hold_weights, target_w)

            if is_cash_transition:
                do_skip = False
            else:
                if target_state == "CASH":
                    do_skip = False
                else:
                    turnover_skip = (turnover_stocks < REBALANCE_MIN_TURNOVER_STOCKS) and (turnover < REBALANCE_MIN_TURNOVER_PCT)
                    overlap_skip = (USE_OVERLAP_SKIP and (ov >= OVERLAP_SKIP_TH))
                    do_skip = (turnover_skip or overlap_skip)

            if do_skip:
                reb_skip += 1
            else:
                reb_exec += 1
                if target_state == "CASH":
                    current_hold_weights = {}
                    current_state = "CASH"
                else:
                    trading_cost = turnover * TOTAL_COST_PER_TRADE
                    total_cost += trading_cost
                    current_hold_weights = new_hold
                    current_state = "EQUITY"

        if is_cash:
            port_ret_net = 0.0
            port_ret_gross = 0.0
        else:
            df_day = daily_port[daily_port["Date"] == dt].copy()
            if len(df_day) == 0:
                continue
            df_day["weighted_ret"] = df_day["ret_d"].fillna(0) * df_day["Weight"]
            port_ret_gross = df_day["weighted_ret"].sum() * LEVERAGE_FIXED
            port_ret_net = port_ret_gross - trading_cost

        rows.append({
            "Date": pd.Timestamp(dt),
            "port_ret_net": port_ret_net,
            "port_ret_gross": port_ret_gross,
            "trading_cost": trading_cost,
            "dd": 0.0,  # later
            "is_cash": is_cash
        })

    df_daily = pd.DataFrame(rows)
    if len(df_daily) == 0:
        return None

    # dd compute on wealth series
    wealth = INITIAL_CAPITAL * (1 + df_daily["port_ret_net"]).cumprod()
    peak = wealth.cummax()
    df_daily["dd"] = wealth / peak - 1

    summ = summarize_oos(df_daily, total_cost)
    summ["OOS年"] = int(oos_start.year)
    summ["CASH月数"] = cash_months
    summ["リバランス実行回数"] = reb_exec
    summ["リバランス見送り回数"] = reb_skip
    return summ


# ============================================================
# Rolling evaluation loop
# ============================================================
# Determine feasible OOS years from data range
min_year = daily_start.year
max_year = daily_end.year

# We need: IS window of 3y + next 1y OOS fully/partially available
# We'll generate OOS years starting from ROLL_START_YEAR+3 to max_year (but not beyond data)
oos_years = []
for oos_y in range(ROLL_START_YEAR + IS_YEARS, max_year + 1):
    # OOS year must have some data
    oos_start = pd.Timestamp(f"{oos_y}-01-01")
    oos_end = pd.Timestamp(f"{oos_y}-12-31")
    if oos_start > daily_end:
        continue
    oos_years.append(oos_y)

print("\nローリング検証設定:")
print(f"  IS=過去{IS_YEARS}年, OOS=次の1年")
print(f"  固定パラメータ: VOL3_Q={VOL3_Q_FIXED}, TREND3_TH={TREND3_TH_FIXED}, OFF_TH={OFF_TH}")
print("  OOS対象年:", oos_years)

results = []

for oos_y in oos_years:
    is_start = pd.Timestamp(f"{oos_y-IS_YEARS}-01-01")
    is_end   = pd.Timestamp(f"{oos_y-1}-12-31")
    oos_start = pd.Timestamp(f"{oos_y}-01-01")
    oos_end = pd.Timestamp(f"{oos_y}-12-31")
    if oos_end > daily_end:
        oos_end = daily_end  # 最終年は部分年になる可能性

    # 月末ベースのIS期間に合わせる（MonthEndなので月末に寄せる）
    is_end_monthend = pd.Timestamp(f"{oos_y-1}-12-31")

    print(f"\nOOS年 {oos_y}: IS {is_start.date()}〜{is_end.date()} → OOS {oos_start.date()}〜{oos_end.date()}")

    # Build monthly targets using IS-only vol3_th
    df_portfolio_all, vol3_th_is = build_monthly_targets_with_is_vol3th(
        is_start=is_start, is_end=is_end_monthend,
        vol3_q=VOL3_Q_FIXED, trend3_th=TREND3_TH_FIXED
    )

    # Run OOS year
    summ = run_oos_year(df_portfolio_all, oos_start, oos_end)
    if summ is None:
        continue

    summ["IS開始"] = is_start.date()
    summ["IS終了"] = is_end.date()
    summ["VOL3_Q(IS)"] = VOL3_Q_FIXED
    summ["Vol3閾値(IS)"] = round(vol3_th_is, 6)
    summ["TREND3_TH"] = TREND3_TH_FIXED
    results.append(summ)

df_out = pd.DataFrame(results)
if len(df_out) == 0:
    raise ValueError("ローリング結果が空です（期間やデータ範囲を確認してください）")

# ============================================================
# 日本語まとめ表（OOS年ごと）
# ============================================================
cols = [
    "OOS年",
    "IS開始", "IS終了",
    "OOS_期間開始", "OOS_期間終了", "OOS_日数",
    "VOL3_Q(IS)", "Vol3閾値(IS)", "TREND3_TH",
    "OOS_年率(%)", "OOS_Sharpe", "OOS_最大DD(%)", "OOS_最大DD日",
    "CASH月数", "OOS_取引コスト累積(%)",
    "リバランス実行回数", "リバランス見送り回数",
    "OOS_最終資産",
]
df_show = df_out[cols].copy()

# 丸め
df_show["OOS_年率(%)"] = df_show["OOS_年率(%)"].round(2)
df_show["OOS_Sharpe"] = df_show["OOS_Sharpe"].round(4)
df_show["OOS_最大DD(%)"] = df_show["OOS_最大DD(%)"].round(2)
df_show["OOS_取引コスト累積(%)"] = df_show["OOS_取引コスト累積(%)"].round(2)

print("\n" + "="*140)
print("【ローリング検証：3年IS→次1年OOS（OOS年ごとの成績）】 日本語まとめ表")
print("="*140)
print(df_show.to_string(index=False))

out_path = ANALYSIS_DIR / "rolling_oos_3y_is_1y_oos_ja.csv"
df_show.to_csv(out_path, index=False, encoding="utf-8-sig")
print("\n保存しました:", out_path)


スナップショット読込中...
EWMAスコア計算中...
日次データ読込中...
日次データ期間: 2016-01-15 〜 2026-01-09

ローリング検証設定:
  IS=過去3年, OOS=次の1年
  固定パラメータ: VOL3_Q=0.92, TREND3_TH=-0.025, OFF_TH=-0.115
  OOS対象年: [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

OOS年 2019: IS 2016-01-01〜2018-12-31 → OOS 2019-01-01〜2019-12-31

OOS年 2020: IS 2017-01-01〜2019-12-31 → OOS 2020-01-01〜2020-12-31

OOS年 2021: IS 2018-01-01〜2020-12-31 → OOS 2021-01-01〜2021-12-31

OOS年 2022: IS 2019-01-01〜2021-12-31 → OOS 2022-01-01〜2022-12-31

OOS年 2023: IS 2020-01-01〜2022-12-31 → OOS 2023-01-01〜2023-12-31

OOS年 2024: IS 2021-01-01〜2023-12-31 → OOS 2024-01-01〜2024-12-31

OOS年 2025: IS 2022-01-01〜2024-12-31 → OOS 2025-01-01〜2025-12-31

OOS年 2026: IS 2023-01-01〜2025-12-31 → OOS 2026-01-01〜2026-01-09

【ローリング検証：3年IS→次1年OOS（OOS年ごとの成績）】 日本語まとめ表
 OOS年       IS開始       IS終了   OOS_期間開始   OOS_期間終了  OOS_日数  VOL3_Q(IS)  Vol3閾値(IS)  TREND3_TH  OOS_年率(%)  OOS_Sharpe  OOS_最大DD(%)  OOS_最大DD日  CASH月数  OOS_取引コスト累積(%)  リバランス実行回数  リバランス見送り回数  OOS_最終資産
 2019 2016-01-01 201

In [17]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
import os
import logging
from datetime import datetime

warnings.filterwarnings('ignore')

# ===================================
# ロギング設定
# ===================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('backtest_long_only_october_unit_constraints.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ===================================
# グローバル変数
# ===================================
CACHE_FILE = 'topix_quarterly_statements.csv'

# 税金パラメータ
TAX_RATE = 0.20315  # 譲渡益税 20.315%

# 単位株制限
UNIT_SHARES = 100  # 100株単位

# -----------------------------------
# DDガード設定（要件）
# -----------------------------------
USE_DD_GUARD = True
DD_STOP = -0.10  # -10%で撤退（ピークからのDD）

REENTRY_MODE = "topix_filter"  # 要件
TOPIX_TICKER = "TOPIX"         # TOPIXデータに使うCode（後述の関数で生成）
TOPIX_MA_DAYS = 200

# -----------------------------------
# 売買コスト（要件）
# 往復0.4%（ターンオーバー比例）
# → ここでは「コスト率 = turnover * 0.004」を採用
#   turnover=売買金額 / エクイティ（全資産） として扱う（=月次側と揃えやすい）
# -----------------------------------
TOTAL_COST_ROUNDTRIP = 0.004


# ===================================
# A. 財務データ読み込み（列名自動判定版）
# ===================================

def load_financial_data(cache_filename: str = CACHE_FILE) -> pd.DataFrame:
    """財務諸表データをキャッシュから読み込み（列名を自動判定）"""

    if not os.path.exists(cache_filename):
        logger.error(f"財務データファイル '{cache_filename}' が見つかりません")
        return pd.DataFrame()

    try:
        df = pd.read_csv(
            cache_filename,
            encoding='utf-8-sig',
            parse_dates=['DisclosedDate'],
            low_memory=False
        )

        logger.info(f"財務データ読み込み成功: {len(df):,}件")

        column_mapping = {
            'IssuedShareTotal': ['IssuedShareTotal', 'NumberOfIssuedAndOutstandingSharesAtTheEndOfFiscalYearIncludingTreasuryStock'],
            'Equity': ['Equity', 'NetAssets', 'TotalEquity'],
            'Profit': ['Profit', 'NetIncome', 'ProfitAttributableToOwnersOfParent']
        }

        available_cols = df.columns.tolist()
        required_columns = {}

        for target_col, possible_names in column_mapping.items():
            found = False
            for possible_name in possible_names:
                if possible_name in available_cols:
                    required_columns[target_col] = possible_name
                    found = True
                    break
            if not found:
                required_columns[target_col] = None

        rename_dict = {v: k for k, v in required_columns.items() if v is not None}
        df = df.rename(columns=rename_dict)

        for col in ['IssuedShareTotal', 'Equity', 'Profit']:
            if col not in df.columns:
                df[col] = 0

        base_cols = ['Code', 'DisclosedDate']
        if 'CompanyName' in df.columns:
            base_cols.append('CompanyName')

        final_cols = base_cols + ['Profit', 'Equity', 'IssuedShareTotal']
        df = df[final_cols].copy()

        logger.info(f"使用列: {df.columns.tolist()}")

        return df

    except Exception as e:
        logger.error(f"財務データ読み込みエラー: {e}", exc_info=True)
        return pd.DataFrame()


# ===================================
# B. 株価データ読み込み（メモリ効率化版）
# ===================================

def load_existing_price_data(ohlcv_dir: str = './OHLCV_Adjusted') -> pd.DataFrame:
    """株価データ読み込み（メモリ効率化版）"""

    if not os.path.exists(ohlcv_dir):
        logger.error(f"株価データディレクトリ '{ohlcv_dir}' が見つかりません。")
        return pd.DataFrame()

    csv_files = sorted([
        f for f in os.listdir(ohlcv_dir)
        if f.startswith('OHLCV_Adjusted_') and f.endswith('.csv') and f != 'OHLCV_Adjusted_TOPIX.csv'
    ])

    if not csv_files:
        logger.error(f"ディレクトリ '{ohlcv_dir}' 内にCSVファイルが見つかりません。")
        return pd.DataFrame()

    logger.info(f"株価ファイル数: {len(csv_files)}個")

    all_dataframes = []
    usecols = ['Date', 'Ticker', 'AdjustmentClose']

    for csv_file in tqdm(csv_files, desc="株価ファイル読み込み中"):
        file_path = os.path.join(ohlcv_dir, csv_file)
        try:
            df = pd.read_csv(
                file_path,
                usecols=usecols,
                parse_dates=['Date'],
                dtype={'Ticker': 'Int64', 'AdjustmentClose': 'float32'}
            )

            df = df.drop_duplicates(subset=['Ticker', 'Date'], keep='first')
            all_dataframes.append(df)

        except Exception as e:
            logger.warning(f"ファイル読み込みエラー ({csv_file}): {e}")
            continue

    if not all_dataframes:
        return pd.DataFrame()

    logger.info("全ファイルを結合中...")
    df_all = pd.concat(all_dataframes, ignore_index=True)

    logger.info(f"結合完了: {len(df_all):,}件")

    df_all = df_all.rename(columns={'Ticker': 'Code', 'AdjustmentClose': 'Close'})
    df_all['Code'] = df_all['Code'].astype('str').str.replace('<NA>', '0').str.zfill(4)
    df_all = df_all.dropna(subset=['Close'])

    logger.info("重複除去 & ソート中...")
    df_all = df_all.sort_values(['Code', 'Date'])
    df_all = df_all.drop_duplicates(subset=['Code', 'Date'], keep='first')

    logger.info(f"重複除去後: {len(df_all):,}件")

    return df_all


# ===================================
# TOPIXデータ作成（既存データから近似）
# ・あなたのコードの topix_return は「全銘柄の一部平均」を使っていたので、
#   同じ発想で日次TOPIX（疑似）を作る：上位N銘柄の等ウェイト指数（Close平均）
# ・厳密TOPIXが欲しければ OHLCV_Adjusted_TOPIX.csv を読み込む方式に差し替え可能
# ===================================

def build_pseudo_topix_series(prices_df: pd.DataFrame, n: int = 200) -> pd.Series:
    """
    疑似TOPIX（日次）：任意のn銘柄の等ウェイト平均Close
    ※最小差分のための簡易指標
    """
    codes = prices_df['Code'].dropna().unique().tolist()
    if len(codes) == 0:
        return pd.Series(dtype=float)

    use_codes = codes[:min(n, len(codes))]

    sub = prices_df[prices_df['Code'].isin(use_codes)].copy()
    piv = sub.pivot_table(index='Date', columns='Code', values='Close', aggfunc='last').sort_index()
    topix = piv.mean(axis=1).ffill()
    topix.name = TOPIX_TICKER
    return topix


def compute_topix_filter_signal(topix_close: pd.Series, ma_days: int = 200) -> pd.Series:
    """
    True: TOPIXが200MAを上回る（リスクオン/再エントリー許可）
    """
    ma = topix_close.rolling(ma_days, min_periods=ma_days).mean()
    signal = topix_close > ma
    signal = signal.fillna(False)
    return signal


# ===================================
# C. 財務指標計算（チャンク処理版）
# ===================================

def safe_code_to_int(code_series: pd.Series) -> pd.Series:
    cleaned = code_series.astype(str).str.replace(r'\D', '', regex=True)
    cleaned = cleaned.replace('', '0')
    return pd.to_numeric(cleaned, errors='coerce').fillna(0).astype('int64')


def calculate_market_metrics_fast_chunked(
    statements_df: pd.DataFrame,
    prices_df: pd.DataFrame,
    chunk_size: int = 200
) -> pd.DataFrame:
    logger.info("時価総額・PBR・ROE計算中（チャンク処理版）...")

    if prices_df.empty:
        logger.error("株価データが空です。株価データを読み込んでください。")
        return pd.DataFrame()

    if 'Code' not in prices_df.columns or 'Date' not in prices_df.columns or 'Close' not in prices_df.columns:
        logger.error(f"株価データに必要な列が存在しません。存在する列: {prices_df.columns.tolist()}")
        return pd.DataFrame()

    statements_df = statements_df.copy()
    statements_df['Profit'] = pd.to_numeric(statements_df['Profit'], errors='coerce').fillna(0)
    statements_df['Equity'] = pd.to_numeric(statements_df['Equity'], errors='coerce').fillna(0)
    statements_df['IssuedShareTotal'] = pd.to_numeric(statements_df['IssuedShareTotal'], errors='coerce').fillna(1)

    statements_df = statements_df[(statements_df['Equity'] > 0) & (statements_df['IssuedShareTotal'] > 0)]
    logger.info(f"有効な財務データ: {len(statements_df):,}件")

    logger.info("Code列を整数型に変換中...")
    statements_df['Code_int'] = safe_code_to_int(statements_df['Code'])
    prices_df = prices_df.copy()
    prices_df['Code_int'] = safe_code_to_int(prices_df['Code'])

    statements_df = statements_df[statements_df['Code_int'] > 0]
    prices_df = prices_df[prices_df['Code_int'] > 0]

    statements_df = statements_df.sort_values(['Code_int', 'DisclosedDate']).reset_index(drop=True)
    prices_df = prices_df.sort_values(['Code_int', 'Date']).reset_index(drop=True)

    statements_df = statements_df.drop_duplicates(subset=['Code_int', 'DisclosedDate'], keep='first')
    prices_df = prices_df.drop_duplicates(subset=['Code_int', 'Date'], keep='first')

    logger.info(f"ソート・重複除去後: 財務 {len(statements_df):,}件, 株価 {len(prices_df):,}件")

    logger.info("銘柄ごとにマージ中...")

    statements_groups = list(statements_df.groupby('Code_int'))
    prices_dict = {code: group for code, group in prices_df.groupby('Code_int')}

    merged_list = []
    num_chunks = (len(statements_groups) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="マージ処理"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, len(statements_groups))

        chunk_groups = statements_groups[start_idx:end_idx]

        for code, stmt_code in chunk_groups:
            if code not in prices_dict:
                continue

            price_code = prices_dict[code]
            if len(price_code) == 0:
                continue

            stmt_code = stmt_code.sort_values('DisclosedDate').reset_index(drop=True)
            price_code = price_code.sort_values('Date').reset_index(drop=True)

            try:
                merged = pd.merge_asof(
                    stmt_code,
                    price_code[['Date', 'Close']],
                    left_on='DisclosedDate',
                    right_on='Date',
                    direction='backward',  # ルックアヘッド回避
                    tolerance=pd.Timedelta(days=10)
                )
                if not merged.empty:
                    merged_list.append(merged)
            except Exception:
                continue

    if not merged_list:
        logger.error("マージ結果が空です")
        return pd.DataFrame()

    df_merged = pd.concat(merged_list, ignore_index=True)
    df_merged = df_merged.dropna(subset=['Close'])

    logger.info(f"マージ完了: {len(df_merged):,}件")

    df_merged['MarketCap'] = df_merged['Close'] * df_merged['IssuedShareTotal']
    df_merged['PBR'] = df_merged['MarketCap'] / df_merged['Equity']
    df_merged['ROE'] = (df_merged['Profit'] / df_merged['Equity']) * 100

    result_cols = ['Code', 'DisclosedDate', 'Close', 'MarketCap', 'PBR', 'ROE', 'Date']
    if 'CompanyName' in df_merged.columns:
        result_cols.insert(1, 'CompanyName')

    result_df = df_merged[result_cols].copy()
    result_df = result_df.rename(columns={'Close': 'StockPrice', 'Date': 'PriceDate'})

    mask = (
        (result_df['PBR'] > 0) &
        (result_df['PBR'] < 50) &
        (result_df['ROE'] > -100) &
        (result_df['ROE'] < 100) &
        (result_df['MarketCap'] > 1_000_000_000)
    )
    result_df = result_df[mask].copy()

    logger.info(f"計算完了: {len(result_df):,}件")

    return result_df


# ===================================
# D. ポートフォリオ構築（100株単位制限版）
# ===================================

def build_unit_share_portfolio(
    stock_candidates: pd.DataFrame,
    target_positions: int = 20,
    initial_capital: float = 10_000_000
) -> dict:
    if len(stock_candidates) == 0:
        return {'stocks': [], 'shares': [], 'prices': [], 'amounts': []}

    selected = stock_candidates.head(target_positions).copy()
    capital_per_stock = initial_capital / len(selected)

    stocks, shares_list, prices_list, amounts_list = [], [], [], []

    for _, row in selected.iterrows():
        code = row['Code']
        price = row['StockPrice']

        required_amount = price * UNIT_SHARES
        if required_amount <= capital_per_stock:
            shares = int(capital_per_stock // required_amount) * UNIT_SHARES
            if shares > 0:
                stocks.append(code)
                shares_list.append(shares)
                prices_list.append(price)
                amounts_list.append(shares * price)

    return {'stocks': stocks, 'shares': shares_list, 'prices': prices_list, 'amounts': amounts_list}


# ===================================
# 売買コスト（ターンオーバー比例）
# ===================================

def calc_turnover_cost(prev_equity: float, trade_value: float) -> float:
    """
    turnover = trade_value / prev_equity
    cost = turnover * TOTAL_COST_ROUNDTRIP
    """
    if prev_equity <= 0:
        return 0.0
    turnover = trade_value / prev_equity
    return turnover * TOTAL_COST_ROUNDTRIP


# ===================================
# 日次シミュレーション（DDガード + TOPIX200MA再エントリー + コスト）
# ===================================

def simulate_period_daily_with_dd_guard(
    portfolio: dict,
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
    prices_df: pd.DataFrame,
    topix_signal: pd.Series,
    initial_capital: float = 10_000_000
) -> dict:
    """
    年次区間（start_date〜end_date）を日次で評価し、
    DD_STOPで全撤退、TOPIX>200MAで再エントリー。
    売買コスト：往復0.4%（ターンオーバー比例）
    """

    if (not portfolio['stocks']) or prices_df.empty:
        return {
            "equity_curve": pd.Series(dtype=float),
            "final_equity": initial_capital,
            "gross_return": 0.0,
            "net_return": 0.0,
            "total_cost_rate": 0.0,
            "cash_days": 0,
            "dd_events": 0,
            "max_dd_daily": 0.0
        }

    # 対象期間の日付リストを作る（prices_dfのDateベース）
    all_dates = prices_df['Date'].dropna().sort_values().unique()
    all_dates = pd.DatetimeIndex(all_dates)
    period_dates = all_dates[(all_dates >= start_date) & (all_dates <= end_date)]
    if len(period_dates) == 0:
        return {
            "equity_curve": pd.Series(dtype=float),
            "final_equity": initial_capital,
            "gross_return": 0.0,
            "net_return": 0.0,
            "total_cost_rate": 0.0,
            "cash_days": 0,
            "dd_events": 0,
            "max_dd_daily": 0.0
        }

    stocks = list(portfolio['stocks'])
    shares = np.array(portfolio['shares'], dtype=float)

    # 価格pivotを当該期間だけ作る（最小差分、メモリ節約）
    sub_prices = prices_df[
        (prices_df['Code'].isin(stocks)) &
        (prices_df['Date'].isin(period_dates))
    ].copy()
    piv = sub_prices.pivot_table(index='Date', columns='Code', values='Close', aggfunc='last').sort_index()
    piv = piv.ffill()

    # 欠損が残る銘柄は落とす（最小実装）
    piv = piv.dropna(axis=1, how='any')
    kept = piv.columns.tolist()
    if len(kept) == 0:
        return {
            "equity_curve": pd.Series(dtype=float),
            "final_equity": initial_capital,
            "gross_return": 0.0,
            "net_return": 0.0,
            "total_cost_rate": 0.0,
            "cash_days": len(period_dates),
            "dd_events": 0,
            "max_dd_daily": 0.0
        }

    idx_map = [stocks.index(c) for c in kept]
    stocks = kept
    shares = shares[idx_map]

    # 初期建て（start_date 近傍の価格で建てた前提：portfolio['prices']を採用）
    # ※実際の約定日近似を厳密化するなら、start_dateのCloseで建て直す（第2段）
    start_prices = np.array(portfolio['prices'], dtype=float)[idx_map]
    invested = float(np.sum(shares * start_prices))
    cash = float(initial_capital - invested)

    # 初回建てコスト（新規買い＝売買金額 invested）
    # turnover = invested / equity
    total_cost_rate = 0.0
    total_cost_rate += calc_turnover_cost(prev_equity=initial_capital, trade_value=invested)

    in_market = True
    peak = initial_capital
    dd_events = 0
    cash_days = 0

    equity_list = []

    for dt, row in piv.iterrows():
        # TOPIXシグナル（再エントリー判定用）
        # topix_signalのindexにdtがない場合はFalse扱い
        topix_ok = bool(topix_signal.loc[dt]) if dt in topix_signal.index else False

        # 評価
        if in_market:
            stock_value = float(np.nansum(shares * row.values))
            equity = stock_value + cash
        else:
            equity = cash
            cash_days += 1

        # peak更新＆DD
        if equity > peak:
            peak = equity
        dd = (equity - peak) / peak  # 0以下

        # DDガード：撤退（全売り）
        if USE_DD_GUARD and in_market and (dd <= DD_STOP):
            # 全売却（売買金額 = stock_value）
            trade_value = float(np.nansum(shares * row.values))
            # コスト控除
            cost_rate = calc_turnover_cost(prev_equity=equity, trade_value=trade_value)
            total_cost_rate += cost_rate

            # キャッシュ化（コスト反映は equity から控除）
            equity_after_cost = equity * (1.0 - cost_rate)
            cash = equity_after_cost
            in_market = False
            dd_events += 1

        # 再エントリー：TOPIX > 200MA
        elif USE_DD_GUARD and (not in_market) and (REENTRY_MODE == "topix_filter") and topix_ok:
            # 全買い戻し（同じsharesを買い戻す前提）
            buy_value = float(np.nansum(shares * row.values))
            if buy_value <= cash:
                cost_rate = calc_turnover_cost(prev_equity=cash, trade_value=buy_value)
                total_cost_rate += cost_rate

                cash_after_cost = cash * (1.0 - cost_rate)
                # 現金から購入代金を支払い
                cash = cash_after_cost - buy_value
                in_market = True

        equity_list.append((dt, equity))

    equity_series = pd.Series(
        data=[v for _, v in equity_list],
        index=pd.DatetimeIndex([d for d, _ in equity_list]),
        name="Equity"
    )

    # 日次最大DD
    running_max = equity_series.cummax()
    dd_series = (equity_series - running_max) / running_max
    max_dd_daily = float(dd_series.min()) * 100.0

    final_equity_gross = float(equity_series.iloc[-1])
    gross_return = (final_equity_gross / initial_capital) - 1.0

    # 取引コスト反映（すでにequityに反映しているが、ここは「率」を返す）
    # 税：現状のコードに合わせ、区間利益に対して課税（簡略）
    taxable_profit = max(final_equity_gross - initial_capital, 0.0)
    tax_amount = taxable_profit * TAX_RATE
    final_equity_net = final_equity_gross - tax_amount
    net_return = (final_equity_net / initial_capital) - 1.0

    return {
        "equity_curve": equity_series,
        "final_equity_gross": final_equity_gross,
        "final_equity_net": final_equity_net,
        "gross_return": gross_return,
        "net_return": net_return,
        "tax_rate_total": (tax_amount / initial_capital),
        "total_cost_rate": total_cost_rate,
        "cash_days": cash_days,
        "dd_events": dd_events,
        "max_dd_daily": max_dd_daily
    }


# ===================================
# F. バックテスト（10月1日リバランス・100株単位版）DDガード正式実装
# ===================================

def smbc_value_quality_backtest_october_unit(
    enhanced_financial_data: pd.DataFrame,
    prices_df: pd.DataFrame,
    initial_capital: float = 10_000_000
) -> pd.DataFrame:
    """SMBC割安高質戦略（10月1日リバランス・100株単位制限版） + DDガード + TOPIX200MA再エントリー + コスト"""

    logger.info(f"10月1日リバランス戦略バックテスト実行中（DDガード・TOPIX200MA・コスト込み）...")

    # 10月1日のリバランス日を生成（2016年10月〜2025年10月）
    rebalance_dates = [pd.Timestamp(f'{year}-10-01') for year in range(2016, 2026)]

    logger.info(f"リバランス日数: {len(rebalance_dates)}回")
    logger.info(f"リバランス日: {[d.strftime('%Y-%m-%d') for d in rebalance_dates]}")

    # 疑似TOPIX（日次）と200MAフィルタ
    logger.info("疑似TOPIXを構築中（等ウェイト平均）...")
    topix_close = build_pseudo_topix_series(prices_df, n=200)
    if topix_close.empty:
        logger.error("TOPIX系列の生成に失敗しました。")
        return pd.DataFrame()

    topix_signal = compute_topix_filter_signal(topix_close, ma_days=TOPIX_MA_DAYS)
    logger.info("TOPIX 200日MAフィルタ生成完了")

    strategy_results = []

    for i, rebalance_date in enumerate(tqdm(rebalance_dates[:-1], desc="10月1日リバランス戦略（DDガード付き）")):
        next_rebalance = rebalance_dates[i + 1]

        # rebalance_date 時点で利用可能な財務を最新で1行
        current_data = enhanced_financial_data[
            enhanced_financial_data['DisclosedDate'] <= rebalance_date
        ].copy()
        current_data = current_data.sort_values('DisclosedDate').groupby('Code').tail(1)

        if len(current_data) < 100:
            logger.warning(f"{rebalance_date}: データ不足（{len(current_data)}銘柄）")
            continue

        # PBRとROEでランキング
        current_data['PBR_Rank'] = current_data['PBR'].rank(method='first', ascending=True)
        current_data['ROE_Rank'] = current_data['ROE'].rank(method='first', ascending=False)

        # 四分位分割
        current_data['PBR_Quartile'] = pd.qcut(current_data['PBR_Rank'], q=4, labels=[1, 2, 3, 4])
        current_data['ROE_Quartile'] = pd.qcut(current_data['ROE_Rank'], q=4, labels=[1, 2, 3, 4])

        # ロング候補：低PBR × 高ROE
        long_candidates = current_data[
            (current_data['PBR_Quartile'] == 1) &
            (current_data['ROE_Quartile'] == 4)
        ].nsmallest(50, 'PBR')

        # 100株単位ポートフォリオ構築
        long_portfolio = build_unit_share_portfolio(
            long_candidates,
            target_positions=20,
            initial_capital=initial_capital
        )

        invested = sum(long_portfolio['amounts'])
        logger.info(f"{rebalance_date.strftime('%Y-%m')}: {len(long_portfolio['stocks'])}銘柄選定、投資額{invested:,.0f}円")

        # 年次区間を日次で評価（DDガード + TOPIXフィルタ再エントリー + コスト）
        sim = simulate_period_daily_with_dd_guard(
            portfolio=long_portfolio,
            start_date=rebalance_date,
            end_date=next_rebalance,
            prices_df=prices_df,
            topix_signal=topix_signal,
            initial_capital=initial_capital
        )

        # TOPIX年次リターン（従来関数のまま）
        topix_return = calculate_topix_return(rebalance_date, next_rebalance, prices_df)

        strategy_results.append({
            'date': next_rebalance,

            'strategy_return_gross': sim['gross_return'],
            'strategy_return_net': sim['net_return'],
            'tax': sim['tax_rate_total'],
            'topix_return': topix_return,

            'long_count': len(long_portfolio['stocks']),
            'investment_ratio': invested / initial_capital,

            # 追加：DDガード診断
            'period_max_dd_daily': sim['max_dd_daily'],
            'dd_events': sim['dd_events'],
            'cash_days': sim['cash_days'],

            # 追加：売買コスト（率）
            'trading_cost_rate_total': sim['total_cost_rate'],
        })

    return pd.DataFrame(strategy_results)


# ===================================
# TOPIX年次リターン（元コードのまま）
# ===================================

def calculate_topix_return(start_date: pd.Timestamp, end_date: pd.Timestamp, prices_df: pd.DataFrame) -> float:
    """TOPIX基準リターン計算（簡易：銘柄の一部等ウェイト）"""
    all_stocks = prices_df['Code'].unique()[:100].tolist()

    start_window = prices_df[
        (prices_df['Code'].isin(all_stocks)) &
        (prices_df['Date'] >= start_date - pd.Timedelta(days=5)) &
        (prices_df['Date'] <= start_date + pd.Timedelta(days=5))
    ].sort_values(['Code', 'Date']).groupby('Code').first()

    end_window = prices_df[
        (prices_df['Code'].isin(all_stocks)) &
        (prices_df['Date'] >= end_date - pd.Timedelta(days=5)) &
        (prices_df['Date'] <= end_date + pd.Timedelta(days=5))
    ].sort_values(['Code', 'Date']).groupby('Code').last()

    common_codes = start_window.index.intersection(end_window.index)
    if len(common_codes) == 0:
        return 0.0

    start_prices = start_window.loc[common_codes, 'Close'].values
    end_prices = end_window.loc[common_codes, 'Close'].values
    returns = (end_prices - start_prices) / start_prices

    return float(np.mean(returns))


# ===================================
# G. パフォーマンス分析（元コード流用）
# ※年次シリーズのMDDは「年次」になりやすいので、
#   追加で results_df['period_max_dd_daily'] の最小値を別途参照してください
# ===================================

def calculate_annual_performance(returns: pd.Series, dates: pd.Series = None) -> float:
    if len(returns) == 0:
        return 0.0
    cumulative = (1 + returns).prod()
    years = len(returns)  # 年次リバランス前提
    return ((cumulative ** (1 / years)) - 1) * 100 if years > 0 else 0.0


def calculate_sharpe_ratio(returns: pd.Series, risk_free_rate: float = 0.0, dates: pd.Series = None) -> float:
    if len(returns) == 0 or returns.std() == 0:
        return 0.0
    excess_returns = returns - risk_free_rate
    mean_return = excess_returns.mean()
    std_return = excess_returns.std()
    return float((mean_return / std_return) * np.sqrt(1))


def calculate_max_drawdown(returns: pd.Series) -> float:
    if len(returns) == 0:
        return 0.0
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max
    return float(drawdown.min() * 100)


def calculate_additional_metrics(results_df: pd.DataFrame) -> dict:
    returns = results_df['strategy_return_net']

    metrics = {
        'annual_return': calculate_annual_performance(returns),
        'sharpe_ratio': calculate_sharpe_ratio(returns),
        'max_drawdown_annual_series': calculate_max_drawdown(returns),
        'avg_investment_ratio': results_df['investment_ratio'].mean() * 100,
        'avg_trading_cost_rate': results_df['trading_cost_rate_total'].mean() * 100,
        'sum_trading_cost_rate': results_df['trading_cost_rate_total'].sum() * 100,
        # 重要：日次DD（年次区間内の最大DD）の最悪
        'worst_daily_dd_in_periods': results_df['period_max_dd_daily'].min(),
        'dd_events_total': results_df['dd_events'].sum(),
        'cash_days_total': results_df['cash_days'].sum(),
    }
    return metrics


def print_performance_report(results_df: pd.DataFrame, metrics: dict):
    print("\n" + "=" * 90)
    print("📊 年次戦略（10/1年次リバランス）DDガード + TOPIX200MA再エントリー + コスト込み")
    print("=" * 90)

    print("\n【要件パラメータ】")
    print(f"  DD_STOP:                {DD_STOP*100:.1f}%")
    print(f"  REENTRY_MODE:           {REENTRY_MODE}（TOPIX > 200MA）")
    print(f"  売買コスト（往復）:      {TOTAL_COST_ROUNDTRIP*100:.2f}% × ターンオーバー")
    print(f"  税率:                   {TAX_RATE*100:.3f}%")

    print("\n【税引き後（年次リターン系列ベース）】")
    print(f"  年率リターン:           {metrics['annual_return']:.2f}%")
    print(f"  シャープレシオ:          {metrics['sharpe_ratio']:.2f}")
    print(f"  最大DD（年次系列）:      {metrics['max_drawdown_annual_series']:.2f}%")

    print("\n【DDガード診断（重要：年次区間内の日次DD）】")
    print(f"  最悪の区間内日次DD:      {metrics['worst_daily_dd_in_periods']:.2f}%")
    print(f"  DD撤退回数（合計）:      {int(metrics['dd_events_total'])}回")
    print(f"  CASH日数（合計）:        {int(metrics['cash_days_total'])}日")

    print("\n【売買コスト】")
    print(f"  平均コスト率（年次）:    {metrics['avg_trading_cost_rate']:.3f}%")
    print(f"  コスト率合計（全期間）:  {metrics['sum_trading_cost_rate']:.3f}%")

    print("\n【年別（10月→翌10月）】")
    show_cols = [
        'date', 'strategy_return_net', 'strategy_return_gross', 'tax',
        'trading_cost_rate_total', 'period_max_dd_daily', 'dd_events', 'cash_days',
        'long_count', 'investment_ratio'
    ]
    df_show = results_df[show_cols].copy()
    df_show['date'] = df_show['date'].dt.strftime('%Y-%m-%d')
    df_show['strategy_return_net'] = df_show['strategy_return_net'] * 100
    df_show['strategy_return_gross'] = df_show['strategy_return_gross'] * 100
    df_show['tax'] = df_show['tax'] * 100
    df_show['trading_cost_rate_total'] = df_show['trading_cost_rate_total'] * 100
    df_show['investment_ratio'] = df_show['investment_ratio'] * 100

    print(df_show.to_string(index=False, justify='center',
                            formatters={
                                'strategy_return_net': '{:.2f}%'.format,
                                'strategy_return_gross': '{:.2f}%'.format,
                                'tax': '{:.3f}%'.format,
                                'trading_cost_rate_total': '{:.3f}%'.format,
                                'period_max_dd_daily': '{:.2f}%'.format,
                                'investment_ratio': '{:.1f}%'.format,
                            }))


# ===================================
# I. メイン実行
# ===================================

if __name__ == "__main__":

    print("=" * 80)
    print("🏆 SMBC「割安高質」戦略 - 10月1日リバランス版（100株単位制限）")
    print("     DDガード（-10%撤退）+ TOPIX200MA再エントリー + コスト（往復0.4%）")
    print("=" * 80)
    print(f"\n📋 検証概要:")
    print(f"  戦略名: 割安高質（ロングオンリー）")
    print(f"  検証期間: 2016年10月〜2025年10月")
    print(f"  リバランス: 年次（10月1日）")
    print(f"  ポジション: ロングのみ（ショートなし）")
    print(f"  制約: 100株単位購入制限")
    print(f"  DD_STOP: {DD_STOP*100:.1f}%")
    print(f"  再エントリー: TOPIXが200日MAを上回る")
    print(f"  売買コスト: 往復0.4%×ターンオーバー")
    print(f"  譲渡益税: {TAX_RATE*100:.3f}%")
    print(f"  実行日時: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)

    # データ読み込み
    statements_df = load_financial_data()
    if statements_df.empty:
        logger.error("財務データ読み込み失敗")
        raise SystemExit(1)

    prices_df = load_existing_price_data()
    if prices_df.empty:
        logger.error("株価データ読み込み失敗")
        raise SystemExit(1)

    # 財務指標計算
    enhanced_financial_data = calculate_market_metrics_fast_chunked(statements_df, prices_df, chunk_size=200)
    if enhanced_financial_data.empty:
        logger.error("財務指標計算失敗")
        raise SystemExit(1)

    # バックテスト実行（DDガード正式版）
    results = smbc_value_quality_backtest_october_unit(enhanced_financial_data, prices_df, initial_capital=10_000_000)
    if results.empty:
        logger.error("バックテスト実行失敗")
        raise SystemExit(1)

    # 指標計算・出力
    metrics = calculate_additional_metrics(results)
    print_performance_report(results, metrics)

    # 保存
    out_csv = 'backtest_results_october_unit_ddguard_topix200ma_cost.csv'
    results.to_csv(out_csv, index=False, encoding='utf-8-sig')
    logger.info(f"結果をCSVに保存: {out_csv}")


🏆 SMBC「割安高質」戦略 - 10月1日リバランス版（100株単位制限）
     DDガード（-10%撤退）+ TOPIX200MA再エントリー + コスト（往復0.4%）

📋 検証概要:
  戦略名: 割安高質（ロングオンリー）
  検証期間: 2016年10月〜2025年10月
  リバランス: 年次（10月1日）
  ポジション: ロングのみ（ショートなし）
  制約: 100株単位購入制限
  DD_STOP: -10.0%
  再エントリー: TOPIXが200日MAを上回る
  売買コスト: 往復0.4%×ターンオーバー
  譲渡益税: 20.315%
  実行日時: 2026-01-21 05:35:56


2026-01-21 05:35:57,689 - INFO - 財務データ読み込み成功: 64,222件
2026-01-21 05:35:57,723 - INFO - 使用列: ['Code', 'DisclosedDate', 'CompanyName', 'Profit', 'Equity', 'IssuedShareTotal']
2026-01-21 05:35:57,726 - INFO - 株価ファイル数: 50個
株価ファイル読み込み中: 100%|██████████| 50/50 [00:07<00:00,  6.47it/s]
2026-01-21 05:36:05,473 - INFO - 全ファイルを結合中...
2026-01-21 05:36:05,515 - INFO - 結合完了: 9,929,369件
2026-01-21 05:36:09,039 - INFO - 重複除去 & ソート中...
2026-01-21 05:36:11,104 - INFO - 重複除去後: 9,606,444件
2026-01-21 05:36:11,115 - INFO - 時価総額・PBR・ROE計算中（チャンク処理版）...
2026-01-21 05:36:11,122 - INFO - 有効な財務データ: 62,741件
2026-01-21 05:36:11,123 - INFO - Code列を整数型に変換中...
2026-01-21 05:36:19,486 - INFO - ソート・重複除去後: 財務 62,463件, 株価 9,605,995件
2026-01-21 05:36:19,488 - INFO - 銘柄ごとにマージ中...
マージ処理: 100%|██████████| 9/9 [00:01<00:00,  4.93it/s]
2026-01-21 05:36:22,178 - INFO - マージ完了: 62,360件
2026-01-21 05:36:22,192 - INFO - 計算完了: 62,108件
2026-01-21 05:36:22,373 - INFO - 10月1日リバランス戦略バックテスト実行中（DDガード・TOPIX200MA・コスト込み）...
2026-01-21 05:36:


📊 年次戦略（10/1年次リバランス）DDガード + TOPIX200MA再エントリー + コスト込み

【要件パラメータ】
  DD_STOP:                -10.0%
  REENTRY_MODE:           topix_filter（TOPIX > 200MA）
  売買コスト（往復）:      0.40% × ターンオーバー
  税率:                   20.315%

【税引き後（年次リターン系列ベース）】
  年率リターン:           7.40%
  シャープレシオ:          0.45
  最大DD（年次系列）:      -23.16%

【DDガード診断（重要：年次区間内の日次DD）】
  最悪の区間内日次DD:      -27.23%
  DD撤退回数（合計）:      27回
  CASH日数（合計）:        910日

【売買コスト】
  平均コスト率（年次）:    2.337%
  コスト率合計（全期間）:  21.030%

【年別（10月→翌10月）】
   date    strategy_return_net strategy_return_gross   tax   trading_cost_rate_total period_max_dd_daily  dd_events  cash_days  long_count investment_ratio
2017-10-01        14.69%               18.44%         3.746%          0.754%               -10.34%           1         115          20          93.8%      
2018-10-01         1.07%                1.35%         0.273%          4.596%               -17.07%           6         152          20          90.9%      
2019-10-01       -14.79%              -14